# Final reproducible solution

In [ ]:

from pathlib import Path
import os
import random
import shutil
import sys
import platform
import subprocess
import importlib.metadata as metadata

import numpy as np
import pandas as pd

RANDOM_SEED = 2026
SELECTED_CANDIDATE_317_CONFIG = "huber030_g0.80_c0.008"

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd().resolve()
while not (ROOT / "analysis").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

external_input = ROOT / "train.csv"
compat_input = ROOT / "train_2.csv"

# Organizer-facing input policy:
# - train.csv is the authoritative external input when present, including closed-test data;
# - train_2.csv is kept as the compatibility path used by the historical pipeline code.
# This lets the same notebook run locally on the repository data and on organizer-provided data.
if external_input.exists():
    active_input = external_input
    shutil.copyfile(external_input, compat_input)
else:
    active_input = compat_input

assert compat_input.exists(), "Expected train.csv or train_2.csv in project root"
assert (ROOT / "analysis").exists(), "Expected analysis/ directory in project root"

try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass

(ROOT / "results").mkdir(exist_ok=True)
(ROOT / "submissions").mkdir(exist_ok=True)

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Project root:", ROOT)
print("Active input:", active_input)
print("Pipeline compatibility input:", compat_input)
print("RANDOM_SEED:", RANDOM_SEED)
print("Selected candidate 317 config:", SELECTED_CANDIDATE_317_CONFIG)


## Installed Libraries

Фиксируем версии ключевых библиотек и сохраняем полный `pip freeze` в `results/final_reproducible/package_versions.csv`.


In [ ]:

key_packages = ["numpy", "pandas", "scikit-learn", "scipy", "torch", "catboost", "joblib", "threadpoolctl", "nbformat", "nbclient"]
versions = []
for package in key_packages:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "not installed"
    versions.append({"package": package, "version": version})
versions_df = pd.DataFrame(versions)
display(versions_df)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True).splitlines()
freeze_df = pd.DataFrame({"requirement": freeze})
versions_dir = ROOT / "results" / "final_reproducible"
versions_dir.mkdir(parents=True, exist_ok=True)
versions_df.to_csv(versions_dir / "key_package_versions.csv", index=False)
freeze_df.to_csv(versions_dir / "package_versions.csv", index=False)
required = ["numpy", "pandas", "scikit-learn", "torch", "catboost"]
missing = versions_df.loc[versions_df["package"].isin(required) & versions_df["version"].eq("not installed"), "package"].tolist()
assert not missing, f"Missing required packages: {missing}"


## Inlined Helper Code


### Helper A. CatBoost/base utility definitions


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")


import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from catboost import CatBoostRegressor, Pool
except ImportError:
    CatBoostRegressor = None
    Pool = None


ROOT = ROOT
TRAIN_PATH = ROOT / "train_2.csv"
RESULTS_DIR = ROOT / "results" / "catboost_residual"
SUBMISSIONS_DIR = ROOT / "submissions"
TARGET = "РТО"
SEED = 2026

CAT_COLS = [
    "Дата открытия, категориальный",
    "Торговая площадь, категориальный",
    "Населенный пункт",
    "Регион",
]

STATIC_NUM_COLS = [
    "Численность населения",
    "Количество домохозяйств",
    "Трафик пеший, в час",
    "Трафик авто, в час",
    "Маркетплейсы, доставки, постаматы (100 м)",
    "Медицинские уч. и аптеки (300 м)",
    "Школы (300 м)",
    "Остановки (300 м)",
    "Продуктовые магазины (500 м)",
    "Пятерочки (500 м)",
    "Количество касс",
    "Флаг алкогольной лицензии",
]

BEHAVIOR_COLS = [
    "Среднее количество промо товаров в чеке",
    "Среднее количество товаров в чеке",
    "Среднее количество отмен",
    "Рабочие часы в день",
]

MODEL_PARAMS = dict(
    loss_function="MAE",
    eval_metric="MAE",
    iterations=250,
    learning_rate=0.04,
    depth=5,
    l2_leaf_reg=12.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=False,
)

ALPHA_GRID = [0.0, 0.03, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
CLIP_GRID = [None, 0.10, 0.05, 0.02]


def mape(y_true: np.ndarray | pd.Series, y_pred: np.ndarray | pd.Series) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(y_true == 0, 1.0, y_true)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100)


def make_panel(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["period"] = out["Год"] * 12 + out["Месяц"]
    out = out.sort_values(["new_id", "period"]).reset_index(drop=True)
    for col in CAT_COLS:
        out[col] = out[col].astype(str)

    target_group = out.groupby("new_id")[TARGET]
    for lag in [1, 2, 3, 4, 5, 6, 12, 13]:
        out[f"lag{lag}"] = target_group.shift(lag)

    out["rolling3_mean"] = out[["lag1", "lag2", "lag3"]].mean(axis=1)
    out["momentum_raw"] = out["lag1"] / out["rolling3_mean"]
    out["momentum_clip"] = out["momentum_raw"].clip(0.95, 1.05)
    out["momentum_corr"] = out["momentum_clip"] ** 0.25

    out["store_mar_feb_ratio"] = out["lag12"] / out["lag13"]
    out["delta_yoy"] = out["lag12"] + (out["lag1"] - out["lag13"])
    out["trend_yoy"] = out["lag12"] * (out["lag1"] / out["lag13"])
    out["diff1"] = out["lag1"] - out["lag2"]
    out["diff2"] = out["lag2"] - out["lag3"]
    out["accel1"] = out["diff1"] - out["diff2"]
    out["lag1_to_lag12"] = out["lag1"] / out["lag12"]
    out["lag1_to_lag13"] = out["lag1"] / out["lag13"]

    shifted = target_group.shift(1)
    for window in [3, 6, 12]:
        rolled = shifted.groupby(out["new_id"]).rolling(window, min_periods=2)
        out[f"roll{window}_mean"] = rolled.mean().reset_index(level=0, drop=True)
        out[f"roll{window}_std"] = rolled.std().reset_index(level=0, drop=True)
        out[f"roll{window}_min"] = rolled.min().reset_index(level=0, drop=True)
        out[f"roll{window}_max"] = rolled.max().reset_index(level=0, drop=True)

    out["volatility_rel3"] = out["roll3_std"] / out["roll3_mean"]
    out["volatility_rel6"] = out["roll6_std"] / out["roll6_mean"]

    for col in BEHAVIOR_COLS:
        behavior_group = out.groupby("new_id")[col]
        for lag in [1, 2, 3, 12, 13]:
            out[f"{col}_lag{lag}"] = behavior_group.shift(lag)
        out[f"{col}_diff1"] = out[f"{col}_lag1"] - out[f"{col}_lag2"]
        out[f"{col}_yoy_shift"] = out[f"{col}_lag1"] - out[f"{col}_lag13"]

    return out.replace([np.inf, -np.inf], np.nan)


def feature_cols() -> list[str]:
    cols = [
        "lag1",
        "lag2",
        "lag3",
        "lag4",
        "lag5",
        "lag6",
        "lag12",
        "lag13",
        "rolling3_mean",
        "momentum_raw",
        "momentum_clip",
        "momentum_corr",
        "store_mar_feb_ratio",
        "delta_yoy",
        "trend_yoy",
        "diff1",
        "diff2",
        "accel1",
        "lag1_to_lag12",
        "lag1_to_lag13",
        "roll3_mean",
        "roll3_std",
        "roll3_min",
        "roll3_max",
        "roll6_mean",
        "roll6_std",
        "roll6_min",
        "roll6_max",
        "roll12_mean",
        "roll12_std",
        "roll12_min",
        "roll12_max",
        "volatility_rel3",
        "volatility_rel6",
    ]
    cols += STATIC_NUM_COLS + CAT_COLS
    for col in BEHAVIOR_COLS:
        cols += [
            f"{col}_lag1",
            f"{col}_lag2",
            f"{col}_lag3",
            f"{col}_lag12",
            f"{col}_lag13",
            f"{col}_diff1",
            f"{col}_yoy_shift",
        ]
    return cols


def add_base(frame: pd.DataFrame, seasonal_factor: float) -> pd.DataFrame:
    out = frame.copy()
    out["best_base"] = out["lag1"] * seasonal_factor * out["momentum_corr"]
    return out


def apply_residual(base: np.ndarray, pred_log: np.ndarray, alpha: float, clip: float | None) -> np.ndarray:
    raw = base * np.exp(alpha * pred_log)
    if clip is None:
        return raw
    return base * np.clip(raw / base, 1 - clip, 1 + clip)


def bin_summary(frame: pd.DataFrame, key: str, bins: int = 5) -> pd.DataFrame:
    cols = list(dict.fromkeys(["fold", key, TARGET, "best_base", "pred_alpha_050", "pred_alpha_060"]))
    work = frame[cols].copy()
    work["bucket"] = pd.qcut(work[key], q=bins, duplicates="drop")
    rows = []
    for bucket, group in work.groupby("bucket", observed=True):
        rows.append(
            {
                "segment": key,
                "bucket": str(bucket),
                "stores": len(group),
                "base_mape": mape(group[TARGET], group["best_base"]),
                "alpha_050_mape": mape(group[TARGET], group["pred_alpha_050"]),
                "alpha_060_mape": mape(group[TARGET], group["pred_alpha_060"]),
                "gain_050": mape(group[TARGET], group["best_base"]) - mape(group[TARGET], group["pred_alpha_050"]),
                "gain_060": mape(group[TARGET], group["best_base"]) - mape(group[TARGET], group["pred_alpha_060"]),
            }
        )
    return pd.DataFrame(rows)


def main() -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

    raw = pd.read_csv(TRAIN_PATH, low_memory=False)
    period_label = pd.to_datetime(
        raw["Год"].astype(str) + "-" + raw["Месяц"].astype(str).str.zfill(2) + "-01"
    ).dt.strftime("%Y-%m")
    rto = raw.assign(period_label=period_label).pivot_table(
        index="new_id", columns="period_label", values=TARGET, aggfunc="mean"
    )

    ratio_2023 = (rto["2023-03"] / rto["2023-02"]).replace([np.inf, -np.inf], np.nan)
    ratio_2024 = (rto["2024-03"] / rto["2024-02"]).replace([np.inf, -np.inf], np.nan)
    validation_factor = float(ratio_2023.median())
    final_factor = float(pd.concat([ratio_2023, ratio_2024]).dropna().median())

    features = feature_cols()
    cat_feature_idx = [features.index(col) for col in CAT_COLS]
    panel = add_base(make_panel(raw), validation_factor)
    march_2024 = panel[
        (panel["Год"] == 2024)
        & (panel["Месяц"] == 3)
        & (panel[TARGET] > 0)
        & (panel["best_base"] > 0)
    ].copy()
    march_2024["target_log_ratio"] = np.log(march_2024[TARGET] / march_2024["best_base"])

    cv_rows = []
    oof_parts = []
    for fold in range(5):
        valid_mask = (march_2024["new_id"] % 5) == fold
        train = march_2024[~valid_mask].copy()
        valid = march_2024[valid_mask].copy()

        model = CatBoostRegressor(**MODEL_PARAMS)
        model.fit(Pool(train[features], train["target_log_ratio"], cat_features=cat_feature_idx))
        pred_log = model.predict(valid[features])
        y = valid[TARGET].to_numpy(dtype=float)
        base = valid["best_base"].to_numpy(dtype=float)

        valid_oof = valid[
            [
                "new_id",
                TARGET,
                "best_base",
                "momentum_raw",
                "momentum_clip",
                "volatility_rel3",
                "volatility_rel6",
                "lag1_to_lag12",
                "Регион",
                "Торговая площадь, категориальный",
            ]
        ].copy()
        valid_oof["fold"] = fold
        valid_oof["pred_log"] = pred_log
        valid_oof["correction_alpha_050"] = np.exp(0.5 * pred_log)
        valid_oof["pred_alpha_050"] = apply_residual(base, pred_log, 0.5, None)
        valid_oof["pred_alpha_060"] = apply_residual(base, pred_log, 0.6, None)
        oof_parts.append(valid_oof)

        for alpha in ALPHA_GRID:
            for clip in CLIP_GRID:
                pred = apply_residual(base, pred_log, alpha, clip)
                correction = pred / base
                cv_rows.append(
                    {
                        "fold": fold,
                        "alpha": alpha,
                        "clip": "no_clip" if clip is None else f"clip_{clip:.2f}",
                        "mape_ml": mape(y, pred),
                        "mape_base": mape(y, base),
                        "mean_abs_correction_pct": float(np.mean(np.abs(correction - 1)) * 100),
                        "p99_correction": float(pd.Series(correction).quantile(0.99)),
                        "max_correction": float(pd.Series(correction).max()),
                    }
                )

    cv = pd.DataFrame(cv_rows)
    cv_summary = (
        cv.groupby(["alpha", "clip"], as_index=False)
        .agg(
            mape_ml=("mape_ml", "mean"),
            mape_base=("mape_base", "mean"),
            mean_abs_correction_pct=("mean_abs_correction_pct", "mean"),
            p99_correction=("p99_correction", "mean"),
            max_correction=("max_correction", "max"),
        )
        .sort_values("mape_ml")
        .reset_index(drop=True)
    )
    cv_summary["gain_vs_base"] = cv_summary["mape_base"] - cv_summary["mape_ml"]
    oof = pd.concat(oof_parts, ignore_index=True)

    segment_summary = pd.concat(
        [
            bin_summary(oof, "best_base"),
            bin_summary(oof, "momentum_raw"),
            bin_summary(oof, "volatility_rel3"),
            bin_summary(oof, "lag1_to_lag12"),
        ],
        ignore_index=True,
    )
    region_summary = []
    for region, group in oof.groupby("Регион"):
        if len(group) >= 150:
            base_mape = mape(group[TARGET], group["best_base"])
            region_summary.append(
                {
                    "Регион": region,
                    "stores": len(group),
                    "base_mape": base_mape,
                    "alpha_050_mape": mape(group[TARGET], group["pred_alpha_050"]),
                    "alpha_060_mape": mape(group[TARGET], group["pred_alpha_060"]),
                    "gain_050": base_mape - mape(group[TARGET], group["pred_alpha_050"]),
                    "gain_060": base_mape - mape(group[TARGET], group["pred_alpha_060"]),
                }
            )
    region_summary = pd.DataFrame(region_summary).sort_values("gain_050", ascending=False)

    tail_summary = oof["correction_alpha_050"].describe(
        percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
    ).to_frame("alpha_050_oof_correction")
    tail_stores = oof.assign(abs_corr=(oof["correction_alpha_050"] - 1).abs()).sort_values(
        "abs_corr", ascending=False
    ).head(50)

    final_model = CatBoostRegressor(**MODEL_PARAMS)
    final_model.fit(Pool(march_2024[features], march_2024["target_log_ratio"], cat_features=cat_feature_idx))

    feb_2025 = raw[(raw["Год"] == 2025) & (raw["Месяц"] == 2)].copy()
    mar_2025 = feb_2025.copy()
    mar_2025["Год"] = 2025
    mar_2025["Месяц"] = 3
    mar_2025[TARGET] = np.nan
    pred_panel = add_base(make_panel(pd.concat([raw, mar_2025], ignore_index=True)), final_factor)
    pred_rows = pred_panel[(pred_panel["Год"] == 2025) & (pred_panel["Месяц"] == 3)].copy()
    pred_log = final_model.predict(pred_rows[features])
    base = pred_rows["best_base"].to_numpy(dtype=float)

    candidate_specs = [
        ("candidate_cb_alpha050_noclip", 0.5, None),
        ("candidate_cb_alpha060_noclip", 0.6, None),
        ("candidate_cb_alpha050_clip010", 0.5, 0.10),
    ]
    candidate_rows = []
    candidate_predictions = {}
    for name, alpha, clip in candidate_specs:
        pred = np.maximum(apply_residual(base, pred_log, alpha, clip), 0)
        candidate_predictions[name] = pred
        correction = pred / base
        candidate_rows.append(
            {
                "candidate": name,
                "alpha": alpha,
                "clip": "no_clip" if clip is None else clip,
                "rows": len(pred_rows),
                "mean_abs_correction_pct": float(np.mean(np.abs(correction - 1)) * 100),
                "p99_correction": float(pd.Series(correction).quantile(0.99)),
                "max_correction": float(pd.Series(correction).max()),
            }
        )
    candidate_summary = pd.DataFrame(candidate_rows)

    pick = cv_summary[(cv_summary["clip"] == "no_clip") & (cv_summary["alpha"].isin([0.5, 0.6]))].sort_values(
        "mape_ml"
    ).iloc[0]
    recommended_name = "candidate_cb_alpha060_noclip" if float(pick["alpha"]) == 0.6 else "candidate_cb_alpha050_noclip"
    recommended_submission = pred_rows[["new_id"]].copy()
    recommended_submission["rto"] = np.round(candidate_predictions[recommended_name], 2)
    recommended_submission = recommended_submission.sort_values("new_id").reset_index(drop=True)
    recommended_path = SUBMISSIONS_DIR / "test_next.csv"
    recommended_submission.to_csv(recommended_path, index=False)

    cv.to_csv(RESULTS_DIR / "cv_folds.csv", index=False)
    cv_summary.to_csv(RESULTS_DIR / "cv_summary.csv", index=False)
    oof.to_csv(RESULTS_DIR / "oof_march_2024.csv", index=False)
    segment_summary.to_csv(RESULTS_DIR / "segment_summary.csv", index=False)
    region_summary.to_csv(RESULTS_DIR / "region_summary.csv", index=False)
    tail_summary.to_csv(RESULTS_DIR / "correction_tail_summary.csv")
    tail_stores.to_csv(RESULTS_DIR / "largest_oof_corrections.csv", index=False)
    candidate_summary.to_csv(RESULTS_DIR / "candidate_summary.csv", index=False)

    feature_importance = pd.DataFrame(
        {
            "feature": features,
            "importance": final_model.get_feature_importance(
                Pool(march_2024[features], march_2024["target_log_ratio"], cat_features=cat_feature_idx),
                type="FeatureImportance",
            ),
        }
    ).sort_values("importance", ascending=False)
    feature_importance.to_csv(RESULTS_DIR / "feature_importance.csv", index=False)

    report = {
        "validation_factor": validation_factor,
        "final_factor": final_factor,
        "march_2024_rows": len(march_2024),
        "cv_top": cv_summary.head(12).to_dict(orient="records"),
        "candidate_summary": candidate_summary.to_dict(orient="records"),
        "recommended_submission": str(recommended_path),
        "recommended_candidate": recommended_name,
        "recommended_basis": {
            "alpha": float(pick["alpha"]),
            "clip": str(pick["clip"]),
            "oof_mape": float(pick["mape_ml"]),
            "oof_gain_vs_base": float(pick["gain_vs_base"]),
        },
        "feature_importance_top": feature_importance.head(15).to_dict(orient="records"),
    }
    (RESULTS_DIR / "summary.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2))

CB_BEHAVIOR_COLS = BEHAVIOR_COLS
CB_CAT_COLS = CAT_COLS
CB_STATIC_NUM_COLS = STATIC_NUM_COLS
CB_SUBMISSIONS_DIR = SUBMISSIONS_DIR
CB_TARGET = TARGET
CB_TRAIN_PATH = TRAIN_PATH
CB_add_base = add_base
CB_apply_residual = apply_residual
CB_make_panel = make_panel
CB_mape = mape


### Helper B. Core heuristic baseline definitions


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")


from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool


ROOT = ROOT
TRAIN_PATH = ROOT / "train_2.csv"
RESULTS_DIR = ROOT / "results"
SUBMISSIONS_DIR = ROOT / "submissions"
TARGET = "РТО"
SEED = 2026

STATIC_CAT_COLS = [
    "Регион",
    "Торговая площадь, категориальный",
    "Дата открытия, категориальный",
]

STATIC_NUM_COLS = [
    "Численность населения",
    "Количество домохозяйств",
    "Трафик пеший, в час",
    "Трафик авто, в час",
    "Маркетплейсы, доставки, постаматы (100 м)",
    "Медицинские уч. и аптеки (100 м)",
    "Остановки (100 м)",
    "Оптики (100 м)",
    "ПВЗ (100 м)",
    "Спорт (100 м)",
    "Цветы (100 м)",
    "Алкоголь (100 м)",
    "Кафе и рестораны (100 м)",
    "Косметика (100 м)",
    "Продуктовые магазины (100 м)",
    "Промтовары магазины (100 м)",
    "Алкоголь (500 м)",
    "Кафе и рестораны (500 м)",
    "Косметика (500 м)",
    "Продуктовые магазины (500 м)",
    "Промтовары магазины (500 м)",
]


@dataclass(frozen=True)
class HeuristicSpec:
    candidate_id: int
    name: str
    calendar_lambda: float
    factor_method: str = "median"
    momentum_exp: float = 0.25
    rebound_lift: float = 0.0
    structural_cut: float = 0.0
    group_col: str | None = None
    group_shrink_k: float = 2000.0


def mape(y_true: pd.Series | np.ndarray, y_pred: pd.Series | np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)


def weighted_median(values: pd.Series, weights: pd.Series) -> float:
    frame = pd.DataFrame({"value": values, "weight": weights}).replace([np.inf, -np.inf], np.nan).dropna()
    frame = frame.sort_values("value")
    cutoff = frame["weight"].sum() * 0.5
    return float(frame.loc[frame["weight"].cumsum() >= cutoff, "value"].iloc[0])


def period_label(frame: pd.DataFrame) -> pd.Series:
    return pd.to_datetime(
        frame["Год"].astype(str) + "-" + frame["Месяц"].astype(str).str.zfill(2) + "-01"
    ).dt.strftime("%Y-%m")


def label(year: int, month: int, months_back: int = 0) -> str:
    return (pd.Timestamp(year=year, month=month, day=1) - pd.DateOffset(months=months_back)).strftime("%Y-%m")


def safe_div(num: pd.Series, den: pd.Series) -> pd.Series:
    return (num / den).replace([np.inf, -np.inf], np.nan)


def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    raw = pd.read_csv(TRAIN_PATH, low_memory=False)
    raw["period_label"] = period_label(raw)
    rto = raw.pivot_table(index="new_id", columns="period_label", values=TARGET, aggfunc="mean")
    static_cols = ["new_id"] + [c for c in STATIC_CAT_COLS + STATIC_NUM_COLS if c in raw.columns]
    static = raw.sort_values(["new_id", "Год", "Месяц"]).groupby("new_id", as_index=False)[static_cols].first()
    return raw, rto, static.set_index("new_id")


def build_target_frame(rto: pd.DataFrame, static: pd.DataFrame, year: int, month: int, include_target: bool) -> pd.DataFrame:
    out = pd.DataFrame(index=rto.index)
    out["new_id"] = out.index.astype(int)
    out["lag1"] = rto[label(year, month, 1)]
    out["lag2"] = rto[label(year, month, 2)]
    out["lag3"] = rto[label(year, month, 3)]
    out["lag12"] = rto[label(year, month, 12)]
    out["lag13"] = rto[label(year, month, 13)]
    out["rolling3_mean"] = out[["lag1", "lag2", "lag3"]].mean(axis=1)
    out["rolling2_pre_mean"] = out[["lag2", "lag3"]].mean(axis=1)
    out["momentum_raw"] = safe_div(out["lag1"], out["rolling3_mean"])
    out["momentum_clip"] = out["momentum_raw"].fillna(1.0).clip(0.95, 1.05)
    out["feb_to_jan"] = safe_div(out["lag1"], out["lag2"])
    out["jan_to_dec"] = safe_div(out["lag2"], out["lag3"])
    out["lag1_to_lag12"] = safe_div(out["lag1"], out["lag12"])
    out["lag1_to_lag13"] = safe_div(out["lag1"], out["lag13"])
    out["store_mar_feb_ratio_prev"] = safe_div(out["lag12"], out["lag13"])
    out["volatility_rel3"] = out[["lag1", "lag2", "lag3"]].std(axis=1) / out["rolling3_mean"]
    out["dip_depth"] = ((out["rolling2_pre_mean"] - out["lag1"]) / out["rolling2_pre_mean"]).clip(lower=0, upper=0.25).fillna(0)
    out["dip_score"] = (out["dip_depth"] / 0.25).clip(0, 1)
    out["temporary_dip_score"] = out["dip_score"] * (
        out["jan_to_dec"].fillna(1.0).ge(0.98)
        & out["lag1_to_lag12"].fillna(1.0).ge(1.00)
    ).astype(float)
    out["structural_decline_score"] = (
        out["feb_to_jan"].fillna(1.0).lt(0.97)
        & out["jan_to_dec"].fillna(1.0).lt(0.99)
        & out["lag1_to_lag12"].fillna(1.0).lt(1.02)
    ).astype(float)
    if include_target:
        out[TARGET] = rto[label(year, month)]
    out = out.join(static, how="left")
    for col in STATIC_CAT_COLS:
        if col in out.columns:
            out[col] = out[col].astype(str).fillna("__nan__")
    return out.reset_index(drop=True)


def ratio_frame(rto: pd.DataFrame, static: pd.DataFrame, year: int, calendar_lambda: float, target_non_leap: bool) -> pd.DataFrame:
    ratio = safe_div(rto[f"{year}-03"], rto[f"{year}-02"])
    # February 2024 had 29 days. Convert it partially toward a non-leap February ratio for 2025,
    # or partially down when using 2023 to predict leap-year March 2024 validation.
    if year == 2024 and target_non_leap:
        ratio = ratio * ((1.0 - calendar_lambda) + calendar_lambda * (29.0 / 28.0))
    elif year == 2023 and not target_non_leap:
        ratio = ratio * ((1.0 - calendar_lambda) + calendar_lambda * (28.0 / 29.0))
    frame = pd.DataFrame({"new_id": ratio.index.astype(int), "ratio": ratio}).dropna().reset_index(drop=True)
    return frame.join(static, on="new_id", how="left")


def global_factor(ratios: pd.Series, method: str) -> float:
    ratios = ratios.replace([np.inf, -np.inf], np.nan).dropna()
    if method == "median":
        return float(ratios.median())
    if method == "mape_weighted_median":
        return weighted_median(ratios, 1.0 / ratios)
    if method == "aggregate_mean":
        return float(ratios.mean())
    raise ValueError(f"Unknown factor method: {method}")


def shrink_group_factors(
    ratio_data: pd.DataFrame,
    target_frame: pd.DataFrame,
    group_col: str,
    fallback_factor: float,
    shrink_k: float,
) -> pd.Series:
    stats = ratio_data.dropna(subset=["ratio"]).groupby(group_col, observed=True)["ratio"].agg(["median", "count"])
    correction = np.log(stats["median"] / fallback_factor) * (stats["count"] / (stats["count"] + shrink_k))
    mapped = target_frame[group_col].map(correction).fillna(0.0).to_numpy(dtype=float)
    return pd.Series(fallback_factor * np.exp(mapped), index=target_frame.index)


def factors_for_spec(
    spec: HeuristicSpec,
    val_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    rto: pd.DataFrame,
    static: pd.DataFrame,
) -> tuple[pd.Series, pd.Series, float, float]:
    val_ratio_data = ratio_frame(rto, static, 2023, spec.calendar_lambda, target_non_leap=False)
    final_ratio_2023 = ratio_frame(rto, static, 2023, spec.calendar_lambda, target_non_leap=True)
    final_ratio_2024 = ratio_frame(rto, static, 2024, spec.calendar_lambda, target_non_leap=True)
    final_ratio_data = pd.concat([final_ratio_2023, final_ratio_2024], ignore_index=True)

    val_global = global_factor(val_ratio_data["ratio"], spec.factor_method)
    final_global = global_factor(final_ratio_data["ratio"], spec.factor_method)

    if spec.group_col is None:
        return (
            pd.Series(val_global, index=val_frame.index),
            pd.Series(final_global, index=test_frame.index),
            val_global,
            final_global,
        )

    val_factor = shrink_group_factors(val_ratio_data, val_frame, spec.group_col, val_global, spec.group_shrink_k)
    final_factor = shrink_group_factors(final_ratio_data, test_frame, spec.group_col, final_global, spec.group_shrink_k)
    return val_factor, final_factor, val_global, final_global


def predict_from_spec(frame: pd.DataFrame, factor: pd.Series, spec: HeuristicSpec) -> np.ndarray:
    pred = frame["lag1"].to_numpy(dtype=float) * factor.to_numpy(dtype=float)
    pred *= frame["momentum_clip"].to_numpy(dtype=float) ** spec.momentum_exp
    if spec.rebound_lift:
        pred *= np.exp(spec.rebound_lift * frame["temporary_dip_score"].to_numpy(dtype=float))
    if spec.structural_cut:
        pred *= np.exp(-spec.structural_cut * frame["structural_decline_score"].to_numpy(dtype=float))
    return np.maximum(pred, 0.0)


def specs() -> list[HeuristicSpec]:
    return [
        HeuristicSpec(43, "raw_median_mom025", 0.0),
        HeuristicSpec(44, "cal0125_median_mom025", 0.125),
        HeuristicSpec(45, "cal025_median_mom025", 0.25),
        HeuristicSpec(46, "cal0375_median_mom025", 0.375),
        HeuristicSpec(47, "cal050_median_mom025", 0.50),
        HeuristicSpec(48, "cal025_mape_wmedian_mom025", 0.25, factor_method="mape_weighted_median"),
        HeuristicSpec(49, "cal025_median_mom000", 0.25, momentum_exp=0.0),
        HeuristicSpec(50, "cal025_median_mom0125", 0.25, momentum_exp=0.125),
        HeuristicSpec(51, "cal025_median_mom0375", 0.25, momentum_exp=0.375),
        HeuristicSpec(52, "cal025_tempdip_lift010", 0.25, rebound_lift=0.010),
        HeuristicSpec(53, "cal025_tempdip_lift020", 0.25, rebound_lift=0.020),
        HeuristicSpec(54, "cal025_tempdip_lift030", 0.25, rebound_lift=0.030),
        HeuristicSpec(55, "cal025_tempdip020_structcut010", 0.25, rebound_lift=0.020, structural_cut=0.010),
        HeuristicSpec(56, "cal0125_tempdip020_structcut010", 0.125, rebound_lift=0.020, structural_cut=0.010),
        HeuristicSpec(57, "cal0375_tempdip020_structcut010", 0.375, rebound_lift=0.020, structural_cut=0.010),
        HeuristicSpec(58, "cal025_region_k2000", 0.25, group_col="Регион", group_shrink_k=2000),
        HeuristicSpec(59, "cal025_area_k2000", 0.25, group_col="Торговая площадь, категориальный", group_shrink_k=2000),
        HeuristicSpec(60, "cal025_opening_k2000", 0.25, group_col="Дата открытия, категориальный", group_shrink_k=2000),
        HeuristicSpec(61, "cal025_region_k500", 0.25, group_col="Регион", group_shrink_k=500),
        HeuristicSpec(62, "cal025_opening_k500", 0.25, group_col="Дата открытия, категориальный", group_shrink_k=500),
    ]


def add_shrunk_factor_features(frame: pd.DataFrame, rto: pd.DataFrame, static: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    base_spec = HeuristicSpec(0, "cal025", 0.25)
    for group_col, shrink_k in [
        ("Регион", 2000),
        ("Торговая площадь, категориальный", 2000),
        ("Дата открытия, категориальный", 2000),
    ]:
        spec = HeuristicSpec(0, "factor", 0.25, group_col=group_col, group_shrink_k=shrink_k)
        val_factor, _, val_global, _ = factors_for_spec(spec, out, out, rto, static)
        out[f"factor_{group_col}_k{shrink_k}"] = val_factor / val_global
    _, _, val_global, _ = factors_for_spec(base_spec, out, out, rto, static)
    out["cal025_global_factor"] = val_global
    return out


def run_shap_diagnostic(val_frame: pd.DataFrame, current_pred: np.ndarray, rto: pd.DataFrame, static: pd.DataFrame) -> None:
    frame = add_shrunk_factor_features(val_frame, rto, static)
    frame["target_log_residual"] = np.log(frame[TARGET].to_numpy(dtype=float) / current_pred)
    feature_cols = [
        "momentum_raw",
        "momentum_clip",
        "feb_to_jan",
        "jan_to_dec",
        "lag1_to_lag12",
        "lag1_to_lag13",
        "store_mar_feb_ratio_prev",
        "volatility_rel3",
        "dip_score",
        "temporary_dip_score",
        "structural_decline_score",
        "factor_Регион_k2000",
        "factor_Торговая площадь, категориальный_k2000",
        "factor_Дата открытия, категориальный_k2000",
    ] + [col for col in STATIC_NUM_COLS if col in frame.columns] + STATIC_CAT_COLS

    cat_idx = [feature_cols.index(col) for col in STATIC_CAT_COLS if col in feature_cols]
    pool = Pool(frame[feature_cols], frame["target_log_residual"], cat_features=cat_idx)
    model = CatBoostRegressor(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=350,
        learning_rate=0.035,
        depth=3,
        l2_leaf_reg=30.0,
        random_seed=SEED,
        allow_writing_files=False,
        verbose=False,
    )
    model.fit(pool)
    shap_values = model.get_feature_importance(pool, type="ShapValues")[:, :-1]
    shap_summary = pd.DataFrame(
        {
            "feature": feature_cols,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
            "mean_shap": shap_values.mean(axis=0),
        }
    ).sort_values("mean_abs_shap", ascending=False)
    shap_summary.to_csv(RESULTS_DIR / "heuristic_direction_shap_summary.csv", index=False)

    bin_rows = []
    numeric_cols = [col for col in feature_cols if col not in STATIC_CAT_COLS and pd.api.types.is_numeric_dtype(frame[col])]
    for col in numeric_cols:
        values = frame[col].replace([np.inf, -np.inf], np.nan)
        try:
            bins = pd.qcut(values, q=5, duplicates="drop")
        except ValueError:
            continue
        tmp = frame.assign(_bin=bins)
        for bin_name, part in tmp.groupby("_bin", observed=True):
            if len(part) < 50:
                continue
            bin_rows.append(
                {
                    "feature": col,
                    "bin": str(bin_name),
                    "n": len(part),
                    "mean_feature": float(part[col].mean()),
                    "mean_log_residual": float(part["target_log_residual"].mean()),
                    "mape_current_base": mape(part[TARGET], current_pred[part.index]),
                }
            )
    pd.DataFrame(bin_rows).to_csv(RESULTS_DIR / "heuristic_direction_feature_bins.csv", index=False)


def main() -> None:
    RESULTS_DIR.mkdir(exist_ok=True)
    SUBMISSIONS_DIR.mkdir(exist_ok=True)
    _raw, rto, static = load_data()
    val_frame = build_target_frame(rto, static, 2024, 3, include_target=True)
    test_frame = build_target_frame(rto, static, 2025, 3, include_target=False)

    rows = []
    current_pred = None
    for spec in specs():
        val_factor, test_factor, val_global, final_global = factors_for_spec(spec, val_frame, test_frame, rto, static)
        val_pred = predict_from_spec(val_frame, val_factor, spec)
        test_pred = predict_from_spec(test_frame, test_factor, spec)

        if spec.name == "raw_median_mom025":
            current_pred = val_pred.copy()

        out = test_frame[["new_id"]].copy()
        out["rto"] = np.round(test_pred, 2)
        file_name = f"stage2_candidate_{spec.candidate_id:02d}_baseline_{spec.name}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)

        ratio_to_current = val_pred / current_pred if current_pred is not None else np.ones_like(val_pred)
        rows.append(
            {
                "candidate_id": spec.candidate_id,
                "file": file_name,
                "name": spec.name,
                "family": "heuristic_baseline",
                "calendar_lambda": spec.calendar_lambda,
                "factor_method": spec.factor_method,
                "momentum_exp": spec.momentum_exp,
                "rebound_lift": spec.rebound_lift,
                "structural_cut": spec.structural_cut,
                "group_col": spec.group_col or "",
                "group_shrink_k": spec.group_shrink_k if spec.group_col else np.nan,
                "val_global_factor": val_global,
                "final_global_factor": final_global,
                "val_mape_march2024": mape(val_frame[TARGET], val_pred),
                "val_weak_momentum_mape": mape(
                    val_frame.loc[val_frame["momentum_raw"] < 0.935766244455909, TARGET],
                    val_pred[val_frame["momentum_raw"] < 0.935766244455909],
                ),
                "val_temp_dip_mape": mape(
                    val_frame.loc[val_frame["temporary_dip_score"] > 0, TARGET],
                    val_pred[val_frame["temporary_dip_score"] > 0],
                )
                if int((val_frame["temporary_dip_score"] > 0).sum())
                else np.nan,
                "val_struct_decline_mape": mape(
                    val_frame.loc[val_frame["structural_decline_score"] > 0, TARGET],
                    val_pred[val_frame["structural_decline_score"] > 0],
                )
                if int((val_frame["structural_decline_score"] > 0).sum())
                else np.nan,
                "test_mean_rto": float(np.mean(test_pred)),
                "test_total_rto": float(np.sum(test_pred)),
                "val_mean_abs_change_vs_current_pct": float(np.mean(np.abs(ratio_to_current - 1.0)) * 100),
                "val_p95_abs_change_vs_current_pct": float(np.quantile(np.abs(ratio_to_current - 1.0), 0.95) * 100),
            }
        )

    summary = pd.DataFrame(rows).sort_values("val_mape_march2024").reset_index(drop=True)
    summary.to_csv(RESULTS_DIR / "heuristic_baseline_direction_candidates.csv", index=False)

    current_row = summary[summary["name"] == "raw_median_mom025"].iloc[0]
    diagnostic_rows = []
    for _, row in summary.iterrows():
        diagnostic_rows.append(
            {
                "candidate_id": row["candidate_id"],
                "name": row["name"],
                "delta_val_mape_vs_raw_median_mom025": row["val_mape_march2024"] - current_row["val_mape_march2024"],
                "direction_question": direction_question(str(row["name"])),
                "public_status": "not_submitted",
                "public_score": "",
                "public_mape": "",
                "decision_after_public": "",
            }
        )
    pd.DataFrame(diagnostic_rows).to_csv(RESULTS_DIR / "heuristic_public_experiment_template.csv", index=False)

    if current_pred is not None:
        run_shap_diagnostic(val_frame, current_pred, rto, static)

    print("Wrote heuristic candidates:")
    print(summary[["candidate_id", "name", "val_mape_march2024", "final_global_factor", "val_mean_abs_change_vs_current_pct"]].to_string(index=False))
    print("\nTop SHAP summary:")
    shap_path = RESULTS_DIR / "heuristic_direction_shap_summary.csv"
    if shap_path.exists():
        print(pd.read_csv(shap_path).head(20).to_string(index=False))


def direction_question(name: str) -> str:
    if name.startswith("cal") and "tempdip" not in name and "region" not in name and "area" not in name and "opening" not in name:
        return "Does partial leap/calendar normalization improve public baseline direction?"
    if "tempdip" in name:
        return "Does weak-February temporary-dip rebound improve public direction?"
    if "region" in name or "area" in name or "opening" in name:
        return "Do heavily shrunk cohort seasonal factors transfer to public?"
    if "mom" in name:
        return "Does changing conservative momentum exponent improve public direction?"
    return "General heuristic direction test"


H_RESULTS_DIR = RESULTS_DIR
H_SUBMISSIONS_DIR = SUBMISSIONS_DIR
H_TARGET = TARGET
H_HeuristicSpec = HeuristicSpec
H_STATIC_CAT_COLS = STATIC_CAT_COLS
H_STATIC_NUM_COLS = STATIC_NUM_COLS
H_build_target_frame = build_target_frame
H_factors_for_spec = factors_for_spec
H_load_data = load_data
H_mape = mape
H_predict_from_spec = predict_from_spec
H_safe_div = safe_div
H_label = label
H_period_label = period_label


### Helper C. Baseline search v3 definitions


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")


import numpy as np
import pandas as pd



BASE_SPEC = HeuristicSpec(70, "cal050_median_mom000", 0.50, momentum_exp=0.0)


def adjusted_store_ratio(frame: pd.DataFrame, target_non_leap: bool, calendar_lambda: float) -> pd.Series:
    ratio = safe_div(frame["lag12"], frame["lag13"])
    if target_non_leap:
        return ratio * ((1.0 - calendar_lambda) + calendar_lambda * (29.0 / 28.0))
    return ratio * ((1.0 - calendar_lambda) + calendar_lambda * (28.0 / 29.0))


def reliability(frame: pd.DataFrame, mode: str, volume_k: float) -> np.ndarray:
    if mode == "constant":
        return np.ones(len(frame), dtype=float)
    if mode == "volume":
        return (frame["lag13"].to_numpy(float) / (frame["lag13"].to_numpy(float) + volume_k)).clip(0.0, 1.0)
    if mode == "stable_volume":
        volume = (frame["lag13"].to_numpy(float) / (frame["lag13"].to_numpy(float) + volume_k)).clip(0.0, 1.0)
        stable = (1.0 - frame["volatility_rel3"].fillna(0.0).to_numpy(float) / 0.12).clip(0.0, 1.0)
        return volume * stable
    raise ValueError(mode)


def store_seasonal_pred(
    frame: pd.DataFrame,
    base_pred: np.ndarray,
    global_factor: float,
    target_non_leap: bool,
    calendar_lambda: float,
    weight: float,
    log_clip: float,
    rel_mode: str,
    volume_k: float,
    extra_gate: np.ndarray | None = None,
) -> np.ndarray:
    ratio = adjusted_store_ratio(frame, target_non_leap, calendar_lambda).fillna(global_factor).clip(0.85, 1.45)
    log_delta = np.log(ratio.to_numpy(float) / global_factor)
    log_delta = np.clip(log_delta, -log_clip, log_clip)
    rel = reliability(frame, rel_mode, volume_k)
    if extra_gate is not None:
        rel = rel * extra_gate.astype(float)
    return base_pred * np.exp(weight * rel * log_delta)


def store_gate(frame: pd.DataFrame, momentum_min: float | None, vol_max: float | None, avoid_structural: bool) -> np.ndarray:
    gate = np.ones(len(frame), dtype=bool)
    if momentum_min is not None:
        gate &= frame["momentum_raw"].fillna(1.0).to_numpy(float) >= momentum_min
    if vol_max is not None:
        gate &= frame["volatility_rel3"].fillna(0.0).to_numpy(float) <= vol_max
    if avoid_structural:
        gate &= frame["structural_decline_score"].to_numpy(float) <= 0
    return gate.astype(float)


def yoy_modulation_pred(
    frame: pd.DataFrame,
    base_pred: np.ndarray,
    target_non_leap: bool,
    beta: float,
    log_clip: float,
) -> np.ndarray:
    if target_non_leap:
        growth = safe_div(frame["lag1"], frame["lag13"]) * (29.0 / 28.0)
    else:
        growth = safe_div(frame["lag1"], frame["lag13"]) * (28.0 / 29.0)
    growth = growth.replace([np.inf, -np.inf], np.nan).fillna(growth.median()).clip(0.75, 1.60)
    centered = np.log(growth / growth.median())
    return base_pred * np.exp(beta * np.clip(centered.to_numpy(float), -log_clip, log_clip))


def state_adjustment_pred(base_pred: np.ndarray, frame: pd.DataFrame, dip_lift: float, structural_cut: float, vol_cut: float) -> np.ndarray:
    pred = base_pred.copy()
    pred *= np.exp(dip_lift * frame["temporary_dip_score"].to_numpy(float))
    pred *= np.exp(-structural_cut * frame["structural_decline_score"].to_numpy(float))
    if vol_cut:
        vol = frame["volatility_rel3"].fillna(0.0).to_numpy(float)
        pred *= np.exp(-vol_cut * np.clip(vol - 0.07, 0.0, 0.20))
    return pred


def add_row(rows: list[dict], name: str, family: str, val_pred: np.ndarray, test_pred: np.ndarray, val_frame: pd.DataFrame, base_val: np.ndarray, base_test: np.ndarray, params: dict) -> None:
    rows.append(
        {
            "name": name,
            "family": family,
            "proxy_mape_march2024": mape(val_frame[TARGET], val_pred),
            "proxy_delta_vs_base": mape(val_frame[TARGET], val_pred) - mape(val_frame[TARGET], base_val),
            "val_mean_abs_change_vs_base_pct": float(np.mean(np.abs(val_pred / base_val - 1.0)) * 100),
            "val_p95_abs_change_vs_base_pct": float(np.quantile(np.abs(val_pred / base_val - 1.0), 0.95) * 100),
            "test_mean_abs_change_vs_base_pct": float(np.mean(np.abs(test_pred / base_test - 1.0)) * 100),
            "test_p95_abs_change_vs_base_pct": float(np.quantile(np.abs(test_pred / base_test - 1.0), 0.95) * 100),
            "test_mean_rto": float(np.mean(test_pred)),
            **params,
        }
    )


def segment_diagnostics(summary: pd.DataFrame, val_frame: pd.DataFrame, predictions: dict[str, np.ndarray]) -> pd.DataFrame:
    segments = {
        "all": np.ones(len(val_frame), dtype=bool),
        "weak_momentum_q20": val_frame["momentum_raw"] <= val_frame["momentum_raw"].quantile(0.20),
        "strong_momentum_q80": val_frame["momentum_raw"] >= val_frame["momentum_raw"].quantile(0.80),
        "high_vol_q80": val_frame["volatility_rel3"] >= val_frame["volatility_rel3"].quantile(0.80),
        "low_vol_q20": val_frame["volatility_rel3"] <= val_frame["volatility_rel3"].quantile(0.20),
        "temp_dip": val_frame["temporary_dip_score"] > 0,
        "struct_decline": val_frame["structural_decline_score"] > 0,
    }
    rows = []
    for name, pred in predictions.items():
        for seg_name, mask in segments.items():
            mask = np.asarray(mask, dtype=bool)
            if mask.sum() < 50:
                continue
            rows.append(
                {
                    "name": name,
                    "segment": seg_name,
                    "n": int(mask.sum()),
                    "proxy_mape": mape(val_frame.loc[mask, TARGET], pred[mask]),
                }
            )
    return pd.DataFrame(rows)


def main() -> None:
    RESULTS_DIR.mkdir(exist_ok=True)
    _raw, rto, static = load_data()
    val_frame = build_target_frame(rto, static, 2024, 3, include_target=True)
    test_frame = build_target_frame(rto, static, 2025, 3, include_target=False)

    val_factor, test_factor, val_global, final_global = factors_for_spec(BASE_SPEC, val_frame, test_frame, rto, static)
    base_val = predict_from_spec(val_frame, val_factor, BASE_SPEC)
    base_test = predict_from_spec(test_frame, test_factor, BASE_SPEC)

    rows: list[dict] = []
    keep_predictions = {"base_cal050_mom000": base_val}

    # Store-level March/February seasonality memory, shrunk toward the global factor.
    for calendar_lambda in [0.50, 0.60]:
        val_factor_l, test_factor_l, val_global_l, final_global_l = factors_for_spec(
            HeuristicSpec(0, "base", calendar_lambda, momentum_exp=0.0), val_frame, test_frame, rto, static
        )
        base_val_l = predict_from_spec(val_frame, val_factor_l, HeuristicSpec(0, "base", calendar_lambda, momentum_exp=0.0))
        base_test_l = predict_from_spec(test_frame, test_factor_l, HeuristicSpec(0, "base", calendar_lambda, momentum_exp=0.0))
        keep_predictions[f"base_cal{calendar_lambda:.2f}_mom000"] = base_val_l
        for weight in [0.10, 0.20, 0.30, 0.40, 0.50]:
            for log_clip in [0.03, 0.05, 0.08, 0.12]:
                for rel_mode, volume_k in [("constant", 1.0), ("volume", 50_000_000.0), ("volume", 150_000_000.0), ("stable_volume", 100_000_000.0)]:
                    val_pred = store_seasonal_pred(val_frame, base_val_l, val_global_l, False, calendar_lambda, weight, log_clip, rel_mode, volume_k)
                    test_pred = store_seasonal_pred(test_frame, base_test_l, final_global_l, True, calendar_lambda, weight, log_clip, rel_mode, volume_k)
                    name = f"cal{calendar_lambda:.2f}_storeseason_w{weight:.2f}_clip{log_clip:.2f}_{rel_mode}_k{int(volume_k/1e6)}m"
                    add_row(
                        rows,
                        name,
                        "store_seasonality_shrink",
                        val_pred,
                        test_pred,
                        val_frame,
                        base_val,
                        base_test,
                        {
                            "calendar_lambda": calendar_lambda,
                            "weight": weight,
                            "log_clip": log_clip,
                            "rel_mode": rel_mode,
                            "volume_k": volume_k,
                        },
                    )
                    keep_predictions[name] = val_pred

        for weight in [0.30, 0.50, 0.70]:
            for log_clip in [0.03, 0.05, 0.08]:
                for momentum_min in [0.96, 0.98, 1.00, 1.02, None]:
                    for vol_max in [0.08, 0.10, 0.12, None]:
                        if momentum_min is None and vol_max is None:
                            continue
                        val_gate = store_gate(val_frame, momentum_min, vol_max, avoid_structural=True)
                        test_gate = store_gate(test_frame, momentum_min, vol_max, avoid_structural=True)
                        val_pred = store_seasonal_pred(
                            val_frame,
                            base_val_l,
                            val_global_l,
                            False,
                            calendar_lambda,
                            weight,
                            log_clip,
                            "stable_volume",
                            100_000_000.0,
                            extra_gate=val_gate,
                        )
                        test_pred = store_seasonal_pred(
                            test_frame,
                            base_test_l,
                            final_global_l,
                            True,
                            calendar_lambda,
                            weight,
                            log_clip,
                            "stable_volume",
                            100_000_000.0,
                            extra_gate=test_gate,
                        )
                        mom_label = "none" if momentum_min is None else f"{momentum_min:.2f}"
                        vol_label = "none" if vol_max is None else f"{vol_max:.2f}"
                        name = f"cal{calendar_lambda:.2f}_storeseason_gated_w{weight:.2f}_clip{log_clip:.2f}_mom{mom_label}_vol{vol_label}"
                        add_row(
                            rows,
                            name,
                            "store_seasonality_gated",
                            val_pred,
                            test_pred,
                            val_frame,
                            base_val,
                            base_test,
                            {
                                "calendar_lambda": calendar_lambda,
                                "weight": weight,
                                "log_clip": log_clip,
                                "momentum_min": momentum_min if momentum_min is not None else np.nan,
                                "vol_max": vol_max if vol_max is not None else np.nan,
                                "avoid_structural": True,
                                "val_gate_share": float(val_gate.mean()),
                                "test_gate_share": float(test_gate.mean()),
                            },
                        )
                        keep_predictions[name] = val_pred

    for beta in [-0.20, -0.10, -0.05, 0.05, 0.10, 0.20]:
        for log_clip in [0.05, 0.10, 0.20]:
            val_pred = yoy_modulation_pred(val_frame, base_val, False, beta, log_clip)
            test_pred = yoy_modulation_pred(test_frame, base_test, True, beta, log_clip)
            name = f"cal050_yoymod_beta{beta:+.2f}_clip{log_clip:.2f}"
            add_row(
                rows,
                name,
                "feb_yoy_modulation",
                val_pred,
                test_pred,
                val_frame,
                base_val,
                base_test,
                {"beta": beta, "log_clip": log_clip},
            )
            keep_predictions[name] = val_pred

    for dip_lift in [0.00, 0.01, 0.02, 0.03, 0.05]:
        for structural_cut in [0.00, 0.01, 0.02, 0.03]:
            for vol_cut in [0.00, 0.05, 0.10]:
                if dip_lift == 0 and structural_cut == 0 and vol_cut == 0:
                    continue
                val_pred = state_adjustment_pred(base_val, val_frame, dip_lift, structural_cut, vol_cut)
                test_pred = state_adjustment_pred(base_test, test_frame, dip_lift, structural_cut, vol_cut)
                name = f"cal050_state_dip{dip_lift:.2f}_struct{structural_cut:.2f}_vol{vol_cut:.2f}"
                add_row(
                    rows,
                    name,
                    "state_adjustment",
                    val_pred,
                    test_pred,
                    val_frame,
                    base_val,
                    base_test,
                    {"dip_lift": dip_lift, "structural_cut": structural_cut, "vol_cut": vol_cut},
                )
                keep_predictions[name] = val_pred

    summary = pd.DataFrame(rows).sort_values("proxy_mape_march2024").reset_index(drop=True)
    summary.to_csv(RESULTS_DIR / "baseline_heuristics_v3_search.csv", index=False)
    summary.head(50).to_csv(RESULTS_DIR / "baseline_heuristics_v3_top50.csv", index=False)

    best_names = ["base_cal050_mom000"] + summary.head(12)["name"].tolist()
    for row in summary.head(12).itertuples(index=False):
        if row.name not in keep_predictions:
            pass
    seg = segment_diagnostics(summary, val_frame, {k: v for k, v in keep_predictions.items() if k in best_names or k == "base_cal050_mom000"})
    seg.to_csv(RESULTS_DIR / "baseline_heuristics_v3_segment_diagnostics.csv", index=False)

    print("Top v3 heuristic baselines:")
    print(
        summary[
            [
                "name",
                "family",
                "proxy_mape_march2024",
                "proxy_delta_vs_base",
                "val_mean_abs_change_vs_base_pct",
                "test_mean_abs_change_vs_base_pct",
                "test_mean_rto",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )
    print("\nFamily best:")
    print(
        summary.groupby("family", as_index=False)
        .first()[["family", "name", "proxy_mape_march2024", "proxy_delta_vs_base", "test_mean_abs_change_vs_base_pct"]]
        .sort_values("proxy_mape_march2024")
        .to_string(index=False)
    )


### Helper D. Baseline search v4 definitions


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")


import calendar
from datetime import date

import numpy as np
import pandas as pd



RU_HOLIDAYS = {(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (2, 23), (3, 8), (5, 1), (5, 9), (6, 12), (11, 4)}


def month_exposure(year: int, month: int, weekend_weight: float, holiday_weight: float) -> float:
    exposure = 0.0
    for day in range(1, calendar.monthrange(year, month)[1] + 1):
        dt = date(year, month, day)
        if (month, day) in RU_HOLIDAYS and dt.weekday() < 5:
            exposure += holiday_weight
        elif dt.weekday() >= 5:
            exposure += weekend_weight
        else:
            exposure += 1.0
    return exposure


def exposure_factor(rto: pd.DataFrame, target_year: int, weekend_weight: float, holiday_weight: float) -> float:
    ratios = []
    target_exp = month_exposure(target_year, 3, weekend_weight, holiday_weight) / month_exposure(target_year, 2, weekend_weight, holiday_weight)
    for year in [2023, 2024]:
        if year >= target_year:
            continue
        ratio = safe_div(rto[f"{year}-03"], rto[f"{year}-02"])
        source_exp = month_exposure(year, 3, weekend_weight, holiday_weight) / month_exposure(year, 2, weekend_weight, holiday_weight)
        ratios.append(ratio / source_exp)
    return float(pd.concat(ratios).dropna().median() * target_exp)


def current_191(frame: pd.DataFrame, factor: float, target_non_leap: bool) -> np.ndarray:
    base = frame["lag1"].to_numpy(float) * factor
    gate = store_gate(frame, None, 0.12, avoid_structural=True)
    return store_seasonal_pred(
        frame,
        base,
        factor,
        target_non_leap=target_non_leap,
        calendar_lambda=0.575,
        weight=0.70,
        log_clip=0.03,
        rel_mode="stable_volume",
        volume_k=100_000_000.0,
        extra_gate=gate,
    )


def factor_for_lambda(rto: pd.DataFrame, static: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame, calendar_lambda: float) -> tuple[float, float]:
    spec = HeuristicSpec(0, "base", calendar_lambda, momentum_exp=0.0)
    _vf, _tf, val_global, final_global = factors_for_spec(spec, val, test, rto, static)
    return val_global, final_global


def residual_diagnostics(val: pd.DataFrame, pred: np.ndarray) -> None:
    frame = val.copy()
    frame["pred"] = pred
    frame["log_residual"] = np.log(frame[TARGET].to_numpy(float) / pred)
    frame["ape"] = np.abs((frame[TARGET].to_numpy(float) - pred) / frame[TARGET].to_numpy(float)) * 100
    numeric_features = [
        "momentum_raw",
        "volatility_rel3",
        "lag1_to_lag12",
        "lag1_to_lag13",
        "store_mar_feb_ratio_prev",
        "temporary_dip_score",
        "structural_decline_score",
        "lag1",
    ]
    for col in [c for c in STATIC_NUM_COLS if c in frame.columns]:
        numeric_features.append(col)
    rows = []
    for col in numeric_features:
        values = frame[col].replace([np.inf, -np.inf], np.nan)
        if values.notna().sum() < 500 or values.nunique(dropna=True) < 5:
            continue
        try:
            bins = pd.qcut(values, 5, duplicates="drop")
        except ValueError:
            continue
        tmp = frame.assign(_bin=bins)
        for bin_name, part in tmp.groupby("_bin", observed=True):
            if len(part) < 100:
                continue
            rows.append(
                {
                    "feature": col,
                    "bin": str(bin_name),
                    "n": len(part),
                    "mean_feature": float(part[col].mean()),
                    "mean_log_residual": float(part["log_residual"].mean()),
                    "median_log_residual": float(part["log_residual"].median()),
                    "mape": mape(part[TARGET], part["pred"]),
                    "mean_ape": float(part["ape"].mean()),
                }
            )
    for col in [c for c in STATIC_CAT_COLS if c in frame.columns]:
        stats = frame.groupby(col, observed=True).agg(
            n=(TARGET, "size"),
            mean_log_residual=("log_residual", "mean"),
            median_log_residual=("log_residual", "median"),
            mean_ape=("ape", "mean"),
        )
        stats = stats[stats["n"] >= 200].sort_values("mean_ape", ascending=False).head(40).reset_index()
        stats.insert(0, "feature", col)
        stats.to_csv(RESULTS_DIR / f"baseline_v4_residual_cat_{col}.csv", index=False)
    pd.DataFrame(rows).to_csv(RESULTS_DIR / "baseline_v4_residual_numeric_bins.csv", index=False)


def add_row(rows: list[dict], name: str, family: str, val_pred: np.ndarray, test_pred: np.ndarray, val: pd.DataFrame, ref_val: np.ndarray, ref_test: np.ndarray, params: dict) -> None:
    rows.append(
        {
            "name": name,
            "family": family,
            "proxy_mape_march2024": mape(val[TARGET], val_pred),
            "delta_vs_191_proxy": mape(val[TARGET], val_pred) - mape(val[TARGET], ref_val),
            "val_mean_abs_change_vs_191_pct": float(np.mean(np.abs(val_pred / ref_val - 1.0)) * 100),
            "test_mean_abs_change_vs_191_pct": float(np.mean(np.abs(test_pred / ref_test - 1.0)) * 100),
            "test_p95_abs_change_vs_191_pct": float(np.quantile(np.abs(test_pred / ref_test - 1.0), 0.95) * 100),
            "test_mean_rto": float(np.mean(test_pred)),
            **params,
        }
    )


def sign_asym_store_pred(frame: pd.DataFrame, base: np.ndarray, factor: float, target_non_leap: bool, calendar_lambda: float, w_pos: float, w_neg: float, clip_pos: float, clip_neg: float, vol_max: float | None) -> np.ndarray:
    ratio = safe_div(frame["lag12"], frame["lag13"])
    if target_non_leap:
        ratio *= (1.0 - calendar_lambda) + calendar_lambda * (29.0 / 28.0)
    else:
        ratio *= (1.0 - calendar_lambda) + calendar_lambda * (28.0 / 29.0)
    ratio = ratio.replace([np.inf, -np.inf], np.nan).fillna(factor).clip(0.85, 1.45)
    delta = np.log(ratio.to_numpy(float) / factor)
    pos = np.clip(delta, 0.0, clip_pos)
    neg = np.clip(delta, -clip_neg, 0.0)
    volume = (frame["lag13"].to_numpy(float) / (frame["lag13"].to_numpy(float) + 100_000_000.0)).clip(0.0, 1.0)
    stable = (1.0 - frame["volatility_rel3"].fillna(0.0).to_numpy(float) / 0.12).clip(0.0, 1.0)
    rel = volume * stable
    if vol_max is not None:
        rel *= (frame["volatility_rel3"].fillna(0.0).to_numpy(float) <= vol_max).astype(float)
    rel *= (frame["structural_decline_score"].to_numpy(float) <= 0).astype(float)
    return base * np.exp(rel * (w_pos * pos + w_neg * neg))


def group_ratio_factor(frame: pd.DataFrame, base_factor: float, group_col: str, shrink_k: float, calendar_lambda: float, target_non_leap: bool) -> pd.Series:
    ratio = safe_div(frame["lag12"], frame["lag13"])
    if target_non_leap:
        ratio *= (1.0 - calendar_lambda) + calendar_lambda * (29.0 / 28.0)
    else:
        ratio *= (1.0 - calendar_lambda) + calendar_lambda * (28.0 / 29.0)
    tmp = pd.DataFrame({"ratio": ratio, group_col: frame[group_col]})
    stats = tmp.replace([np.inf, -np.inf], np.nan).dropna().groupby(group_col, observed=True)["ratio"].agg(["median", "count"])
    correction = np.log(stats["median"] / base_factor) * stats["count"] / (stats["count"] + shrink_k)
    return pd.Series(np.exp(frame[group_col].map(correction).fillna(0.0).to_numpy(float)), index=frame.index)


def main() -> None:
    RESULTS_DIR.mkdir(exist_ok=True)
    _raw, rto, static = load_data()
    val = build_target_frame(rto, static, 2024, 3, include_target=True)
    test = build_target_frame(rto, static, 2025, 3, include_target=False)
    val_factor_0575, final_factor_0575 = factor_for_lambda(rto, static, val, test, 0.575)
    ref_val = current_191(val, val_factor_0575, target_non_leap=False)
    ref_test = current_191(test, final_factor_0575, target_non_leap=True)
    residual_diagnostics(val, ref_val)

    rows: list[dict] = []
    base_cache: dict[float, tuple[np.ndarray, np.ndarray, float, float]] = {}
    for lam in [0.525, 0.55, 0.575, 0.60, 0.625]:
        vf, ff = factor_for_lambda(rto, static, val, test, lam)
        base_cache[lam] = (val["lag1"].to_numpy(float) * vf, test["lag1"].to_numpy(float) * ff, vf, ff)

    for lam in [0.55, 0.575, 0.60]:
        base_val, base_test, vf, ff = base_cache[lam]
        for w_pos in [0.4, 0.7, 1.0]:
            for w_neg in [0.2, 0.5, 0.7, 1.0]:
                for clip_pos, clip_neg in [(0.03, 0.03), (0.04, 0.02), (0.02, 0.04), (0.05, 0.03)]:
                    val_pred = sign_asym_store_pred(val, base_val, vf, False, lam, w_pos, w_neg, clip_pos, clip_neg, 0.12)
                    test_pred = sign_asym_store_pred(test, base_test, ff, True, lam, w_pos, w_neg, clip_pos, clip_neg, 0.12)
                    add_row(rows, f"signasym_lam{lam:.3f}_wp{w_pos:.1f}_wn{w_neg:.1f}_cp{clip_pos:.2f}_cn{clip_neg:.2f}", "sign_asymmetric_storeseason", val_pred, test_pred, val, ref_val, ref_test, {"calendar_lambda": lam, "w_pos": w_pos, "w_neg": w_neg, "clip_pos": clip_pos, "clip_neg": clip_neg})

    for weekend_weight, holiday_weight in [(1.0, 1.0), (1.1, 0.8), (1.2, 0.6), (1.2, 1.0), (1.2, 1.2)]:
        vf = exposure_factor(rto, 2024, weekend_weight, holiday_weight)
        ff = exposure_factor(rto, 2025, weekend_weight, holiday_weight)
        base_val = val["lag1"].to_numpy(float) * vf
        base_test = test["lag1"].to_numpy(float) * ff
        for weight in [0.5, 0.7]:
            val_pred = store_seasonal_pred(val, base_val, vf, False, 0.575, weight, 0.03, "stable_volume", 100_000_000.0, extra_gate=store_gate(val, None, 0.12, True))
            test_pred = store_seasonal_pred(test, base_test, ff, True, 0.575, weight, 0.03, "stable_volume", 100_000_000.0, extra_gate=store_gate(test, None, 0.12, True))
            add_row(rows, f"exposure_ww{weekend_weight:.1f}_hw{holiday_weight:.1f}_store_w{weight:.1f}", "exposure_plus_storeseason", val_pred, test_pred, val, ref_val, ref_test, {"weekend_weight": weekend_weight, "holiday_weight": holiday_weight, "weight": weight})

    for group_col in [c for c in ["Регион", "Торговая площадь, категориальный", "Дата открытия, категориальный"] if c in val.columns]:
        for shrink_k in [500, 1000, 2000, 5000]:
            mult_val = group_ratio_factor(val, val_factor_0575, group_col, shrink_k, 0.575, False)
            mult_test = group_ratio_factor(test, final_factor_0575, group_col, shrink_k, 0.575, True)
            for blend in [0.10, 0.20, 0.30]:
                val_pred = ref_val * np.exp(blend * np.log(mult_val.to_numpy(float)))
                test_pred = ref_test * np.exp(blend * np.log(mult_test.to_numpy(float)))
                add_row(rows, f"group_{group_col}_k{shrink_k}_b{blend:.2f}", "group_seasonality_blend", val_pred, test_pred, val, ref_val, ref_test, {"group_col": group_col, "shrink_k": shrink_k, "blend": blend})

    for weak_cut in [0.935, 0.945, 0.955]:
        weak = (val["momentum_raw"].fillna(1.0).to_numpy(float) < weak_cut).astype(float)
        weak_t = (test["momentum_raw"].fillna(1.0).to_numpy(float) < weak_cut).astype(float)
        for lift in [-0.015, -0.010, 0.010, 0.015]:
            val_pred = ref_val * np.exp(lift * weak)
            test_pred = ref_test * np.exp(lift * weak_t)
            add_row(rows, f"weak_mom_cut{weak_cut:.3f}_lift{lift:+.3f}", "state_residual_adjustment", val_pred, test_pred, val, ref_val, ref_test, {"weak_cut": weak_cut, "lift": lift})

    summary = pd.DataFrame(rows).sort_values("proxy_mape_march2024").reset_index(drop=True)
    summary.to_csv(RESULTS_DIR / "baseline_heuristics_v4_search.csv", index=False)
    summary.head(60).to_csv(RESULTS_DIR / "baseline_heuristics_v4_top60.csv", index=False)

    print("Current 191 proxy MAPE:", mape(val[TARGET], ref_val))
    print("Top v4 heuristics:")
    print(summary[["name", "family", "proxy_mape_march2024", "delta_vs_191_proxy", "test_mean_abs_change_vs_191_pct", "test_p95_abs_change_vs_191_pct", "test_mean_rto"]].head(40).to_string(index=False))
    print("\nFamily best:")
    print(summary.groupby("family", as_index=False).first()[["family", "name", "proxy_mape_march2024", "delta_vs_191_proxy", "test_mean_abs_change_vs_191_pct"]].sort_values("proxy_mape_march2024").to_string(index=False))


### Helper E. Multi-period validation definitions


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")


import calendar
from dataclasses import dataclass

import numpy as np
import pandas as pd



VALID_PERIODS = [(2024, 3), (2024, 11), (2024, 12), (2025, 1), (2025, 2)]


@dataclass(frozen=True)
class StoreSeasonSpec:
    name: str
    calendar_lambda: float
    weight: float
    log_clip: float
    vol_max: float | None
    avoid_structural: bool


def label(year: int, month: int) -> str:
    return f"{year:04d}-{month:02d}"


def shift(year: int, month: int, months_back: int) -> tuple[int, int]:
    stamp = pd.Timestamp(year=year, month=month, day=1) - pd.DateOffset(months=months_back)
    return int(stamp.year), int(stamp.month)


def adjusted_ratio(ratio: pd.Series, source_year: int, target_year: int, target_month: int, calendar_lambda: float) -> pd.Series:
    if target_month != 3:
        return ratio
    source_feb_days = calendar.monthrange(source_year, 2)[1]
    target_feb_days = calendar.monthrange(target_year, 2)[1]
    adjustment = (1.0 - calendar_lambda) + calendar_lambda * (source_feb_days / target_feb_days)
    return ratio * adjustment


def target_frame(rto: pd.DataFrame, year: int, month: int) -> pd.DataFrame:
    lag1_y, lag1_m = shift(year, month, 1)
    lag2_y, lag2_m = shift(year, month, 2)
    lag3_y, lag3_m = shift(year, month, 3)
    lag12_y, lag12_m = shift(year, month, 12)
    lag13_y, lag13_m = shift(year, month, 13)
    frame = pd.DataFrame(index=rto.index)
    frame["new_id"] = frame.index.astype(int)
    frame["lag1"] = rto[label(lag1_y, lag1_m)]
    frame["lag2"] = rto[label(lag2_y, lag2_m)]
    frame["lag3"] = rto[label(lag3_y, lag3_m)]
    frame["lag12"] = rto[label(lag12_y, lag12_m)]
    frame["lag13"] = rto[label(lag13_y, lag13_m)]
    frame[TARGET] = rto[label(year, month)]
    frame["rolling3_mean"] = frame[["lag1", "lag2", "lag3"]].mean(axis=1)
    frame["rolling2_pre_mean"] = frame[["lag2", "lag3"]].mean(axis=1)
    frame["momentum_raw"] = safe_div(frame["lag1"], frame["rolling3_mean"])
    frame["momentum_clip"] = frame["momentum_raw"].fillna(1.0).clip(0.95, 1.05)
    frame["feb_to_jan"] = safe_div(frame["lag1"], frame["lag2"])
    frame["jan_to_dec"] = safe_div(frame["lag2"], frame["lag3"])
    frame["lag1_to_lag12"] = safe_div(frame["lag1"], frame["lag12"])
    frame["volatility_rel3"] = frame[["lag1", "lag2", "lag3"]].std(axis=1) / frame["rolling3_mean"]
    frame["structural_decline_score"] = (
        frame["feb_to_jan"].fillna(1.0).lt(0.97)
        & frame["jan_to_dec"].fillna(1.0).lt(0.99)
        & frame["lag1_to_lag12"].fillna(1.0).lt(1.02)
    ).astype(float)
    return frame.reset_index(drop=True)


def historical_factor(rto: pd.DataFrame, target_year: int, target_month: int, calendar_lambda: float) -> float:
    ratios = []
    for source_year in sorted({int(c[:4]) for c in rto.columns if c.endswith(f"-{target_month:02d}")}):
        if source_year >= target_year:
            continue
        prev_y, prev_m = shift(source_year, target_month, 1)
        source = label(source_year, target_month)
        prev = label(prev_y, prev_m)
        if source not in rto.columns or prev not in rto.columns:
            continue
        ratio = safe_div(rto[source], rto[prev])
        ratios.append(adjusted_ratio(ratio, source_year, target_year, target_month, calendar_lambda))
    if not ratios:
        raise ValueError(f"No historical ratios for {target_year}-{target_month:02d}")
    return float(pd.concat(ratios).replace([np.inf, -np.inf], np.nan).dropna().median())


def seasonal_pred(frame: pd.DataFrame, factor: float, momentum_exp: float) -> np.ndarray:
    return frame["lag1"].to_numpy(float) * factor * frame["momentum_clip"].to_numpy(float) ** momentum_exp


def storeseason_pred(frame: pd.DataFrame, base_pred: np.ndarray, global_factor: float, year: int, month: int, spec: StoreSeasonSpec) -> np.ndarray:
    source_year = year - 1
    store_ratio = adjusted_ratio(safe_div(frame["lag12"], frame["lag13"]), source_year, year, month, spec.calendar_lambda)
    store_ratio = store_ratio.replace([np.inf, -np.inf], np.nan).fillna(global_factor).clip(0.85, 1.45)
    log_delta = np.clip(np.log(store_ratio.to_numpy(float) / global_factor), -spec.log_clip, spec.log_clip)
    volume = (frame["lag13"].to_numpy(float) / (frame["lag13"].to_numpy(float) + 100_000_000.0)).clip(0.0, 1.0)
    stable = (1.0 - frame["volatility_rel3"].fillna(0.0).to_numpy(float) / 0.12).clip(0.0, 1.0)
    rel = volume * stable
    if spec.vol_max is not None:
        rel *= (frame["volatility_rel3"].fillna(0.0).to_numpy(float) <= spec.vol_max).astype(float)
    if spec.avoid_structural:
        rel *= (frame["structural_decline_score"].to_numpy(float) <= 0).astype(float)
    return base_pred * np.exp(spec.weight * rel * log_delta)


def main() -> None:
    RESULTS_DIR.mkdir(exist_ok=True)
    _raw, rto, _static = load_data()
    store_specs = [
        StoreSeasonSpec("store_v3_188_like", 0.60, 0.70, 0.03, 0.12, True),
        StoreSeasonSpec("store_v3_189_like", 0.60, 0.50, 0.05, 0.12, True),
        StoreSeasonSpec("store_v3_190_like", 0.60, 0.50, 0.05, None, True),
    ]
    rows = []
    for year, month in VALID_PERIODS:
        frame = target_frame(rto, year, month)
        y = frame[TARGET]
        factors = {
            "seasonal_lam000_mom025": historical_factor(rto, year, month, 0.00),
            "seasonal_lam050_mom000": historical_factor(rto, year, month, 0.50),
            "seasonal_lam060_mom000": historical_factor(rto, year, month, 0.60),
            "seasonal_lam060_mom025": historical_factor(rto, year, month, 0.60),
        }
        preds = {
            "seasonal_lam000_mom025": seasonal_pred(frame, factors["seasonal_lam000_mom025"], 0.25),
            "seasonal_lam050_mom000": seasonal_pred(frame, factors["seasonal_lam050_mom000"], 0.0),
            "seasonal_lam060_mom000": seasonal_pred(frame, factors["seasonal_lam060_mom000"], 0.0),
            "seasonal_lam060_mom025": seasonal_pred(frame, factors["seasonal_lam060_mom025"], 0.25),
            "delta_yoy": frame["lag12"].to_numpy(float) + (frame["lag1"].to_numpy(float) - frame["lag13"].to_numpy(float)),
            "trend_yoy": frame["lag12"].to_numpy(float) * safe_div(frame["lag1"], frame["lag13"]).fillna(1.0).to_numpy(float),
            "lag1": frame["lag1"].to_numpy(float),
        }
        base60 = preds["seasonal_lam060_mom000"]
        for spec in store_specs:
            preds[spec.name] = storeseason_pred(frame, base60, factors["seasonal_lam060_mom000"], year, month, spec)
        for name, pred in preds.items():
            rows.append(
                {
                    "period": f"{year}-{month:02d}",
                    "name": name,
                    "rows": len(frame),
                    "factor": factors.get(name, np.nan),
                    "mape": mape(y, pred),
                    "mean_pred": float(np.mean(pred)),
                    "mean_abs_change_vs_lam060_mom000_pct": float(np.mean(np.abs(pred / base60 - 1.0)) * 100),
                }
            )
    detail = pd.DataFrame(rows)
    detail.to_csv(RESULTS_DIR / "baseline_multiperiod_validation_detail.csv", index=False)
    summary = (
        detail.groupby("name", as_index=False)
        .agg(
            avg_mape=("mape", "mean"),
            worst_mape=("mape", "max"),
            avg_abs_change_vs_lam060=("mean_abs_change_vs_lam060_mom000_pct", "mean"),
        )
        .sort_values("avg_mape")
        .reset_index(drop=True)
    )
    march = detail[detail["period"] == "2024-03"][["name", "mape"]].rename(columns={"mape": "march2024_mape"})
    summary = summary.merge(march, on="name", how="left")
    summary.to_csv(RESULTS_DIR / "baseline_multiperiod_validation_summary.csv", index=False)
    pivot = detail.pivot(index="name", columns="period", values="mape").reset_index()
    pivot.to_csv(RESULTS_DIR / "baseline_multiperiod_validation_pivot.csv", index=False)
    print("Multi-period baseline validation summary:")
    print(summary.to_string(index=False))
    print("\nPivot:")
    print(pivot.to_string(index=False))


### Helper F. Structural temporal204 definitions


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")


import calendar
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd




CURRENT_LAMBDA = 0.55


@dataclass(frozen=True)
class CandidateSpec:
    name: str
    family: str
    params: dict


def current_198_march(frame: pd.DataFrame, factor: float, target_non_leap: bool) -> np.ndarray:
    base = frame["lag1"].to_numpy(float) * factor
    return sign_asym_store_pred(
        frame,
        base,
        factor,
        target_non_leap=target_non_leap,
        calendar_lambda=CURRENT_LAMBDA,
        w_pos=0.2,
        w_neg=1.4,
        clip_pos=0.02,
        clip_neg=0.03,
        vol_max=0.12,
    )


def current_like_generic(frame: pd.DataFrame, factor: float, year: int, month: int) -> np.ndarray:
    base = frame["lag1"].to_numpy(float) * factor
    target_non_leap = month == 3 and calendar.monthrange(year, 2)[1] == 28
    return sign_asym_store_pred(
        frame,
        base,
        factor,
        target_non_leap=target_non_leap,
        calendar_lambda=CURRENT_LAMBDA,
        w_pos=0.2,
        w_neg=1.4,
        clip_pos=0.02,
        clip_neg=0.03,
        vol_max=0.12,
    )


def days_in_months(year: int, months: list[int]) -> int:
    return sum(calendar.monthrange(year, m)[1] for m in months)


def two_month_temporal_anchor(rto: pd.DataFrame, frame: pd.DataFrame, year: int, month: int, daily_adjust: bool, clip_lo: float, clip_hi: float) -> np.ndarray:
    prev1_y, prev1_m = shift(year, month, 1)
    prev2_y, prev2_m = shift(year, month, 2)
    py, pm = shift(year, month, 12)
    pprev1_y, pprev1_m = shift(year, month, 13)
    pprev2_y, pprev2_m = shift(year, month, 14)

    current_two = rto[label(prev1_y, prev1_m)] + rto[label(prev2_y, prev2_m)]
    previous_two = rto[label(pprev1_y, pprev1_m)] + rto[label(pprev2_y, pprev2_m)]
    growth = safe_div(current_two, previous_two)
    if daily_adjust:
        current_days = days_in_months(prev1_y, [prev1_m]) + days_in_months(prev2_y, [prev2_m])
        previous_days = days_in_months(pprev1_y, [pprev1_m]) + days_in_months(pprev2_y, [pprev2_m])
        growth *= previous_days / current_days
    growth = growth.replace([np.inf, -np.inf], np.nan).fillna(growth.median()).clip(clip_lo, clip_hi)
    anchor = rto[label(py, pm)] * growth
    return pd.Series(anchor, index=rto.index).reindex(frame["new_id"]).to_numpy(float)


def blend_anchor(base: np.ndarray, anchor: np.ndarray, alpha: float, log_clip: float) -> np.ndarray:
    ratio = np.log(np.maximum(anchor, 1.0) / np.maximum(base, 1.0))
    return base * np.exp(alpha * np.clip(ratio, -log_clip, log_clip))


def group_total_correction(
    frame: pd.DataFrame,
    pred: np.ndarray,
    group_col: str,
    target_year: int,
    target_month: int,
    source_years: list[int],
    rto: pd.DataFrame,
    alpha: float,
    log_clip: float,
    shrink_k: float,
) -> np.ndarray:
    tmp = frame[["new_id", group_col, "lag1"]].copy()
    tmp["pred"] = pred
    group_current = tmp.groupby(group_col, observed=True).agg(current_pred=("pred", "sum"), current_lag1=("lag1", "sum"), n=("new_id", "size"))

    ratio_rows = []
    static_group = frame.set_index("new_id")[group_col]
    for source_year in source_years:
        source = label(source_year, target_month)
        prev_y, prev_m = shift(source_year, target_month, 1)
        prev = label(prev_y, prev_m)
        if source not in rto.columns or prev not in rto.columns:
            continue
        source_frame = pd.DataFrame(
            {
                "new_id": rto.index.astype(int),
                "target": rto[source].to_numpy(float),
                "prev": rto[prev].to_numpy(float),
            }
        ).join(static_group, on="new_id", how="inner")
        group_sum = source_frame.groupby(group_col, observed=True).agg(target=("target", "sum"), prev=("prev", "sum"))
        ratio = safe_div(group_sum["target"], group_sum["prev"])
        if target_month == 3:
            source_feb_days = calendar.monthrange(source_year, 2)[1]
            target_feb_days = calendar.monthrange(target_year, 2)[1]
            ratio *= (1.0 - CURRENT_LAMBDA) + CURRENT_LAMBDA * (source_feb_days / target_feb_days)
        ratio_rows.append(ratio.rename(source_year))
    if not ratio_rows:
        return pred.copy()
    group_factor = pd.concat(ratio_rows, axis=1).median(axis=1)
    group_target = group_current["current_lag1"] * group_factor.reindex(group_current.index)
    corr = np.log(group_target / group_current["current_pred"]).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    shrink = group_current["n"] / (group_current["n"] + shrink_k)
    corr = np.clip(corr * shrink, -log_clip, log_clip)
    mapped = frame[group_col].map(corr).fillna(0.0).to_numpy(float)
    return pred * np.exp(alpha * mapped)


def bootstrap_delta(y: np.ndarray, ref: np.ndarray, pred: np.ndarray) -> dict:
    rng = np.random.default_rng(2026)
    ref_ape = np.abs((y - ref) / y) * 100
    ape = np.abs((y - pred) / y) * 100
    delta = ape - ref_ape
    idx = rng.integers(0, len(y), size=(1000, len(y)))
    boot = delta[idx].mean(axis=1)
    return {
        "boot_p05": float(np.quantile(boot, 0.05)),
        "boot_p50": float(np.quantile(boot, 0.50)),
        "boot_p95": float(np.quantile(boot, 0.95)),
        "prob_improve": float((boot < 0).mean()),
    }


def main() -> None:
    RESULTS_DIR.mkdir(exist_ok=True)
    _raw, rto, static = load_data()
    val = build_target_frame(rto, static, 2024, 3, include_target=True)
    test = build_target_frame(rto, static, 2025, 3, include_target=False)
    spec = HeuristicSpec(0, "current", CURRENT_LAMBDA, momentum_exp=0.0)
    _vf_series, _tf_series, val_factor, final_factor = factors_for_spec(spec, val, test, rto, static)
    ref_val = current_198_march(val, val_factor, target_non_leap=False)
    ref_test = current_198_march(test, final_factor, target_non_leap=True)

    rows = []
    prediction_cache: dict[str, tuple[np.ndarray, np.ndarray]] = {}

    # Temporal aggregation: target month from same month last year scaled by previous two months growth.
    for daily_adjust in [False, True]:
        for clip_lo, clip_hi in [(0.85, 1.35), (0.90, 1.25), (0.95, 1.20)]:
            anchor_val = two_month_temporal_anchor(rto, val, 2024, 3, daily_adjust, clip_lo, clip_hi)
            anchor_test = two_month_temporal_anchor(rto, test, 2025, 3, daily_adjust, clip_lo, clip_hi)
            for alpha in [0.03, 0.05, 0.08, 0.12, 0.18]:
                for log_clip in [0.02, 0.04, 0.06, 0.10]:
                    pred_val = blend_anchor(ref_val, anchor_val, alpha, log_clip)
                    pred_test = blend_anchor(ref_test, anchor_test, alpha, log_clip)
                    name = f"temporal2_daily{int(daily_adjust)}_clip{clip_lo:.2f}_{clip_hi:.2f}_a{alpha:.2f}_lc{log_clip:.2f}"
                    prediction_cache[name] = (pred_val, pred_test)
                    rows.append(
                        {
                            "name": name,
                            "family": "temporal_two_month_growth",
                            "proxy_mape_march2024": mape(val[TARGET], pred_val),
                            "delta_vs_198_proxy": mape(val[TARGET], pred_val) - mape(val[TARGET], ref_val),
                            "test_mean_abs_change_vs_198_pct": float(np.mean(np.abs(pred_test / ref_test - 1.0)) * 100),
                            "test_p95_abs_change_vs_198_pct": float(np.quantile(np.abs(pred_test / ref_test - 1.0), 0.95) * 100),
                            "test_mean_rto": float(np.mean(pred_test)),
                            "daily_adjust": daily_adjust,
                            "clip_lo": clip_lo,
                            "clip_hi": clip_hi,
                            "alpha": alpha,
                            "log_clip": log_clip,
                        }
                    )

    # Hierarchical/group total correction as a small multiplicative correction to current best baseline.
    for group_col in [c for c in STATIC_CAT_COLS if c in val.columns]:
        for alpha in [0.10, 0.20, 0.35, 0.50]:
            for log_clip in [0.01, 0.02, 0.035, 0.05]:
                for shrink_k in [200, 500, 1000, 2000]:
                    pred_val = group_total_correction(val, ref_val, group_col, 2024, 3, [2023], rto, alpha, log_clip, shrink_k)
                    pred_test = group_total_correction(test, ref_test, group_col, 2025, 3, [2023, 2024], rto, alpha, log_clip, shrink_k)
                    name = f"groupcorr_{group_col}_a{alpha:.2f}_lc{log_clip:.3f}_k{shrink_k}"
                    prediction_cache[name] = (pred_val, pred_test)
                    rows.append(
                        {
                            "name": name,
                            "family": "hierarchical_group_total_correction",
                            "proxy_mape_march2024": mape(val[TARGET], pred_val),
                            "delta_vs_198_proxy": mape(val[TARGET], pred_val) - mape(val[TARGET], ref_val),
                            "test_mean_abs_change_vs_198_pct": float(np.mean(np.abs(pred_test / ref_test - 1.0)) * 100),
                            "test_p95_abs_change_vs_198_pct": float(np.quantile(np.abs(pred_test / ref_test - 1.0), 0.95) * 100),
                            "test_mean_rto": float(np.mean(pred_test)),
                            "group_col": group_col,
                            "alpha": alpha,
                            "log_clip": log_clip,
                            "shrink_k": shrink_k,
                        }
                    )

    summary = pd.DataFrame(rows).sort_values("proxy_mape_march2024").reset_index(drop=True)
    # Bootstrap top candidates and multi-period sanity for the best few per family.
    top_names = []
    for family, part in summary.groupby("family", observed=True):
        top_names.extend(part.head(12)["name"].tolist())
    top_names = list(dict.fromkeys(top_names))

    boot_rows = []
    y = val[TARGET].to_numpy(float)
    for name in top_names:
        pred_val, _pred_test = prediction_cache[name]
        boot_rows.append({"name": name, **bootstrap_delta(y, ref_val, pred_val)})
    boot = pd.DataFrame(boot_rows)

    mp_rows = []
    for name in top_names:
        row = summary.loc[summary["name"] == name].iloc[0]
        vals = []
        for year, month in VALID_PERIODS:
            fr = target_frame(rto, year, month).join(static, on="new_id", how="left")
            factor = historical_factor(rto, year, month, CURRENT_LAMBDA)
            ref = current_like_generic(fr, factor, year, month)
            if row["family"] == "temporal_two_month_growth":
                anchor = two_month_temporal_anchor(rto, fr, year, month, bool(row["daily_adjust"]), float(row["clip_lo"]), float(row["clip_hi"]))
                pred = blend_anchor(ref, anchor, float(row["alpha"]), float(row["log_clip"]))
            else:
                # Multi-period group correction uses all available prior years for that target month.
                source_years = [sy for sy in [2023, 2024] if sy < year]
                pred = group_total_correction(fr, ref, str(row["group_col"]), year, month, source_years, rto, float(row["alpha"]), float(row["log_clip"]), float(row["shrink_k"]))
            vals.append(mape(fr[TARGET], pred))
        mp_rows.append({"name": name, "multiperiod_avg_mape": float(np.mean(vals)), "multiperiod_worst_mape": float(np.max(vals)), "multiperiod_march2024_mape": vals[0]})
    mp = pd.DataFrame(mp_rows)

    final = summary.merge(boot, on="name", how="left").merge(mp, on="name", how="left")
    final.to_csv(RESULTS_DIR / "structural_baselines_v6_search.csv", index=False)
    final.head(80).to_csv(RESULTS_DIR / "structural_baselines_v6_top80.csv", index=False)
    print("Current 198 proxy MAPE:", mape(val[TARGET], ref_val))
    print("Family best:")
    print(final.groupby("family", as_index=False).first()[["family", "name", "proxy_mape_march2024", "delta_vs_198_proxy", "test_mean_abs_change_vs_198_pct", "multiperiod_avg_mape", "boot_p95", "prob_improve"]].sort_values("proxy_mape_march2024").to_string(index=False))
    print("\nTop candidates:")
    print(final[["name", "family", "proxy_mape_march2024", "delta_vs_198_proxy", "test_mean_abs_change_vs_198_pct", "test_p95_abs_change_vs_198_pct", "multiperiod_avg_mape", "boot_p95", "prob_improve"]].head(40).to_string(index=False))


### Helper G. Transfer helper definitions



In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")


import numpy as np
import pandas as pd



MOMENTUM_TRAIN_BINS = [
    -np.inf,
    0.935766244455909,
    0.9524077370639855,
    0.9656151000316019,
    0.9815696834781942,
    np.inf,
]


def read_local_artifact_csv(path: str) -> pd.DataFrame:
    local_path = RESULTS_DIR.parents[0] / path
    if not local_path.exists():
        raise FileNotFoundError(f"Expected local artifact from earlier notebook cells: {local_path}")
    return pd.read_csv(local_path)


def momentum_bin(frame: pd.DataFrame) -> pd.Series:
    return pd.cut(
        frame["momentum_raw"],
        bins=MOMENTUM_TRAIN_BINS,
        labels=[f"q{i}" for i in range(5)],
        include_lowest=True,
    ).astype(str)


def centered_log_residual(log_residual: pd.Series, groups: pd.Series | None) -> pd.Series:
    if groups is None:
        return log_residual - float(log_residual.median())
    med = log_residual.groupby(groups, observed=True).transform("median")
    return log_residual - med.fillna(float(log_residual.median()))


def build_baseline(frame: pd.DataFrame, rto: pd.DataFrame, static: pd.DataFrame, spec: HeuristicSpec, is_test: bool) -> tuple[np.ndarray, float]:
    # factors_for_spec needs both frames but returns the requested side. Use the same frame for the unused side.
    val_factor, test_factor, val_global, final_global = factors_for_spec(spec, frame, frame, rto, static)
    factor = test_factor if is_test else val_factor
    global_factor = final_global if is_test else val_global
    return predict_from_spec(frame, factor, spec), global_factor


def main() -> None:
    RESULTS_DIR.mkdir(exist_ok=True)
    SUBMISSIONS_DIR.mkdir(exist_ok=True)
    _raw, rto, static = load_data()
    val_frame = build_target_frame(rto, static, 2024, 3, include_target=True)
    test_frame = build_target_frame(rto, static, 2025, 3, include_target=False)

    old_spec = HeuristicSpec(0, "old_raw_mom025", 0.0, momentum_exp=0.25)
    new_specs = {
        "cal050_mom000": HeuristicSpec(0, "cal050_mom000", 0.50, momentum_exp=0.0),
        "cal060_mom000": HeuristicSpec(0, "cal060_mom000", 0.60, momentum_exp=0.0),
    }

    old_val_base, _ = build_baseline(val_frame, rto, static, old_spec, is_test=False)
    old_test_base, _ = build_baseline(test_frame, rto, static, old_spec, is_test=True)

    oof = read_local_artifact_csv("results/gru_residual/gru_oof.csv")[["new_id", "pred_log_gru"]]
    val = val_frame.merge(oof, on="new_id", how="inner")
    val_old_base = pd.Series(old_val_base, index=val_frame["new_id"]).reindex(val["new_id"]).to_numpy(dtype=float)
    val["old_gru_ratio_log"] = 0.70 * val["pred_log_gru"].to_numpy(dtype=float)
    val["momentum_train_bin"] = momentum_bin(val)

    test_gru = read_local_artifact_csv("submissions/test_gru_alpha070.csv").rename(columns={"rto": "old_gru"})
    test = test_frame.merge(test_gru, on="new_id", how="inner")
    test_old_base = pd.Series(old_test_base, index=test_frame["new_id"]).reindex(test["new_id"]).to_numpy(dtype=float)
    test["old_gru_ratio_log"] = np.log(test["old_gru"].to_numpy(dtype=float) / test_old_base)
    test["momentum_train_bin"] = momentum_bin(test)

    specs = [
        (94, "cal050_mom000_gru_global_g010", "cal050_mom000", 0.10, "global"),
        (95, "cal050_mom000_gru_global_g020", "cal050_mom000", 0.20, "global"),
        (96, "cal050_mom000_gru_global_g035", "cal050_mom000", 0.35, "global"),
        (97, "cal050_mom000_gru_mombin_g020", "cal050_mom000", 0.20, "momentum_bin"),
        (98, "cal050_mom000_gru_mombin_g035", "cal050_mom000", 0.35, "momentum_bin"),
        (99, "cal060_mom000_gru_global_g020", "cal060_mom000", 0.20, "global"),
        (100, "cal060_mom000_gru_mombin_g020", "cal060_mom000", 0.20, "momentum_bin"),
        (101, "cal050_mom000_gru_mombin_g050", "cal050_mom000", 0.50, "momentum_bin"),
    ]

    rows = []
    for candidate_id, name, base_key, gamma, centering in specs:
        new_val_base, _ = build_baseline(val_frame, rto, static, new_specs[base_key], is_test=False)
        new_test_base, _ = build_baseline(test_frame, rto, static, new_specs[base_key], is_test=True)
        val_new_base = pd.Series(new_val_base, index=val_frame["new_id"]).reindex(val["new_id"]).to_numpy(dtype=float)
        test_new_base = pd.Series(new_test_base, index=test_frame["new_id"]).reindex(test["new_id"]).to_numpy(dtype=float)

        if centering == "momentum_bin":
            val_centered = centered_log_residual(val["old_gru_ratio_log"], val["momentum_train_bin"])
            test_centered = centered_log_residual(test["old_gru_ratio_log"], test["momentum_train_bin"])
        else:
            val_centered = centered_log_residual(val["old_gru_ratio_log"], None)
            test_centered = centered_log_residual(test["old_gru_ratio_log"], None)

        val_pred = val_new_base * np.exp(gamma * val_centered.to_numpy(dtype=float))
        test_pred = test_new_base * np.exp(gamma * test_centered.to_numpy(dtype=float))

        out = test[["new_id"]].copy()
        out["rto"] = np.round(np.maximum(test_pred, 0), 2)
        file_name = f"stage2_candidate_{candidate_id:03d}_baseline_{name}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)

        rows.append(
            {
                "candidate_id": candidate_id,
                "file": file_name,
                "name": name,
                "base_key": base_key,
                "gamma": gamma,
                "centering": centering,
                "val_mape_march2024": mape(val[TARGET], val_pred),
                "test_mean_rto": float(np.mean(test_pred)),
                "test_total_rto": float(np.sum(test_pred)),
                "mean_abs_change_vs_new_base_pct": float(np.mean(np.abs(test_pred / test_new_base - 1.0)) * 100),
                "p95_abs_change_vs_new_base_pct": float(np.quantile(np.abs(test_pred / test_new_base - 1.0), 0.95) * 100),
            }
        )

    summary = pd.DataFrame(rows).sort_values("val_mape_march2024").reset_index(drop=True)
    summary.to_csv(RESULTS_DIR / "new_baseline_residual_transfer_candidates.csv", index=False)

    log_path = RESULTS_DIR / "public_experiment_log.csv"
    if log_path.exists():
        log = pd.read_csv(log_path)
        existing = set(log["candidate_id"].astype(str))
        new_rows = []
        for _, row in summary.iterrows():
            if str(row["candidate_id"]) in existing:
                continue
            new_rows.append(
                {
                    "candidate_id": row["candidate_id"],
                    "file": row["file"],
                    "family": "new_baseline_residual_transfer",
                    "hypothesis": "Can a small centered old-GRU residual improve the new calendar/momentum-zero baseline?",
                    "public_score": "",
                    "notes": "fill public_score after submission",
                    "public_mape": "",
                    "delta_mape_vs_current_best": "",
                    "delta_score_vs_current_best": "",
                }
            )
        if new_rows:
            pd.concat([log, pd.DataFrame(new_rows)], ignore_index=True).to_csv(log_path, index=False)

    print("Wrote residual-transfer candidates:")
    print(summary.to_string(index=False))


# Preserve transfer-helper names imported by later stacking code in the original scripts.
G_MOMENTUM_TRAIN_BINS = MOMENTUM_TRAIN_BINS
G_read_local_artifact_csv = read_local_artifact_csv


## 1. Original GRU Residual

Первый GRU residual над исходной эвристикой: подготовка panel data, обучение GRU, OOF и генерация `test_gru_alpha070.csv`.


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



BEHAVIOR_COLS = CB_BEHAVIOR_COLS
CAT_COLS = CB_CAT_COLS
STATIC_NUM_COLS = CB_STATIC_NUM_COLS
SUBMISSIONS_DIR = CB_SUBMISSIONS_DIR
TARGET = CB_TARGET
TRAIN_PATH = CB_TRAIN_PATH
add_base = CB_add_base
apply_residual = CB_apply_residual
make_panel = CB_make_panel
mape = CB_mape

import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def make_loader_generator(seed: int) -> torch.Generator:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    return generator



ROOT = ROOT
RESULTS_DIR = ROOT / "results" / "gru_residual"
CATBOOST_OOF_PATH = ROOT / "results" / "catboost_feature_audit" / "ablation_oof.csv"
SEED = 2026
ALPHA_GRID = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
SEQ_LEN = 14
EPOCHS = 80
BATCH_SIZE = 512
PATIENCE = 10

SMALL_CAT_COLS = [
    "Дата открытия, категориальный",
    "Торговая площадь, категориальный",
    "Регион",
]


@dataclass
class PreparedData:
    raw: pd.DataFrame
    target_rows: pd.DataFrame
    validation_factor: float
    final_factor: float
    rto_wide: pd.DataFrame
    behavior_wide: dict[str, pd.DataFrame]
    static_frame: pd.DataFrame


class GRUResidual(nn.Module):
    def __init__(
        self,
        seq_dim: int,
        static_num_dim: int,
        cat_cardinalities: list[int],
        hidden_dim: int = 32,
        dropout: float = 0.15,
    ) -> None:
        super().__init__()
        self.gru = nn.GRU(seq_dim, hidden_dim, num_layers=1, batch_first=True)
        self.embeddings = nn.ModuleList()
        emb_dims = []
        for cardinality in cat_cardinalities:
            dim = min(12, max(2, int(np.sqrt(cardinality + 1)) + 1))
            self.embeddings.append(nn.Embedding(cardinality + 1, dim, padding_idx=0))
            emb_dims.append(dim)
        static_in = static_num_dim + sum(emb_dims)
        self.static_net = nn.Sequential(
            nn.Linear(static_in, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 16),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + 16, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, seq: torch.Tensor, static_num: torch.Tensor, cat: torch.Tensor) -> torch.Tensor:
        _, hidden = self.gru(seq)
        seq_state = hidden[-1]
        emb = [embedding(cat[:, idx]) for idx, embedding in enumerate(self.embeddings)]
        static = torch.cat([static_num, *emb], dim=1)
        static_state = self.static_net(static)
        return self.head(torch.cat([seq_state, static_state], dim=1)).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def period_id(year: int, month: int) -> int:
    return year * 12 + month


def mape_score_from_log_residual(
    target_rows: pd.DataFrame, pred_log: np.ndarray, alpha: float
) -> float:
    pred = apply_residual(
        target_rows["best_base"].to_numpy(dtype=float),
        pred_log,
        alpha,
        None,
    )
    return mape(target_rows[TARGET], pred)


def prepare_data() -> PreparedData:
    raw = pd.read_csv(TRAIN_PATH, low_memory=False)
    raw["period"] = raw["Год"] * 12 + raw["Месяц"]
    period_label = pd.to_datetime(
        raw["Год"].astype(str) + "-" + raw["Месяц"].astype(str).str.zfill(2) + "-01"
    ).dt.strftime("%Y-%m")
    rto_wide = raw.assign(period_label=period_label).pivot_table(
        index="new_id", columns="period", values=TARGET, aggfunc="mean"
    )
    ratio_2023 = (
        rto_wide[period_id(2023, 3)] / rto_wide[period_id(2023, 2)]
    ).replace([np.inf, -np.inf], np.nan)
    ratio_2024 = (
        rto_wide[period_id(2024, 3)] / rto_wide[period_id(2024, 2)]
    ).replace([np.inf, -np.inf], np.nan)
    validation_factor = float(ratio_2023.median())
    final_factor = float(pd.concat([ratio_2023, ratio_2024]).dropna().median())

    panel = add_base(make_panel(raw), validation_factor)
    target_rows = panel[
        (panel["Год"] == 2024)
        & (panel["Месяц"] == 3)
        & (panel[TARGET] > 0)
        & (panel["best_base"] > 0)
    ].copy()
    target_rows["target_log_ratio"] = np.log(target_rows[TARGET] / target_rows["best_base"])
    target_rows = target_rows.sort_values("new_id").reset_index(drop=True)

    behavior_wide = {
        col: raw.pivot_table(index="new_id", columns="period", values=col, aggfunc="mean")
        for col in BEHAVIOR_COLS
    }
    static_cols = ["new_id"] + STATIC_NUM_COLS + SMALL_CAT_COLS
    static_frame = (
        raw.sort_values(["new_id", "period"])
        .groupby("new_id", as_index=False)[static_cols]
        .first()
        .set_index("new_id")
    )
    return PreparedData(
        raw=raw,
        target_rows=target_rows,
        validation_factor=validation_factor,
        final_factor=final_factor,
        rto_wide=rto_wide,
        behavior_wide=behavior_wide,
        static_frame=static_frame,
    )


def build_sequence(data: PreparedData, store_ids: np.ndarray, target_period: int) -> np.ndarray:
    hist_periods = list(range(target_period - SEQ_LEN, target_period))
    rto = data.rto_wide.loc[store_ids, hist_periods].to_numpy(dtype=float)
    log_rto = np.log(np.clip(rto, 1.0, None))
    last_log = log_rto[:, [-1]]
    rel_log_rto = log_rto - last_log
    log_growth = np.zeros_like(log_rto)
    log_growth[:, 1:] = log_rto[:, 1:] - log_rto[:, :-1]

    months = np.array([((period - 1) % 12) + 1 for period in hist_periods], dtype=float)
    month_sin = np.sin(2 * np.pi * months / 12)[None, :, None]
    month_cos = np.cos(2 * np.pi * months / 12)[None, :, None]
    calendar = np.repeat(np.concatenate([month_sin, month_cos], axis=2), len(store_ids), axis=0)

    behavior_parts = []
    for col in BEHAVIOR_COLS:
        values = data.behavior_wide[col].loc[store_ids, hist_periods].to_numpy(dtype=float)
        behavior_parts.append(values[:, :, None])
    behavior = np.concatenate(behavior_parts, axis=2)
    seq = np.concatenate([rel_log_rto[:, :, None], log_growth[:, :, None], calendar, behavior], axis=2)
    return seq.astype(np.float32)


def fit_sequence_scaler(seq: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    flat = seq.reshape(-1, seq.shape[-1])
    mean = np.nanmean(flat, axis=0)
    std = np.nanstd(flat, axis=0)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def apply_sequence_scaler(seq: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    out = (seq - mean[None, None, :]) / std[None, None, :]
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def fit_static_encoders(
    static_frame: pd.DataFrame, train_ids: np.ndarray
) -> tuple[np.ndarray, np.ndarray, dict[str, dict[str, int]]]:
    train_static = static_frame.loc[train_ids]
    num_mean = train_static[STATIC_NUM_COLS].astype(float).mean().to_numpy(dtype=np.float32)
    num_std = train_static[STATIC_NUM_COLS].astype(float).std().replace(0, 1).to_numpy(dtype=np.float32)
    encoders = {}
    for col in SMALL_CAT_COLS:
        values = train_static[col].astype(str).fillna("__nan__").unique().tolist()
        encoders[col] = {value: idx + 1 for idx, value in enumerate(values)}
    return num_mean, num_std, encoders


def transform_static(
    static_frame: pd.DataFrame,
    store_ids: np.ndarray,
    num_mean: np.ndarray,
    num_std: np.ndarray,
    encoders: dict[str, dict[str, int]],
) -> tuple[np.ndarray, np.ndarray]:
    static = static_frame.loc[store_ids]
    numeric = static[STATIC_NUM_COLS].astype(float).to_numpy(dtype=np.float32)
    numeric = (numeric - num_mean[None, :]) / num_std[None, :]
    numeric = np.nan_to_num(numeric, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    cats = []
    for col in SMALL_CAT_COLS:
        mapping = encoders[col]
        cats.append(static[col].astype(str).fillna("__nan__").map(mapping).fillna(0).to_numpy(dtype=np.int64))
    return numeric, np.stack(cats, axis=1)


def train_one_fold(
    train_data: tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    valid_data: tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    cat_cardinalities: list[int],
    seed: int,
) -> tuple[np.ndarray, float]:
    set_seed(seed)
    train_seq, train_num, train_cat, y_train = train_data
    valid_seq, valid_num, valid_cat, y_valid = valid_data
    model = GRUResidual(
        seq_dim=train_seq.shape[-1],
        static_num_dim=train_num.shape[1],
        cat_cardinalities=cat_cardinalities,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-3)
    loss_fn = nn.SmoothL1Loss(beta=0.02)

    train_dataset = TensorDataset(
        torch.from_numpy(train_seq),
        torch.from_numpy(train_num),
        torch.from_numpy(train_cat),
        torch.from_numpy(y_train.astype(np.float32)),
    )
    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=make_loader_generator(SEED), num_workers=0)
    valid_tensors = (
        torch.from_numpy(valid_seq),
        torch.from_numpy(valid_num),
        torch.from_numpy(valid_cat),
        torch.from_numpy(y_valid.astype(np.float32)),
    )

    best_state = None
    best_loss = float("inf")
    stale = 0
    for _epoch in range(EPOCHS):
        model.train()
        for batch_seq, batch_num, batch_cat, batch_y in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(batch_seq, batch_num, batch_cat), batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            valid_loss = float(loss_fn(model(valid_tensors[0], valid_tensors[1], valid_tensors[2]), valid_tensors[3]))
        if valid_loss < best_loss - 1e-5:
            best_loss = valid_loss
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(valid_tensors[0], valid_tensors[1], valid_tensors[2]).numpy()
    return pred, best_loss


def run_oof(data: PreparedData) -> tuple[pd.DataFrame, pd.DataFrame]:
    target = data.target_rows.copy()
    store_ids = target["new_id"].to_numpy()
    target_period = period_id(2024, 3)
    full_seq = build_sequence(data, store_ids, target_period)
    y = target["target_log_ratio"].to_numpy(dtype=np.float32)

    fold_rows = []
    oof_parts = []
    for fold in range(5):
        valid_mask = (store_ids % 5) == fold
        train_ids = store_ids[~valid_mask]
        valid_ids = store_ids[valid_mask]
        train_seq_raw = full_seq[~valid_mask]
        valid_seq_raw = full_seq[valid_mask]
        seq_mean, seq_std = fit_sequence_scaler(train_seq_raw)
        train_seq = apply_sequence_scaler(train_seq_raw, seq_mean, seq_std)
        valid_seq = apply_sequence_scaler(valid_seq_raw, seq_mean, seq_std)

        num_mean, num_std, encoders = fit_static_encoders(data.static_frame, train_ids)
        train_num, train_cat = transform_static(data.static_frame, train_ids, num_mean, num_std, encoders)
        valid_num, valid_cat = transform_static(data.static_frame, valid_ids, num_mean, num_std, encoders)
        cat_cardinalities = [len(encoders[col]) for col in SMALL_CAT_COLS]

        pred_log, best_loss = train_one_fold(
            (train_seq, train_num, train_cat, y[~valid_mask]),
            (valid_seq, valid_num, valid_cat, y[valid_mask]),
            cat_cardinalities,
            SEED + fold,
        )
        valid_rows = target.loc[valid_mask].copy()
        valid_rows["fold"] = fold
        valid_rows["pred_log_gru"] = pred_log
        oof_parts.append(valid_rows[["new_id", TARGET, "best_base", "fold", "pred_log_gru"]])
        row = {"fold": fold, "valid_loss": best_loss, "base_mape": mape(valid_rows[TARGET], valid_rows["best_base"])}
        for alpha in ALPHA_GRID:
            row[f"gru_alpha_{alpha:.1f}_mape"] = mape_score_from_log_residual(valid_rows, pred_log, alpha)
        fold_rows.append(row)

    oof = pd.concat(oof_parts, ignore_index=True)
    folds = pd.DataFrame(fold_rows)
    return folds, oof


def compare_with_catboost(oof: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for alpha in ALPHA_GRID:
        pred = apply_residual(oof["best_base"].to_numpy(dtype=float), oof["pred_log_gru"].to_numpy(dtype=float), alpha, None)
        rows.append({"model": "gru", "alpha": alpha, "blend_weight_gru": 1.0, "mape": mape(oof[TARGET], pred)})

    if CATBOOST_OOF_PATH.exists():
        cb = pd.read_csv(CATBOOST_OOF_PATH)
        cb = cb[cb["config"] == "all_new_plus_targeted"][["new_id", "pred_log"]].rename(
            columns={"pred_log": "pred_log_catboost"}
        )
        merged = oof.merge(cb, on="new_id", how="inner")
        for weight in [0.10, 0.20, 0.30, 0.40, 0.50]:
            blended_log = (1 - weight) * merged["pred_log_catboost"].to_numpy(dtype=float) + weight * merged[
                "pred_log_gru"
            ].to_numpy(dtype=float)
            pred = apply_residual(merged["best_base"].to_numpy(dtype=float), blended_log, 0.70, None)
            rows.append({"model": "catboost_gru_blend", "alpha": 0.70, "blend_weight_gru": weight, "mape": mape(merged[TARGET], pred)})
        cb_pred = apply_residual(
            merged["best_base"].to_numpy(dtype=float),
            merged["pred_log_catboost"].to_numpy(dtype=float),
            0.70,
            None,
        )
        rows.append({"model": "catboost_targeted_reference", "alpha": 0.70, "blend_weight_gru": 0.0, "mape": mape(merged[TARGET], cb_pred)})
    return pd.DataFrame(rows).sort_values("mape").reset_index(drop=True)


def make_final_gru_submission(data: PreparedData, alpha: float) -> Path:
    set_seed(SEED)
    train_target = data.target_rows.copy()
    train_ids = train_target["new_id"].to_numpy()
    y_train = train_target["target_log_ratio"].to_numpy(dtype=np.float32)
    train_seq_raw = build_sequence(data, train_ids, period_id(2024, 3))
    seq_mean, seq_std = fit_sequence_scaler(train_seq_raw)
    train_seq = apply_sequence_scaler(train_seq_raw, seq_mean, seq_std)
    num_mean, num_std, encoders = fit_static_encoders(data.static_frame, train_ids)
    train_num, train_cat = transform_static(data.static_frame, train_ids, num_mean, num_std, encoders)
    cat_cardinalities = [len(encoders[col]) for col in SMALL_CAT_COLS]

    # Keep a small validation slice for early stopping in the final fit.
    valid_mask = (train_ids % 5) == 0
    pred_dummy, _ = train_one_fold(
        (train_seq[~valid_mask], train_num[~valid_mask], train_cat[~valid_mask], y_train[~valid_mask]),
        (train_seq[valid_mask], train_num[valid_mask], train_cat[valid_mask], y_train[valid_mask]),
        cat_cardinalities,
        SEED,
    )
    del pred_dummy

    model = GRUResidual(train_seq.shape[-1], train_num.shape[1], cat_cardinalities)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-3)
    loss_fn = nn.SmoothL1Loss(beta=0.02)
    dataset = TensorDataset(
        torch.from_numpy(train_seq),
        torch.from_numpy(train_num),
        torch.from_numpy(train_cat),
        torch.from_numpy(y_train.astype(np.float32)),
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, generator=make_loader_generator(SEED), num_workers=0)
    for _ in range(35):
        model.train()
        for batch_seq, batch_num, batch_cat, batch_y in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(batch_seq, batch_num, batch_cat), batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    feb_2025 = data.raw[(data.raw["Год"] == 2025) & (data.raw["Месяц"] == 2)].copy()
    mar_2025 = feb_2025.copy()
    mar_2025["Год"] = 2025
    mar_2025["Месяц"] = 3
    mar_2025["period"] = period_id(2025, 3)
    mar_2025[TARGET] = np.nan
    pred_panel = add_base(make_panel(pd.concat([data.raw, mar_2025], ignore_index=True)), data.final_factor)
    pred_rows = pred_panel[(pred_panel["Год"] == 2025) & (pred_panel["Месяц"] == 3)].copy()
    pred_rows = pred_rows.sort_values("new_id").reset_index(drop=True)
    pred_ids = pred_rows["new_id"].to_numpy()

    pred_seq = apply_sequence_scaler(build_sequence(data, pred_ids, period_id(2025, 3)), seq_mean, seq_std)
    pred_num, pred_cat = transform_static(data.static_frame, pred_ids, num_mean, num_std, encoders)
    model.eval()
    with torch.no_grad():
        pred_log = model(torch.from_numpy(pred_seq), torch.from_numpy(pred_num), torch.from_numpy(pred_cat)).numpy()
    pred = apply_residual(pred_rows["best_base"].to_numpy(dtype=float), pred_log, alpha, None)
    submission = pred_rows[["new_id"]].copy()
    submission["rto"] = np.round(np.maximum(pred, 0), 2)
    path = SUBMISSIONS_DIR / f"test_gru_alpha{int(alpha * 100):03d}.csv"
    submission.to_csv(path, index=False)
    return path


def main() -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
    data = prepare_data()
    folds, oof = run_oof(data)
    comparison = compare_with_catboost(oof)
    best_gru = comparison[comparison["model"] == "gru"].sort_values("mape").iloc[0]
    submission_path = make_final_gru_submission(data, float(best_gru["alpha"]))

    folds.to_csv(RESULTS_DIR / "gru_folds.csv", index=False)
    oof.to_csv(RESULTS_DIR / "gru_oof.csv", index=False)
    comparison.to_csv(RESULTS_DIR / "gru_comparison.csv", index=False)
    report = {
        "seq_len": SEQ_LEN,
        "epochs": EPOCHS,
        "best_gru": best_gru.to_dict(),
        "comparison": comparison.to_dict(orient="records"),
        "submission": str(submission_path),
    }
    (RESULTS_DIR / "summary.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2))


main()


## 2. Calendar Follow-Up



In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import numpy as np
import pandas as pd



def followup_specs() -> list[HeuristicSpec]:
    return [
        HeuristicSpec(63, "cal055_median_mom025", 0.55),
        HeuristicSpec(64, "cal060_median_mom025", 0.60),
        HeuristicSpec(65, "cal0625_median_mom025", 0.625),
        HeuristicSpec(66, "cal075_median_mom025", 0.75),
        HeuristicSpec(67, "cal0875_median_mom025", 0.875),
        HeuristicSpec(68, "cal100_median_mom025", 1.00),
        HeuristicSpec(69, "cal050_median_mom0125", 0.50, momentum_exp=0.125),
        HeuristicSpec(70, "cal050_median_mom000", 0.50, momentum_exp=0.0),
        HeuristicSpec(71, "cal0625_median_mom0125", 0.625, momentum_exp=0.125),
        HeuristicSpec(72, "cal0625_median_mom000", 0.625, momentum_exp=0.0),
        HeuristicSpec(73, "cal075_median_mom0125", 0.75, momentum_exp=0.125),
        HeuristicSpec(74, "cal050_mape_wmedian_mom025", 0.50, factor_method="mape_weighted_median"),
        HeuristicSpec(75, "cal0625_mape_wmedian_mom025", 0.625, factor_method="mape_weighted_median"),
    ]


def direction_question(spec: HeuristicSpec) -> str:
    if spec.momentum_exp != 0.25:
        return "Does weaker momentum exponent improve aggressive calendar baseline?"
    if spec.factor_method != "median":
        return "Does MAPE-weighted seasonal factor improve aggressive calendar baseline?"
    return "How far should calendar/leap normalization be pushed after candidate 47 public gain?"


def main() -> None:
    RESULTS_DIR.mkdir(exist_ok=True)
    SUBMISSIONS_DIR.mkdir(exist_ok=True)
    _raw, rto, static = load_data()
    val_frame = build_target_frame(rto, static, 2024, 3, include_target=True)
    test_frame = build_target_frame(rto, static, 2025, 3, include_target=False)

    rows = []
    for spec in followup_specs():
        val_factor, test_factor, val_global, final_global = factors_for_spec(spec, val_frame, test_frame, rto, static)
        val_pred = predict_from_spec(val_frame, val_factor, spec)
        test_pred = predict_from_spec(test_frame, test_factor, spec)

        out = test_frame[["new_id"]].copy()
        out["rto"] = np.round(test_pred, 2)
        file_name = f"stage2_candidate_{spec.candidate_id:02d}_baseline_{spec.name}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)

        rows.append(
            {
                "candidate_id": spec.candidate_id,
                "file": file_name,
                "name": spec.name,
                "family": "calendar_followup_baseline",
                "calendar_lambda": spec.calendar_lambda,
                "factor_method": spec.factor_method,
                "momentum_exp": spec.momentum_exp,
                "val_global_factor": val_global,
                "final_global_factor": final_global,
                "val_mape_march2024": mape(val_frame[TARGET], val_pred),
                "test_mean_rto": float(np.mean(test_pred)),
                "test_total_rto": float(np.sum(test_pred)),
                "direction_question": direction_question(spec),
                "public_status": "not_submitted",
                "public_score": "",
                "public_mape": "",
            }
        )

    summary = pd.DataFrame(rows).sort_values("candidate_id").reset_index(drop=True)
    summary.to_csv(RESULTS_DIR / "calendar_followup_candidates.csv", index=False)

    log_path = RESULTS_DIR / "public_experiment_log.csv"
    if log_path.exists():
        log = pd.read_csv(log_path)
        existing = set(log["candidate_id"].astype(str))
        new_rows = []
        for _, row in summary.iterrows():
            if str(row["candidate_id"]) in existing:
                continue
            new_rows.append(
                {
                    "candidate_id": row["candidate_id"],
                    "file": row["file"],
                    "family": row["family"],
                    "hypothesis": row["direction_question"],
                    "public_score": "",
                    "notes": "fill public_score after submission",
                    "public_mape": "",
                    "delta_mape_vs_current_best": "",
                    "delta_score_vs_current_best": "",
                }
            )
        if new_rows:
            pd.concat([log, pd.DataFrame(new_rows)], ignore_index=True).to_csv(log_path, index=False)

    print("Wrote calendar follow-up candidates:")
    print(summary[["candidate_id", "name", "val_mape_march2024", "final_global_factor", "test_mean_rto"]].to_string(index=False))


main()


## 3. New Baseline GRU Residual

GRU residual поверх новой baseline-сборки: подготовка последовательностей, OOF, финальное обучение и CSV-кандидаты.


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")


RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def make_loader_generator(seed: int) -> torch.Generator:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    return generator



SEED = 2026
SEQ_LEN = 14
EPOCHS = 80
FINAL_EPOCHS = 35
BATCH_SIZE = 512
PATIENCE = 10
DEVICE = torch.device("cpu")

OUT_DIR = RESULTS_DIR / "new_baseline_gru_residual"
NEW_BASE_SPEC = HeuristicSpec(0, "cal050_mom000", 0.50, momentum_exp=0.0)
MOMENTUM_TRAIN_BINS = [-np.inf, 0.935766244455909, 0.9524077370639855, 0.9656151000316019, 0.9815696834781942, np.inf]

STATIC_CAT_COLS = [
    "Дата открытия, категориальный",
    "Торговая площадь, категориальный",
    "Регион",
]
STATIC_NUM_COLS = [
    "Численность населения",
    "Количество домохозяйств",
    "Трафик пеший, в час",
    "Трафик авто, в час",
    "Маркетплейсы, доставки, постаматы (100 м)",
    "Медицинские уч. и аптеки (300 м)",
    "Школы (300 м)",
    "Остановки (300 м)",
    "Продуктовые магазины (500 м)",
    "Пятерочки (500 м)",
    "Количество касс",
    "Флаг алкогольной лицензии",
]
BEHAVIOR_COLS = [
    "Среднее количество промо товаров в чеке",
    "Среднее количество товаров в чеке",
    "Среднее количество отмен",
    "Рабочие часы в день",
]


@dataclass
class PreparedData:
    raw: pd.DataFrame
    rto_wide: pd.DataFrame
    behavior_wide: dict[str, pd.DataFrame]
    static_frame: pd.DataFrame
    train_rows: pd.DataFrame
    test_rows: pd.DataFrame


class GRUResidual(nn.Module):
    def __init__(self, seq_dim: int, static_num_dim: int, cat_cardinalities: list[int], hidden_dim: int = 32, dropout: float = 0.15) -> None:
        super().__init__()
        self.gru = nn.GRU(seq_dim, hidden_dim, num_layers=1, batch_first=True)
        self.embeddings = nn.ModuleList()
        emb_dims = []
        for cardinality in cat_cardinalities:
            dim = min(12, max(2, int(np.sqrt(cardinality + 1)) + 1))
            self.embeddings.append(nn.Embedding(cardinality + 1, dim, padding_idx=0))
            emb_dims.append(dim)
        static_in = static_num_dim + sum(emb_dims)
        self.static_net = nn.Sequential(nn.Linear(static_in, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 16), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(hidden_dim + 16, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, seq: torch.Tensor, static_num: torch.Tensor, cat: torch.Tensor) -> torch.Tensor:
        _, hidden = self.gru(seq)
        seq_state = hidden[-1]
        emb = [embedding(cat[:, idx]) for idx, embedding in enumerate(self.embeddings)]
        static = torch.cat([static_num, *emb], dim=1)
        static_state = self.static_net(static)
        return self.head(torch.cat([seq_state, static_state], dim=1)).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def period_id(year: int, month: int) -> int:
    return year * 12 + month


def build_static(raw: pd.DataFrame) -> pd.DataFrame:
    existing_num = [col for col in STATIC_NUM_COLS if col in raw.columns]
    existing_cat = [col for col in STATIC_CAT_COLS if col in raw.columns]
    static = raw.sort_values(["new_id", "period"]).groupby("new_id", as_index=False)[["new_id"] + existing_num + existing_cat].first().set_index("new_id")
    for col in existing_num:
        static[col] = pd.to_numeric(static[col], errors="coerce")
    for col in existing_cat:
        static[col] = static[col].astype(str).fillna("__nan__")
    return static


def prepare_data() -> PreparedData:
    raw, rto_label_wide, static_for_base = load_data()
    raw = raw.copy()
    raw["period"] = raw["Год"] * 12 + raw["Месяц"]
    rto_wide = raw.pivot_table(index="new_id", columns="period", values=TARGET, aggfunc="mean")
    behavior_wide = {
        col: raw.pivot_table(index="new_id", columns="period", values=col, aggfunc="mean")
        for col in BEHAVIOR_COLS
        if col in raw.columns
    }
    static_frame = build_static(raw)

    train_features = build_target_frame(rto_label_wide, static_for_base, 2024, 3, include_target=True)
    test_features = build_target_frame(rto_label_wide, static_for_base, 2025, 3, include_target=False)
    train_factor, test_factor, _val_global, _final_global = factors_for_spec(NEW_BASE_SPEC, train_features, test_features, rto_label_wide, static_for_base)
    train_features["new_base"] = predict_from_spec(train_features, train_factor, NEW_BASE_SPEC)
    test_features["new_base"] = predict_from_spec(test_features, test_factor, NEW_BASE_SPEC)
    train_features = train_features[(train_features[TARGET] > 0) & (train_features["new_base"] > 0)].copy()
    train_features["target_log_ratio"] = np.log(train_features[TARGET] / train_features["new_base"])
    train_features["momentum_train_bin"] = momentum_bin(train_features)
    test_features["momentum_train_bin"] = momentum_bin(test_features)

    return PreparedData(
        raw=raw,
        rto_wide=rto_wide,
        behavior_wide=behavior_wide,
        static_frame=static_frame,
        train_rows=train_features.sort_values("new_id").reset_index(drop=True),
        test_rows=test_features.sort_values("new_id").reset_index(drop=True),
    )


def momentum_bin(frame: pd.DataFrame) -> pd.Series:
    return pd.cut(frame["momentum_raw"], bins=MOMENTUM_TRAIN_BINS, labels=[f"q{i}" for i in range(5)], include_lowest=True).astype(str)


def build_sequence(data: PreparedData, store_ids: np.ndarray, target_period: int) -> np.ndarray:
    hist_periods = list(range(target_period - SEQ_LEN, target_period))
    rto = data.rto_wide.loc[store_ids, hist_periods].to_numpy(dtype=float)
    log_rto = np.log(np.clip(rto, 1.0, None))
    last_log = log_rto[:, [-1]]
    rel_log_rto = log_rto - last_log
    log_growth = np.zeros_like(log_rto)
    log_growth[:, 1:] = log_rto[:, 1:] - log_rto[:, :-1]

    months = np.array([((period - 1) % 12) + 1 for period in hist_periods], dtype=float)
    month_sin = np.sin(2 * np.pi * months / 12)[None, :, None]
    month_cos = np.cos(2 * np.pi * months / 12)[None, :, None]
    calendar = np.repeat(np.concatenate([month_sin, month_cos], axis=2), len(store_ids), axis=0)

    behavior_parts = []
    for col in data.behavior_wide:
        values = data.behavior_wide[col].loc[store_ids, hist_periods].to_numpy(dtype=float)
        behavior_parts.append(values[:, :, None])
    behavior = np.concatenate(behavior_parts, axis=2) if behavior_parts else np.zeros((len(store_ids), len(hist_periods), 0))
    seq = np.concatenate([rel_log_rto[:, :, None], log_growth[:, :, None], calendar, behavior], axis=2)
    return seq.astype(np.float32)


def fit_sequence_scaler(seq: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    flat = seq.reshape(-1, seq.shape[-1])
    mean = np.nanmean(flat, axis=0)
    std = np.nanstd(flat, axis=0)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def apply_sequence_scaler(seq: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    out = (seq - mean[None, None, :]) / std[None, None, :]
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def fit_static_encoders(static_frame: pd.DataFrame, train_ids: np.ndarray) -> tuple[np.ndarray, np.ndarray, dict[str, dict[str, int]], list[str], list[str]]:
    existing_num = [col for col in STATIC_NUM_COLS if col in static_frame.columns]
    existing_cat = [col for col in STATIC_CAT_COLS if col in static_frame.columns]
    train_static = static_frame.loc[train_ids]
    num_mean = train_static[existing_num].astype(float).mean().to_numpy(dtype=np.float32)
    num_std = train_static[existing_num].astype(float).std().replace(0, 1).fillna(1).to_numpy(dtype=np.float32)
    encoders = {}
    for col in existing_cat:
        values = train_static[col].astype(str).fillna("__nan__").unique().tolist()
        encoders[col] = {value: idx + 1 for idx, value in enumerate(values)}
    return num_mean, num_std, encoders, existing_num, existing_cat


def transform_static(static_frame: pd.DataFrame, store_ids: np.ndarray, num_mean: np.ndarray, num_std: np.ndarray, encoders: dict[str, dict[str, int]], num_cols: list[str], cat_cols: list[str]) -> tuple[np.ndarray, np.ndarray]:
    static = static_frame.loc[store_ids]
    numeric = static[num_cols].astype(float).to_numpy(dtype=np.float32)
    numeric = np.nan_to_num((numeric - num_mean[None, :]) / num_std[None, :], nan=0.0, posinf=0.0, neginf=0.0)
    cats = []
    for col in cat_cols:
        mapping = encoders[col]
        cats.append(static[col].astype(str).fillna("__nan__").map(mapping).fillna(0).to_numpy(dtype=np.int64))
    cat_arr = np.stack(cats, axis=1) if cats else np.zeros((len(store_ids), 0), dtype=np.int64)
    return numeric.astype(np.float32), cat_arr


def train_one_fold(train_data, valid_data, cat_cardinalities: list[int], seed: int) -> tuple[np.ndarray, float]:
    set_seed(seed)
    train_seq, train_num, train_cat, y_train = train_data
    valid_seq, valid_num, valid_cat, y_valid = valid_data
    model = GRUResidual(train_seq.shape[-1], train_num.shape[1], cat_cardinalities).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-3)
    loss_fn = nn.SmoothL1Loss(beta=0.02)
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(train_seq),
            torch.from_numpy(train_num),
            torch.from_numpy(train_cat),
            torch.from_numpy(y_train.astype(np.float32)),
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=make_loader_generator(SEED),
        num_workers=0,
        pin_memory=False,
    )
    valid_tensors = (
        torch.from_numpy(valid_seq).to(DEVICE),
        torch.from_numpy(valid_num).to(DEVICE),
        torch.from_numpy(valid_cat).to(DEVICE),
        torch.from_numpy(y_valid.astype(np.float32)).to(DEVICE),
    )
    best_state = None
    best_loss = float("inf")
    stale = 0
    for _epoch in range(EPOCHS):
        model.train()
        for batch_seq, batch_num, batch_cat, batch_y in loader:
            batch_seq = batch_seq.to(DEVICE, non_blocking=True)
            batch_num = batch_num.to(DEVICE, non_blocking=True)
            batch_cat = batch_cat.to(DEVICE, non_blocking=True)
            batch_y = batch_y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = loss_fn(model(batch_seq, batch_num, batch_cat), batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            valid_loss = float(loss_fn(model(valid_tensors[0], valid_tensors[1], valid_tensors[2]), valid_tensors[3]).detach().cpu().item())
        if valid_loss < best_loss - 1e-5:
            best_loss = valid_loss
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= PATIENCE:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(valid_tensors[0], valid_tensors[1], valid_tensors[2]).detach().cpu().numpy()
    return pred, best_loss


def run_oof(data: PreparedData) -> pd.DataFrame:
    target = data.train_rows.copy()
    store_ids = target["new_id"].to_numpy()
    full_seq = build_sequence(data, store_ids, period_id(2024, 3))
    y = target["target_log_ratio"].to_numpy(dtype=np.float32)
    parts = []
    fold_rows = []
    for fold in range(5):
        valid_mask = (store_ids % 5) == fold
        train_ids = store_ids[~valid_mask]
        valid_ids = store_ids[valid_mask]
        seq_mean, seq_std = fit_sequence_scaler(full_seq[~valid_mask])
        train_seq = apply_sequence_scaler(full_seq[~valid_mask], seq_mean, seq_std)
        valid_seq = apply_sequence_scaler(full_seq[valid_mask], seq_mean, seq_std)
        num_mean, num_std, encoders, num_cols, cat_cols = fit_static_encoders(data.static_frame, train_ids)
        train_num, train_cat = transform_static(data.static_frame, train_ids, num_mean, num_std, encoders, num_cols, cat_cols)
        valid_num, valid_cat = transform_static(data.static_frame, valid_ids, num_mean, num_std, encoders, num_cols, cat_cols)
        pred_log, best_loss = train_one_fold((train_seq, train_num, train_cat, y[~valid_mask]), (valid_seq, valid_num, valid_cat, y[valid_mask]), [len(encoders[col]) for col in cat_cols], SEED + fold)
        valid_rows = target.loc[valid_mask].copy()
        valid_rows["fold"] = fold
        valid_rows["pred_log_gru"] = pred_log
        parts.append(valid_rows[["new_id", TARGET, "new_base", "momentum_train_bin", "fold", "pred_log_gru"]])
        fold_rows.append({"fold": fold, "valid_loss": best_loss, "base_mape": mape(valid_rows[TARGET], valid_rows["new_base"])})
    oof = pd.concat(parts, ignore_index=True)
    pd.DataFrame(fold_rows).to_csv(OUT_DIR / "fold_summary.csv", index=False)
    return oof


def center_by_group(values: pd.Series, groups: pd.Series) -> pd.Series:
    med = values.groupby(groups, observed=True).transform("median")
    return values - med.fillna(float(values.median()))


def evaluate_oof(oof: pd.DataFrame) -> pd.DataFrame:
    rows = [{"variant": "new_base", "alpha": 0.0, "center": "none", "mape": mape(oof[TARGET], oof["new_base"])}]
    for center in ["none", "global", "momentum_bin"]:
        if center == "none":
            pred_log = oof["pred_log_gru"]
        elif center == "global":
            pred_log = oof["pred_log_gru"] - float(oof["pred_log_gru"].median())
        else:
            pred_log = center_by_group(oof["pred_log_gru"], oof["momentum_train_bin"])
        for alpha in [0.3, 0.5, 0.7, 0.85, 1.0, 1.15, 1.3]:
            pred = oof["new_base"].to_numpy(dtype=float) * np.exp(alpha * pred_log.to_numpy(dtype=float))
            rows.append({"variant": f"new_gru_{center}", "alpha": alpha, "center": center, "mape": mape(oof[TARGET], pred)})
    return pd.DataFrame(rows).sort_values("mape").reset_index(drop=True)


def train_final_and_predict(data: PreparedData) -> np.ndarray:
    set_seed(SEED)
    train = data.train_rows.copy()
    store_ids = train["new_id"].to_numpy()
    y = train["target_log_ratio"].to_numpy(dtype=np.float32)
    seq_raw = build_sequence(data, store_ids, period_id(2024, 3))
    seq_mean, seq_std = fit_sequence_scaler(seq_raw)
    seq = apply_sequence_scaler(seq_raw, seq_mean, seq_std)
    num_mean, num_std, encoders, num_cols, cat_cols = fit_static_encoders(data.static_frame, store_ids)
    num, cat = transform_static(data.static_frame, store_ids, num_mean, num_std, encoders, num_cols, cat_cols)
    valid_mask = (store_ids % 5) == 0
    pred_dummy, _ = train_one_fold(
        (seq[~valid_mask], num[~valid_mask], cat[~valid_mask], y[~valid_mask]),
        (seq[valid_mask], num[valid_mask], cat[valid_mask], y[valid_mask]),
        [len(encoders[col]) for col in cat_cols],
        SEED,
    )
    del pred_dummy
    model = GRUResidual(seq.shape[-1], num.shape[1], [len(encoders[col]) for col in cat_cols]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-3)
    loss_fn = nn.SmoothL1Loss(beta=0.02)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(seq), torch.from_numpy(num), torch.from_numpy(cat), torch.from_numpy(y)),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=make_loader_generator(SEED),
        num_workers=0,
        pin_memory=False,
    )
    for _epoch in range(FINAL_EPOCHS):
        model.train()
        for batch_seq, batch_num, batch_cat, batch_y in loader:
            batch_seq = batch_seq.to(DEVICE, non_blocking=True)
            batch_num = batch_num.to(DEVICE, non_blocking=True)
            batch_cat = batch_cat.to(DEVICE, non_blocking=True)
            batch_y = batch_y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = loss_fn(model(batch_seq, batch_num, batch_cat), batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    test_ids = data.test_rows["new_id"].to_numpy()
    test_seq = apply_sequence_scaler(build_sequence(data, test_ids, period_id(2025, 3)), seq_mean, seq_std)
    test_num, test_cat = transform_static(data.static_frame, test_ids, num_mean, num_std, encoders, num_cols, cat_cols)
    model.eval()
    with torch.no_grad():
        return (
            model(
                torch.from_numpy(test_seq).to(DEVICE),
                torch.from_numpy(test_num).to(DEVICE),
                torch.from_numpy(test_cat).to(DEVICE),
            )
            .detach()
            .cpu()
            .numpy()
        )


def write_candidates(data: PreparedData, pred_log: np.ndarray, oof: pd.DataFrame, summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    test = data.test_rows.copy()
    test["pred_log_gru"] = pred_log
    test["pred_log_global"] = test["pred_log_gru"] - float(test["pred_log_gru"].median())
    test["pred_log_mombin"] = center_by_group(test["pred_log_gru"], test["momentum_train_bin"])
    candidate_specs = [
        (145, "newgru_faithful_raw_a070", "pred_log_gru", 0.70),
        (146, "newgru_faithful_raw_a085", "pred_log_gru", 0.85),
        (147, "newgru_faithful_raw_a100", "pred_log_gru", 1.00),
        (148, "newgru_faithful_global_a070", "pred_log_global", 0.70),
        (149, "newgru_faithful_global_a085", "pred_log_global", 0.85),
        (150, "newgru_faithful_global_a100", "pred_log_global", 1.00),
        (151, "newgru_faithful_mombin_a070", "pred_log_mombin", 0.70),
        (152, "newgru_faithful_mombin_a100", "pred_log_mombin", 1.00),
        (153, "newgru_faithful_mombin_a115", "pred_log_mombin", 1.15),
        (154, "newgru_faithful_mombin_a130", "pred_log_mombin", 1.30),
    ]
    for candidate_id, name, col, alpha in candidate_specs:
        pred = test["new_base"].to_numpy(dtype=float) * np.exp(alpha * test[col].to_numpy(dtype=float))
        out = test[["new_id"]].copy()
        out["rto"] = np.round(np.maximum(pred, 0), 2)
        file_name = f"stage2_candidate_{candidate_id:03d}_baseline_{name}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)
        rows.append({"candidate_id": candidate_id, "file": file_name, "name": name, "alpha": alpha, "center": col, "test_mean_rto": float(np.mean(pred)), "test_total_rto": float(np.sum(pred))})
    out_summary = pd.DataFrame(rows)
    out_summary.to_csv(OUT_DIR / "submission_candidates.csv", index=False)

    log_path = RESULTS_DIR / "public_experiment_log.csv"
    if log_path.exists():
        log = pd.read_csv(log_path)
        existing = set(log["candidate_id"].astype(str))
        new_rows = []
        for _, row in out_summary.iterrows():
            if str(row["candidate_id"]) in existing:
                continue
            new_rows.append({"candidate_id": row["candidate_id"], "file": row["file"], "family": "new_baseline_gru_retrained", "hypothesis": "Fresh GRU residual trained directly on new calendar/momentum-zero baseline", "public_score": "", "notes": "fill public_score after submission", "public_mape": "", "delta_mape_vs_current_best": "", "delta_score_vs_current_best": ""})
        if new_rows:
            pd.concat([log, pd.DataFrame(new_rows)], ignore_index=True).to_csv(log_path, index=False)
    return out_summary


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Using device: {DEVICE}")
    data = prepare_data()
    oof = run_oof(data)
    oof.to_csv(OUT_DIR / "oof.csv", index=False)
    summary = evaluate_oof(oof)
    summary.to_csv(OUT_DIR / "oof_summary.csv", index=False)
    pred_log = train_final_and_predict(data)
    candidates = write_candidates(data, pred_log, oof, summary)
    metadata = {"seq_len": SEQ_LEN, "epochs": EPOCHS, "final_epochs": FINAL_EPOCHS, "base_spec": NEW_BASE_SPEC.__dict__}
    (OUT_DIR / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    print("OOF summary:")
    print(summary.head(20).to_string(index=False))
    print("\nSubmission candidates:")
    print(candidates.to_string(index=False))


main()


TN_BATCH_SIZE = BATCH_SIZE
TN_BEHAVIOR_COLS = BEHAVIOR_COLS
TN_DEVICE = DEVICE
TN_EPOCHS = EPOCHS
TN_FINAL_EPOCHS = FINAL_EPOCHS
TN_GRUResidual = GRUResidual
TN_MOMENTUM_TRAIN_BINS = MOMENTUM_TRAIN_BINS
TN_PATIENCE = PATIENCE
TN_SEED = SEED
TN_SEQ_LEN = SEQ_LEN
TN_STATIC_CAT_COLS = STATIC_CAT_COLS
TN_STATIC_NUM_COLS = STATIC_NUM_COLS
TN_build_sequence = build_sequence
TN_build_static = build_static
TN_center_by_group = center_by_group
TN_fit_sequence_scaler = fit_sequence_scaler
TN_fit_static_encoders = fit_static_encoders
TN_momentum_bin = momentum_bin
TN_period_id = period_id
TN_set_seed = set_seed
TN_transform_static = transform_static


## 4. Gamma High Sweep



In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import numpy as np
import pandas as pd



MOMENTUM_TRAIN_BINS = [-np.inf, 0.935766244455909, 0.9524077370639855, 0.9656151000316019, 0.9815696834781942, np.inf]


def read_local_artifact_csv(path: str) -> pd.DataFrame:
    local_path = RESULTS_DIR.parents[0] / path
    if not local_path.exists():
        raise FileNotFoundError(f"Expected local artifact from earlier notebook cells: {local_path}")
    return pd.read_csv(local_path)


def momentum_bin(frame: pd.DataFrame) -> pd.Series:
    return pd.cut(frame["momentum_raw"], bins=MOMENTUM_TRAIN_BINS, labels=[f"q{i}" for i in range(5)], include_lowest=True).astype(str)


def centered(log_residual: pd.Series, groups: pd.Series) -> pd.Series:
    med = log_residual.groupby(groups, observed=True).transform("median")
    return log_residual - med.fillna(float(log_residual.median()))


def make_base(frame: pd.DataFrame, rto: pd.DataFrame, static: pd.DataFrame, spec: HeuristicSpec, is_test: bool) -> np.ndarray:
    val_factor, test_factor, _val_global, _final_global = factors_for_spec(spec, frame, frame, rto, static)
    return predict_from_spec(frame, test_factor if is_test else val_factor, spec)


def candidate_specs() -> list[tuple[int, str, HeuristicSpec, float, float | None]]:
    base050 = HeuristicSpec(0, "cal050_mom000", 0.50, momentum_exp=0.0)
    base055 = HeuristicSpec(0, "cal055_mom000", 0.55, momentum_exp=0.0)
    return [
        (114, "cal050_mombin_g075", base050, 0.75, None),
        (115, "cal050_mombin_g085", base050, 0.85, None),
        (116, "cal050_mombin_g090", base050, 0.90, None),
        (117, "cal050_mombin_g095", base050, 0.95, None),
        (118, "cal050_mombin_g100", base050, 1.00, None),
        (119, "cal055_mombin_g070", base055, 0.70, None),
        (120, "cal055_mombin_g080", base055, 0.80, None),
        (121, "cal050_mombin_g080_clip020", base050, 0.80, 0.020),
        (122, "cal050_mombin_g090_clip020", base050, 0.90, 0.020),
        (123, "cal050_mombin_g100_clip020", base050, 1.00, 0.020),
    ]


def main() -> None:
    RESULTS_DIR.mkdir(exist_ok=True)
    SUBMISSIONS_DIR.mkdir(exist_ok=True)
    _raw, rto, static = load_data()
    val_frame = build_target_frame(rto, static, 2024, 3, include_target=True)
    test_frame = build_target_frame(rto, static, 2025, 3, include_target=False)

    old_spec = HeuristicSpec(0, "old_raw_mom025", 0.0, momentum_exp=0.25)
    old_test_base = make_base(test_frame, rto, static, old_spec, is_test=True)

    oof = read_local_artifact_csv("results/gru_residual/gru_oof.csv")[["new_id", "pred_log_gru"]]
    val = val_frame.merge(oof, on="new_id", how="inner")
    val["old_gru_ratio_log"] = 0.70 * val["pred_log_gru"].to_numpy(dtype=float)
    val["momentum_train_bin"] = momentum_bin(val)

    test_gru = read_local_artifact_csv("submissions/test_gru_alpha070.csv").rename(columns={"rto": "old_gru"})
    test = test_frame.merge(test_gru, on="new_id", how="inner")
    test_old_base = pd.Series(old_test_base, index=test_frame["new_id"]).reindex(test["new_id"]).to_numpy(dtype=float)
    test["old_gru_ratio_log"] = np.log(test["old_gru"].to_numpy(dtype=float) / test_old_base)
    test["momentum_train_bin"] = momentum_bin(test)

    val_resid_raw = centered(val["old_gru_ratio_log"], val["momentum_train_bin"]).to_numpy(dtype=float)
    test_resid_raw = centered(test["old_gru_ratio_log"], test["momentum_train_bin"]).to_numpy(dtype=float)

    rows = []
    for candidate_id, name, base_spec, gamma, clip in candidate_specs():
        val_base = make_base(val_frame, rto, static, base_spec, is_test=False)
        test_base = make_base(test_frame, rto, static, base_spec, is_test=True)
        val_base = pd.Series(val_base, index=val_frame["new_id"]).reindex(val["new_id"]).to_numpy(dtype=float)
        test_base = pd.Series(test_base, index=test_frame["new_id"]).reindex(test["new_id"]).to_numpy(dtype=float)
        val_resid = val_resid_raw.copy()
        test_resid = test_resid_raw.copy()
        if clip is not None:
            val_resid = np.clip(val_resid, -clip, clip)
            test_resid = np.clip(test_resid, -clip, clip)

        val_pred = val_base * np.exp(gamma * val_resid)
        test_pred = test_base * np.exp(gamma * test_resid)

        file_name = f"stage2_candidate_{candidate_id:03d}_baseline_{name}.csv"
        out = test[["new_id"]].copy()
        out["rto"] = np.round(np.maximum(test_pred, 0), 2)
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)

        rows.append(
            {
                "candidate_id": candidate_id,
                "file": file_name,
                "name": name,
                "calendar_lambda": base_spec.calendar_lambda,
                "gamma": gamma,
                "clip": clip if clip is not None else "",
                "val_mape_march2024": mape(val[TARGET], val_pred),
                "test_mean_rto": float(np.mean(test_pred)),
                "test_total_rto": float(np.sum(test_pred)),
                "mean_abs_change_vs_base_pct": float(np.mean(np.abs(test_pred / test_base - 1.0)) * 100),
                "p95_abs_change_vs_base_pct": float(np.quantile(np.abs(test_pred / test_base - 1.0), 0.95) * 100),
            }
        )

    summary = pd.DataFrame(rows).sort_values("candidate_id").reset_index(drop=True)
    summary.to_csv(RESULTS_DIR / "gamma_high_sweep_candidates.csv", index=False)

    log_path = RESULTS_DIR / "public_experiment_log.csv"
    if log_path.exists():
        log = pd.read_csv(log_path)
        existing = set(log["candidate_id"].astype(str))
        new_rows = []
        for _, row in summary.iterrows():
            if str(row["candidate_id"]) in existing:
                continue
            new_rows.append(
                {
                    "candidate_id": row["candidate_id"],
                    "file": row["file"],
                    "family": "gamma_high_sweep",
                    "hypothesis": "Find public optimum above gamma 0.70 for mombin-centered residual transfer",
                    "public_score": "",
                    "notes": "fill public_score after submission",
                    "public_mape": "",
                    "delta_mape_vs_current_best": "",
                    "delta_score_vs_current_best": "",
                }
            )
        if new_rows:
            pd.concat([log, pd.DataFrame(new_rows)], ignore_index=True).to_csv(log_path, index=False)

    print("Wrote high-gamma sweep candidates:")
    print(summary.to_string(index=False))


main()


## 5. Faithful GRU Follow-Up

Консервативный follow-up blend, включая candidate 159.


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import numpy as np
import pandas as pd





def read_local_artifact_csv(path: str) -> pd.DataFrame:
    local_path = RESULTS_DIR.parents[0] / path
    if not local_path.exists():
        raise FileNotFoundError(f"Expected local artifact from earlier notebook cells: {local_path}")
    return pd.read_csv(local_path)


def main() -> None:
    base = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_70_baseline_cal050_median_mom000.csv").rename(columns={"rto": "base"})
    g070 = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_148_baseline_newgru_faithful_global_a070.csv").rename(columns={"rto": "g070"})
    g085 = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_149_baseline_newgru_faithful_global_a085.csv").rename(columns={"rto": "g085"})
    g100 = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_150_baseline_newgru_faithful_global_a100.csv").rename(columns={"rto": "g100"})
    transfer = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_118_baseline_cal050_mombin_g100.csv").rename(columns={"rto": "transfer118"})
    frame = base.merge(g070, on="new_id").merge(g085, on="new_id").merge(g100, on="new_id").merge(transfer, on="new_id")

    log_r070 = np.log(frame["g070"] / frame["base"])
    log_r085 = np.log(frame["g085"] / frame["base"])
    log_r100 = np.log(frame["g100"] / frame["base"])
    unit_log = log_r100
    specs = [
        (155, "faithful_global_a0925", frame["base"].to_numpy(float) * np.exp(0.925 * unit_log)),
        (156, "faithful_global_a1050", frame["base"].to_numpy(float) * np.exp(1.050 * unit_log)),
        (157, "faithful_global_a1100", frame["base"].to_numpy(float) * np.exp(1.100 * unit_log)),
        (158, "faithful_global_a1150", frame["base"].to_numpy(float) * np.exp(1.150 * unit_log)),
        (159, "faithful_global_blend_75_25_transfer", 0.75 * frame["g100"].to_numpy(float) + 0.25 * frame["transfer118"].to_numpy(float)),
        (160, "faithful_global_blend_50_50_transfer", 0.50 * frame["g100"].to_numpy(float) + 0.50 * frame["transfer118"].to_numpy(float)),
        (161, "faithful_global_a1050_clip030", frame["base"].to_numpy(float) * np.exp(np.clip(1.050 * unit_log, -0.03, 0.03))),
        (162, "faithful_global_a1100_clip030", frame["base"].to_numpy(float) * np.exp(np.clip(1.100 * unit_log, -0.03, 0.03))),
    ]

    rows = []
    for candidate_id, name, pred in specs:
        out = frame[["new_id"]].copy()
        out["rto"] = np.round(np.maximum(pred, 0), 2)
        file_name = f"stage2_candidate_{candidate_id:03d}_baseline_{name}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)
        rows.append(
            {
                "candidate_id": candidate_id,
                "file": file_name,
                "name": name,
                "test_mean_rto": float(np.mean(pred)),
                "test_total_rto": float(np.sum(pred)),
                "mean_abs_change_vs_150_pct": float(np.mean(np.abs(pred / frame["g100"].to_numpy(float) - 1.0)) * 100),
                "p95_abs_change_vs_150_pct": float(np.quantile(np.abs(pred / frame["g100"].to_numpy(float) - 1.0), 0.95) * 100),
            }
        )
    summary = pd.DataFrame(rows)
    summary.to_csv(RESULTS_DIR / "faithful_gru_followup_candidates.csv", index=False)

    log_path = RESULTS_DIR / "public_experiment_log.csv"
    if log_path.exists():
        log = pd.read_csv(log_path)
        existing = set(log["candidate_id"].astype(str))
        new_rows = []
        for _, row in summary.iterrows():
            if str(row["candidate_id"]) in existing:
                continue
            new_rows.append({"candidate_id": row["candidate_id"], "file": row["file"], "family": "faithful_gru_followup", "hypothesis": "Fine tune faithful globally-centered GRU alpha/blend around candidate 150", "public_score": "", "notes": "fill public_score after submission", "public_mape": "", "delta_mape_vs_current_best": "", "delta_score_vs_current_best": ""})
        if new_rows:
            pd.concat([log, pd.DataFrame(new_rows)], ignore_index=True).to_csv(log_path, index=False)

    print(summary.to_string(index=False))


main()


## 6. Temporal204 Structural Baseline

Структурная baseline-ветка temporal204.


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import sys
from pathlib import Path

import numpy as np
import pandas as pd




def main() -> None:
    RESULTS_DIR.mkdir(exist_ok=True)
    SUBMISSIONS_DIR.mkdir(exist_ok=True)
    _raw, rto, static = load_data()
    val = build_target_frame(rto, static, 2024, 3, include_target=True)
    test = build_target_frame(rto, static, 2025, 3, include_target=False)
    spec = HeuristicSpec(0, "current", CURRENT_LAMBDA, momentum_exp=0.0)
    _vf, _tf, _val_factor, final_factor = factors_for_spec(spec, val, test, rto, static)
    current_pred = current_198_march(test, final_factor, target_non_leap=True)
    current_198_path = SUBMISSIONS_DIR / "stage2_candidate_198_baseline_v4_followup_signasym_lam0550_wp02_wn14_cp002_cn003.csv"
    if current_198_path.exists():
        current_198 = pd.read_csv(current_198_path).rename(columns={"rto": "rto_198"})
    else:
        current_198 = test[["new_id"]].copy()
        current_198["rto_198"] = np.round(np.maximum(current_pred, 0.0), 2)

    candidate_specs = [
        {
            "candidate_id": 203,
            "family": "temporal_two_month_growth",
            "name": "temporal2_raw_clip085_135_a012_lc004",
            "daily_adjust": False,
            "clip_lo": 0.85,
            "clip_hi": 1.35,
            "alpha": 0.12,
            "log_clip": 0.04,
            "selection_reason": "best March proxy for temporal/Q1-style two-month growth anchor",
        },
        {
            "candidate_id": 204,
            "family": "temporal_two_month_growth",
            "name": "temporal2_raw_clip085_135_a012_lc006",
            "daily_adjust": False,
            "clip_lo": 0.85,
            "clip_hi": 1.35,
            "alpha": 0.12,
            "log_clip": 0.06,
            "selection_reason": "best multi-period temporal/Q1-style two-month growth anchor",
        },
        {
            "candidate_id": 205,
            "family": "hierarchical_group_total_correction",
            "name": "group_opening_total_a035_lc001_k1000",
            "group_col": "Дата открытия, категориальный",
            "alpha": 0.35,
            "log_clip": 0.01,
            "shrink_k": 1000,
            "selection_reason": "best lightweight hierarchical group-total diagnostic; lower priority than temporal candidates",
        },
    ]

    rows = []
    for cfg in candidate_specs:
        if cfg["family"] == "temporal_two_month_growth":
            anchor = two_month_temporal_anchor(rto, test, 2025, 3, cfg["daily_adjust"], cfg["clip_lo"], cfg["clip_hi"])
            pred = blend_anchor(current_pred, anchor, cfg["alpha"], cfg["log_clip"])
        else:
            pred = group_total_correction(
                test,
                current_pred,
                cfg["group_col"],
                2025,
                3,
                [2023, 2024],
                rto,
                cfg["alpha"],
                cfg["log_clip"],
                cfg["shrink_k"],
            )

        out = test[["new_id"]].copy()
        out["rto"] = np.round(np.maximum(pred, 0.0), 2)
        file_name = f"stage2_candidate_{cfg['candidate_id']:03d}_baseline_v6_{cfg['name']}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)
        merged = out.merge(current_198, on="new_id")
        ratio = merged["rto"].to_numpy(float) / merged["rto_198"].to_numpy(float) - 1.0
        row = {
            "candidate_id": cfg["candidate_id"],
            "file": file_name,
            "family": cfg["family"],
            "name": cfg["name"],
            "selection_reason": cfg["selection_reason"],
            "test_mean_rto": float(np.mean(pred)),
            "test_total_rto": float(np.sum(pred)),
            "delta_mean_rto_vs_198_pct": float((np.mean(pred) / merged["rto_198"].mean() - 1.0) * 100),
            "mean_abs_change_vs_198_pct": float(np.mean(np.abs(ratio)) * 100),
            "p95_abs_change_vs_198_pct": float(np.quantile(np.abs(ratio), 0.95) * 100),
            "max_abs_change_vs_198_pct": float(np.max(np.abs(ratio)) * 100),
        }
        row.update({k: v for k, v in cfg.items() if k not in row})
        rows.append(row)

    summary = pd.DataFrame(rows)
    summary.to_csv(RESULTS_DIR / "structural_baselines_v6_candidates.csv", index=False)

    log_path = RESULTS_DIR / "public_experiment_log.csv"
    if log_path.exists():
        log = pd.read_csv(log_path)
        existing = set(log["candidate_id"].astype(str))
        new_rows = []
        for _, row in summary.iterrows():
            if str(row["candidate_id"]) in existing:
                continue
            new_rows.append(
                {
                    "candidate_id": row["candidate_id"],
                    "file": row["file"],
                    "family": row["family"],
                    "hypothesis": row["selection_reason"],
                    "public_score": "",
                    "notes": "fill public_score after submission",
                    "public_mape": "",
                    "delta_mape_vs_current_best": "",
                    "delta_score_vs_current_best": "",
                }
            )
        if new_rows:
            pd.concat([log, pd.DataFrame(new_rows)], ignore_index=True).to_csv(log_path, index=False)

    print(summary.to_string(index=False))


main()


## 7. Temporal204 GRU Residual

GRU residual для temporal204.


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



BATCH_SIZE = TN_BATCH_SIZE
BEHAVIOR_COLS = TN_BEHAVIOR_COLS
DEVICE = TN_DEVICE
EPOCHS = TN_EPOCHS
FINAL_EPOCHS = TN_FINAL_EPOCHS
GRUResidual = TN_GRUResidual
MOMENTUM_TRAIN_BINS = TN_MOMENTUM_TRAIN_BINS
PATIENCE = TN_PATIENCE
SEED = TN_SEED
SEQ_LEN = TN_SEQ_LEN
STATIC_CAT_COLS = TN_STATIC_CAT_COLS
STATIC_NUM_COLS = TN_STATIC_NUM_COLS
build_sequence = TN_build_sequence
build_static = TN_build_static
center_by_group = TN_center_by_group
fit_sequence_scaler = TN_fit_sequence_scaler
fit_static_encoders = TN_fit_static_encoders
momentum_bin = TN_momentum_bin
period_id = TN_period_id
set_seed = TN_set_seed
transform_static = TN_transform_static


RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def make_loader_generator(seed: int) -> torch.Generator:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    return generator




OUT_DIR = RESULTS_DIR / "temporal204_gru_residual"
BASE_SPEC = HeuristicSpec(0, "temporal204", CURRENT_LAMBDA, momentum_exp=0.0)


class PreparedData:
    def __init__(self, raw, rto_wide, behavior_wide, static_frame, train_rows, test_rows):
        self.raw = raw
        self.rto_wide = rto_wide
        self.behavior_wide = behavior_wide
        self.static_frame = static_frame
        self.train_rows = train_rows
        self.test_rows = test_rows


def temporal204_base(frame: pd.DataFrame, rto_label_wide: pd.DataFrame, factor: float, year: int, month: int, target_non_leap: bool) -> np.ndarray:
    base_198 = current_198_march(frame, factor, target_non_leap=target_non_leap)
    anchor = two_month_temporal_anchor(
        rto_label_wide,
        frame,
        year,
        month,
        daily_adjust=False,
        clip_lo=0.85,
        clip_hi=1.35,
    )
    return blend_anchor(base_198, anchor, alpha=0.12, log_clip=0.06)


def prepare_data() -> PreparedData:
    raw, rto_label_wide, static_for_base = load_data()
    raw = raw.copy()
    raw["period"] = raw["Год"] * 12 + raw["Месяц"]
    rto_wide = raw.pivot_table(index="new_id", columns="period", values=TARGET, aggfunc="mean")
    behavior_wide = {
        col: raw.pivot_table(index="new_id", columns="period", values=col, aggfunc="mean")
        for col in BEHAVIOR_COLS
        if col in raw.columns
    }
    static_frame = build_static(raw)

    train_features = build_target_frame(rto_label_wide, static_for_base, 2024, 3, include_target=True)
    test_features = build_target_frame(rto_label_wide, static_for_base, 2025, 3, include_target=False)
    _train_factor, _test_factor, val_global, final_global = factors_for_spec(BASE_SPEC, train_features, test_features, rto_label_wide, static_for_base)
    train_features["new_base"] = temporal204_base(train_features, rto_label_wide, val_global, 2024, 3, target_non_leap=False)
    test_features["new_base"] = temporal204_base(test_features, rto_label_wide, final_global, 2025, 3, target_non_leap=True)
    train_features = train_features[(train_features[TARGET] > 0) & (train_features["new_base"] > 0)].copy()
    train_features["target_log_ratio"] = np.log(train_features[TARGET] / train_features["new_base"])
    train_features["momentum_train_bin"] = momentum_bin(train_features)
    test_features["momentum_train_bin"] = momentum_bin(test_features)

    # Sanity: test baseline should reproduce candidate 204 to rounding-level precision.
    cand204 = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_204_baseline_v6_temporal2_raw_clip085_135_a012_lc006.csv").rename(columns={"rto": "rto_204"})
    check = test_features[["new_id", "new_base"]].merge(cand204, on="new_id")
    max_abs_diff = float(np.max(np.abs(np.round(check["new_base"].to_numpy(float), 2) - check["rto_204"].to_numpy(float))))
    if max_abs_diff > 0.02:
        raise RuntimeError(f"Temporal204 base mismatch vs candidate 204: max_abs_diff={max_abs_diff}")

    return PreparedData(
        raw=raw,
        rto_wide=rto_wide,
        behavior_wide=behavior_wide,
        static_frame=static_frame,
        train_rows=train_features.sort_values("new_id").reset_index(drop=True),
        test_rows=test_features.sort_values("new_id").reset_index(drop=True),
    )


def train_one_fold(train_data, valid_data, cat_cardinalities: list[int], seed: int) -> tuple[np.ndarray, float]:
    set_seed(seed)
    train_seq, train_num, train_cat, y_train = train_data
    valid_seq, valid_num, valid_cat, y_valid = valid_data
    model = GRUResidual(train_seq.shape[-1], train_num.shape[1], cat_cardinalities).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-3)
    loss_fn = nn.SmoothL1Loss(beta=0.02)
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(train_seq),
            torch.from_numpy(train_num),
            torch.from_numpy(train_cat),
            torch.from_numpy(y_train.astype(np.float32)),
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=make_loader_generator(SEED),
        num_workers=0,
        pin_memory=False,
    )
    valid_tensors = (
        torch.from_numpy(valid_seq).to(DEVICE),
        torch.from_numpy(valid_num).to(DEVICE),
        torch.from_numpy(valid_cat).to(DEVICE),
        torch.from_numpy(y_valid.astype(np.float32)).to(DEVICE),
    )
    best_state = None
    best_loss = float("inf")
    stale = 0
    for _epoch in range(EPOCHS):
        model.train()
        for batch_seq, batch_num, batch_cat, batch_y in loader:
            batch_seq = batch_seq.to(DEVICE, non_blocking=True)
            batch_num = batch_num.to(DEVICE, non_blocking=True)
            batch_cat = batch_cat.to(DEVICE, non_blocking=True)
            batch_y = batch_y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = loss_fn(model(batch_seq, batch_num, batch_cat), batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            valid_loss = float(loss_fn(model(valid_tensors[0], valid_tensors[1], valid_tensors[2]), valid_tensors[3]).detach().cpu().item())
        if valid_loss < best_loss - 1e-5:
            best_loss = valid_loss
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= PATIENCE:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(valid_tensors[0], valid_tensors[1], valid_tensors[2]).detach().cpu().numpy()
    return pred, best_loss


def run_oof(data: PreparedData) -> pd.DataFrame:
    target = data.train_rows.copy()
    store_ids = target["new_id"].to_numpy()
    full_seq = build_sequence(data, store_ids, period_id(2024, 3))
    y = target["target_log_ratio"].to_numpy(dtype=np.float32)
    parts = []
    fold_rows = []
    for fold in range(5):
        valid_mask = (store_ids % 5) == fold
        train_ids = store_ids[~valid_mask]
        valid_ids = store_ids[valid_mask]
        seq_mean, seq_std = fit_sequence_scaler(full_seq[~valid_mask])
        train_seq = fit_apply_sequence(full_seq[~valid_mask], seq_mean, seq_std)
        valid_seq = fit_apply_sequence(full_seq[valid_mask], seq_mean, seq_std)
        num_mean, num_std, encoders, num_cols, cat_cols = fit_static_encoders(data.static_frame, train_ids)
        train_num, train_cat = transform_static(data.static_frame, train_ids, num_mean, num_std, encoders, num_cols, cat_cols)
        valid_num, valid_cat = transform_static(data.static_frame, valid_ids, num_mean, num_std, encoders, num_cols, cat_cols)
        pred_log, best_loss = train_one_fold((train_seq, train_num, train_cat, y[~valid_mask]), (valid_seq, valid_num, valid_cat, y[valid_mask]), [len(encoders[col]) for col in cat_cols], SEED + fold)
        valid_rows = target.loc[valid_mask].copy()
        valid_rows["fold"] = fold
        valid_rows["pred_log_gru"] = pred_log
        parts.append(valid_rows[["new_id", TARGET, "new_base", "momentum_train_bin", "fold", "pred_log_gru"]])
        fold_rows.append({"fold": fold, "valid_loss": best_loss, "base_mape": mape(valid_rows[TARGET], valid_rows["new_base"])})
    oof = pd.concat(parts, ignore_index=True)
    pd.DataFrame(fold_rows).to_csv(OUT_DIR / "fold_summary.csv", index=False)
    return oof


def fit_apply_sequence(seq: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    out = (seq - mean[None, None, :]) / std[None, None, :]
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def evaluate_oof(oof: pd.DataFrame) -> pd.DataFrame:
    rows = [{"variant": "temporal204_base", "alpha": 0.0, "center": "none", "mape": mape(oof[TARGET], oof["new_base"])}]
    for center in ["none", "global", "momentum_bin"]:
        if center == "none":
            pred_log = oof["pred_log_gru"]
        elif center == "global":
            pred_log = oof["pred_log_gru"] - float(oof["pred_log_gru"].median())
        else:
            pred_log = center_by_group(oof["pred_log_gru"], oof["momentum_train_bin"])
        for alpha in [0.3, 0.5, 0.7, 0.85, 1.0, 1.05, 1.15, 1.3]:
            pred = oof["new_base"].to_numpy(dtype=float) * np.exp(alpha * pred_log.to_numpy(dtype=float))
            rows.append({"variant": f"temporal204_gru_{center}", "alpha": alpha, "center": center, "mape": mape(oof[TARGET], pred)})
    return pd.DataFrame(rows).sort_values("mape").reset_index(drop=True)


def train_final_and_predict(data: PreparedData) -> np.ndarray:
    set_seed(SEED)
    train = data.train_rows.copy()
    store_ids = train["new_id"].to_numpy()
    y = train["target_log_ratio"].to_numpy(dtype=np.float32)
    seq_raw = build_sequence(data, store_ids, period_id(2024, 3))
    seq_mean, seq_std = fit_sequence_scaler(seq_raw)
    seq = fit_apply_sequence(seq_raw, seq_mean, seq_std)
    num_mean, num_std, encoders, num_cols, cat_cols = fit_static_encoders(data.static_frame, store_ids)
    num, cat = transform_static(data.static_frame, store_ids, num_mean, num_std, encoders, num_cols, cat_cols)
    valid_mask = (store_ids % 5) == 0
    pred_dummy, _ = train_one_fold(
        (seq[~valid_mask], num[~valid_mask], cat[~valid_mask], y[~valid_mask]),
        (seq[valid_mask], num[valid_mask], cat[valid_mask], y[valid_mask]),
        [len(encoders[col]) for col in cat_cols],
        SEED,
    )
    del pred_dummy
    model = GRUResidual(seq.shape[-1], num.shape[1], [len(encoders[col]) for col in cat_cols]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=2e-3)
    loss_fn = nn.SmoothL1Loss(beta=0.02)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(seq), torch.from_numpy(num), torch.from_numpy(cat), torch.from_numpy(y)),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=make_loader_generator(SEED),
        num_workers=0,
        pin_memory=False,
    )
    for _epoch in range(FINAL_EPOCHS):
        model.train()
        for batch_seq, batch_num, batch_cat, batch_y in loader:
            batch_seq = batch_seq.to(DEVICE, non_blocking=True)
            batch_num = batch_num.to(DEVICE, non_blocking=True)
            batch_cat = batch_cat.to(DEVICE, non_blocking=True)
            batch_y = batch_y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = loss_fn(model(batch_seq, batch_num, batch_cat), batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    test_ids = data.test_rows["new_id"].to_numpy()
    test_seq = fit_apply_sequence(build_sequence(data, test_ids, period_id(2025, 3)), seq_mean, seq_std)
    test_num, test_cat = transform_static(data.static_frame, test_ids, num_mean, num_std, encoders, num_cols, cat_cols)
    model.eval()
    with torch.no_grad():
        return model(torch.from_numpy(test_seq).to(DEVICE), torch.from_numpy(test_num).to(DEVICE), torch.from_numpy(test_cat).to(DEVICE)).detach().cpu().numpy()


def write_candidates(data: PreparedData, pred_log: np.ndarray, oof: pd.DataFrame, summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    test = data.test_rows.copy()
    test["pred_log_gru"] = pred_log
    test["pred_log_global"] = test["pred_log_gru"] - float(test["pred_log_gru"].median())
    test["pred_log_mombin"] = center_by_group(test["pred_log_gru"], test["momentum_train_bin"])
    base = test["new_base"].to_numpy(dtype=float)
    preds: dict[int, tuple[str, np.ndarray, str, float | None]] = {}
    for candidate_id, name, col, alpha in [
        (206, "temporal204_newgru_raw_a070", "pred_log_gru", 0.70),
        (207, "temporal204_newgru_raw_a100", "pred_log_gru", 1.00),
        (208, "temporal204_newgru_global_a070", "pred_log_global", 0.70),
        (209, "temporal204_newgru_global_a085", "pred_log_global", 0.85),
        (210, "temporal204_newgru_global_a100", "pred_log_global", 1.00),
        (211, "temporal204_newgru_global_a105", "pred_log_global", 1.05),
        (212, "temporal204_newgru_mombin_a100", "pred_log_mombin", 1.00),
        (213, "temporal204_newgru_mombin_a115", "pred_log_mombin", 1.15),
    ]:
        preds[candidate_id] = (name, base * np.exp(alpha * test[col].to_numpy(dtype=float)), col, alpha)

    # Include the previously useful independent old-transfer signal as a small blend candidate.
    transfer = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_118_baseline_cal050_mombin_g100.csv").rename(columns={"rto": "transfer118"})
    best159 = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_159_baseline_faithful_global_blend_75_25_transfer.csv").rename(columns={"rto": "best159"})
    frame_blend = test[["new_id"]].merge(transfer, on="new_id").merge(best159, on="new_id")
    g100 = preds[210][1]
    preds[214] = ("temporal204_newgru_global_a100_blend85_transfer118", 0.85 * g100 + 0.15 * frame_blend["transfer118"].to_numpy(float), "blend_transfer118", None)
    preds[215] = ("temporal204_newgru_global_a100_blend75_transfer118", 0.75 * g100 + 0.25 * frame_blend["transfer118"].to_numpy(float), "blend_transfer118", None)
    preds[216] = ("temporal204_newgru_global_a100_blend85_best159", 0.85 * g100 + 0.15 * frame_blend["best159"].to_numpy(float), "blend_best159", None)

    for candidate_id, (name, pred, center, alpha) in preds.items():
        out = test[["new_id"]].copy()
        out["rto"] = np.round(np.maximum(pred, 0), 2)
        file_name = f"stage2_candidate_{candidate_id:03d}_baseline_{name}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)
        rows.append(
            {
                "candidate_id": candidate_id,
                "file": file_name,
                "name": name,
                "alpha": alpha if alpha is not None else np.nan,
                "center": center,
                "test_mean_rto": float(np.mean(pred)),
                "test_total_rto": float(np.sum(pred)),
                "mean_abs_change_vs_base204_pct": float(np.mean(np.abs(pred / base - 1.0)) * 100),
            }
        )
    out_summary = pd.DataFrame(rows)
    out_summary.to_csv(OUT_DIR / "submission_candidates.csv", index=False)

    log_path = RESULTS_DIR / "public_experiment_log.csv"
    if log_path.exists():
        log = pd.read_csv(log_path)
        existing = set(log["candidate_id"].astype(str))
        new_rows = []
        for _, row in out_summary.iterrows():
            if str(row["candidate_id"]) in existing:
                continue
            new_rows.append(
                {
                    "candidate_id": row["candidate_id"],
                    "file": row["file"],
                    "family": "temporal204_gru_residual",
                    "hypothesis": "Fresh faithful GRU residual trained directly on candidate 204 temporal/Q1 pure baseline; includes best centering and transfer-blend variants.",
                    "public_score": "",
                    "notes": "fill public_score after submission",
                    "public_mape": "",
                    "delta_mape_vs_current_best": "",
                    "delta_score_vs_current_best": "",
                }
            )
        if new_rows:
            pd.concat([log, pd.DataFrame(new_rows)], ignore_index=True).to_csv(log_path, index=False)
    return out_summary


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Using device: {DEVICE}")
    data = prepare_data()
    oof = run_oof(data)
    oof.to_csv(OUT_DIR / "oof.csv", index=False)
    summary = evaluate_oof(oof)
    summary.to_csv(OUT_DIR / "oof_summary.csv", index=False)
    pred_log = train_final_and_predict(data)
    candidates = write_candidates(data, pred_log, oof, summary)
    metadata = {
        "seq_len": SEQ_LEN,
        "epochs": EPOCHS,
        "final_epochs": FINAL_EPOCHS,
        "base": "candidate_204 temporal2_raw_clip085_135_a012_lc006",
        "base_spec": BASE_SPEC.__dict__,
        "device": str(DEVICE),
    }
    (OUT_DIR / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    print("OOF summary:")
    print(summary.head(24).to_string(index=False))
    print("\nSubmission candidates:")
    print(candidates.to_string(index=False))


main()


T204_prepare_data = prepare_data


## 8. Temporal204 Second Residual



In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



prepare_data = T204_prepare_data


RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool




SEED = 2026
OUT_DIR = RESULTS_DIR / "temporal204_second_residual"


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["log_lag1"] = np.log(np.clip(out["lag1"], 1.0, None))
    out["log_new_base"] = np.log(np.clip(out["new_base"], 1.0, None))
    out["base_to_lag12"] = out["new_base"] / out["lag12"]
    out["base_to_lag13"] = out["new_base"] / out["lag13"]
    out["dip_x_yoy"] = out["dip_score"] * out["lag1_to_lag12"].fillna(1.0)
    out["mom_x_vol"] = out["momentum_raw"] * out["volatility_rel3"]
    out["first_corr_abs"] = out["first_corr_log"].abs()
    out["first_corr_x_mom"] = out["first_corr_log"] * out["momentum_raw"].fillna(1.0)
    out["first_corr_x_vol"] = out["first_corr_log"] * out["volatility_rel3"].fillna(0.0)
    out["pred210_to_lag12"] = out["pred210"] / out["lag12"]
    out["pred210_to_lag13"] = out["pred210"] / out["lag13"]
    out["pred210_to_base"] = out["pred210"] / out["new_base"]
    return out.replace([np.inf, -np.inf], np.nan)


def feature_columns(frame: pd.DataFrame) -> list[str]:
    numeric = [
        "lag1",
        "lag2",
        "lag3",
        "lag12",
        "lag13",
        "rolling3_mean",
        "rolling2_pre_mean",
        "momentum_raw",
        "momentum_clip",
        "feb_to_jan",
        "jan_to_dec",
        "lag1_to_lag12",
        "lag1_to_lag13",
        "store_mar_feb_ratio_prev",
        "volatility_rel3",
        "dip_depth",
        "dip_score",
        "temporary_dip_score",
        "structural_decline_score",
        "new_base",
        "pred210",
        "first_corr_log",
        "first_corr_abs",
        "log_lag1",
        "log_new_base",
        "base_to_lag12",
        "base_to_lag13",
        "dip_x_yoy",
        "mom_x_vol",
        "first_corr_x_mom",
        "first_corr_x_vol",
        "pred210_to_lag12",
        "pred210_to_lag13",
        "pred210_to_base",
    ]
    numeric += [col for col in STATIC_NUM_COLS if col in frame.columns]
    cats = [col for col in STATIC_CAT_COLS if col in frame.columns]
    return numeric + cats


def build_frames() -> tuple[pd.DataFrame, pd.DataFrame]:
    data = prepare_data()
    train = data.train_rows.copy()
    test = data.test_rows.copy()

    oof = pd.read_csv(RESULTS_DIR / "temporal204_gru_residual" / "oof.csv")[["new_id", "fold", "pred_log_gru"]]
    oof["first_corr_log"] = oof["pred_log_gru"] - float(oof["pred_log_gru"].median())
    train = train.merge(oof, on="new_id", how="inner", suffixes=("", "_oof"))
    train["pred210"] = train["new_base"].to_numpy(float) * np.exp(train["first_corr_log"].to_numpy(float))
    train["target_second_log_ratio"] = np.log(train[TARGET].to_numpy(float) / train["pred210"].to_numpy(float))
    train["fold"] = train["fold_oof"] if "fold_oof" in train.columns else train["fold"]

    sub204 = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_204_baseline_v6_temporal2_raw_clip085_135_a012_lc006.csv").rename(columns={"rto": "base204"})
    sub210 = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_210_baseline_temporal204_newgru_global_a100.csv").rename(columns={"rto": "pred210_sub"})
    test = test.merge(sub204, on="new_id").merge(sub210, on="new_id")
    test["first_corr_log"] = np.log(test["pred210_sub"].to_numpy(float) / test["base204"].to_numpy(float))
    test["pred210"] = test["pred210_sub"].to_numpy(float)

    train = add_features(train)
    test = add_features(test)
    for col in STATIC_CAT_COLS:
        if col in train.columns:
            train[col] = train[col].astype(str).fillna("__nan__")
            test[col] = test[col].astype(str).fillna("__nan__")
    return train, test


def model_params() -> dict:
    return dict(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=300,
        learning_rate=0.025,
        depth=3,
        l2_leaf_reg=80.0,
        random_seed=SEED,
        allow_writing_files=False,
        verbose=False,
    )


def train_oof(train: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    cat_idx = [features.index(col) for col in STATIC_CAT_COLS if col in features]
    parts = []
    for fold in range(5):
        tr = train[train["fold"] != fold]
        va = train[train["fold"] == fold].copy()
        model = CatBoostRegressor(**model_params())
        model.fit(Pool(tr[features], tr["target_second_log_ratio"], cat_features=cat_idx))
        va["pred_second_log_cb"] = model.predict(Pool(va[features], cat_features=cat_idx))
        parts.append(va[["new_id", TARGET, "new_base", "pred210", "fold", "target_second_log_ratio", "pred_second_log_cb"]])
    return pd.concat(parts, ignore_index=True)


def evaluate_oof(oof: pd.DataFrame) -> pd.DataFrame:
    rows = [{"variant": "pred210", "gamma": 0.0, "clip": "", "mape": mape(oof[TARGET], oof["pred210"])}]
    centered = oof["pred_second_log_cb"] - float(oof["pred_second_log_cb"].median())
    for center_name, corr_base in [("raw", oof["pred_second_log_cb"]), ("global", centered)]:
        for gamma in [0.05, 0.10, 0.15, 0.20, 0.30, 0.45, 0.60]:
            for clip in [None, 0.003, 0.005, 0.008, 0.012, 0.020]:
                corr = gamma * corr_base.to_numpy(float)
                if clip is not None:
                    corr = np.clip(corr, -clip, clip)
                pred = oof["pred210"].to_numpy(float) * np.exp(corr)
                rows.append({"variant": f"second_cb_{center_name}", "gamma": gamma, "clip": "" if clip is None else clip, "mape": mape(oof[TARGET], pred)})
    return pd.DataFrame(rows).sort_values("mape").reset_index(drop=True)


def train_final_and_write(train: pd.DataFrame, test: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    cat_idx = [features.index(col) for col in STATIC_CAT_COLS if col in features]
    model = CatBoostRegressor(**model_params())
    model.fit(Pool(train[features], train["target_second_log_ratio"], cat_features=cat_idx))
    test = test.copy()
    test["pred_second_log_cb"] = model.predict(Pool(test[features], cat_features=cat_idx))
    test["pred_second_log_global"] = test["pred_second_log_cb"] - float(test["pred_second_log_cb"].median())

    sub215 = pd.read_csv(SUBMISSIONS_DIR / "stage2_candidate_215_baseline_temporal204_newgru_global_a100_blend75_transfer118.csv").rename(columns={"rto": "pred215"})
    test = test.merge(sub215, on="new_id")

    specs = [
        (217, "second_cb_on210_global_g010_clip005", "pred210", "pred_second_log_global", 0.10, 0.005),
        (218, "second_cb_on210_global_g020_clip008", "pred210", "pred_second_log_global", 0.20, 0.008),
        (219, "second_cb_on210_global_g030_clip012", "pred210", "pred_second_log_global", 0.30, 0.012),
        (220, "second_cb_on215_global_g010_clip005", "pred215", "pred_second_log_global", 0.10, 0.005),
        (221, "second_cb_on215_global_g020_clip008", "pred215", "pred_second_log_global", 0.20, 0.008),
    ]
    rows = []
    for candidate_id, name, base_col, corr_col, gamma, clip in specs:
        corr = gamma * test[corr_col].to_numpy(float)
        corr = np.clip(corr, -clip, clip)
        pred = test[base_col].to_numpy(float) * np.exp(corr)
        out = test[["new_id"]].copy()
        out["rto"] = np.round(np.maximum(pred, 0), 2)
        file_name = f"stage2_candidate_{candidate_id:03d}_baseline_temporal204_{name}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)
        ratio_base = pred / test[base_col].to_numpy(float) - 1.0
        rows.append(
            {
                "candidate_id": candidate_id,
                "file": file_name,
                "name": name,
                "base_col": base_col,
                "gamma": gamma,
                "clip": clip,
                "test_mean_rto": float(np.mean(pred)),
                "test_total_rto": float(np.sum(pred)),
                "mean_abs_change_vs_base_pct": float(np.mean(np.abs(ratio_base)) * 100),
                "p95_abs_change_vs_base_pct": float(np.quantile(np.abs(ratio_base), 0.95) * 100),
                "max_abs_change_vs_base_pct": float(np.max(np.abs(ratio_base)) * 100),
            }
        )
    summary = pd.DataFrame(rows)
    summary.to_csv(OUT_DIR / "submission_candidates.csv", index=False)

    log_path = RESULTS_DIR / "public_experiment_log.csv"
    if log_path.exists():
        log = pd.read_csv(log_path)
        existing = set(log["candidate_id"].astype(str))
        new_rows = []
        for _, row in summary.iterrows():
            if str(row["candidate_id"]) in existing:
                continue
            new_rows.append(
                {
                    "candidate_id": row["candidate_id"],
                    "file": row["file"],
                    "family": "temporal204_second_residual",
                    "hypothesis": "OOF-safe small CatBoost second residual layer on top of temporal204 GRU residual predictions.",
                    "public_score": "",
                    "notes": "fill public_score after submission",
                    "public_mape": "",
                    "delta_mape_vs_current_best": "",
                    "delta_score_vs_current_best": "",
                }
            )
        if new_rows:
            pd.concat([log, pd.DataFrame(new_rows)], ignore_index=True).to_csv(log_path, index=False)
    return summary


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    train, test = build_frames()
    features = feature_columns(train)
    oof = train_oof(train, features)
    oof.to_csv(OUT_DIR / "oof.csv", index=False)
    oof_summary = evaluate_oof(oof)
    oof_summary.to_csv(OUT_DIR / "oof_summary.csv", index=False)
    candidates = train_final_and_write(train, test, features)
    pd.Series(features).to_csv(OUT_DIR / "features.csv", index=False, header=["feature"])
    print("OOF summary:")
    print(oof_summary.head(30).to_string(index=False))
    print("\nSubmission candidates:")
    print(candidates.to_string(index=False))


main()


## 9. Stacking V11

Stacking-сборка v11, включая pred265.


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



MOMENTUM_TRAIN_BINS = G_MOMENTUM_TRAIN_BINS
read_local_artifact_csv = G_read_local_artifact_csv


RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler



OUT_DIR = RESULTS_DIR / "stacking_v11"


def read_local_artifact(path: str) -> pd.DataFrame:
    local_path = RESULTS_DIR.parents[0] / path
    if not local_path.exists():
        raise FileNotFoundError(f"Expected local artifact from earlier notebook cells: {local_path}")
    return pd.read_csv(local_path)


def momentum_bin(frame: pd.DataFrame) -> pd.Series:
    return pd.cut(
        frame["momentum_raw"],
        bins=MOMENTUM_TRAIN_BINS,
        labels=[f"q{i}" for i in range(5)],
        include_lowest=True,
    ).astype(str)


def make_base(frame: pd.DataFrame, rto: pd.DataFrame, static: pd.DataFrame, spec: HeuristicSpec, is_test: bool) -> np.ndarray:
    val_factor, test_factor, _val_global, _final_global = factors_for_spec(spec, frame, frame, rto, static)
    factor = test_factor if is_test else val_factor
    return predict_from_spec(frame, factor, spec)


def load_submission(candidate_id: int, col: str) -> pd.DataFrame:
    path = sorted(SUBMISSIONS_DIR.glob(f"stage2_candidate_{candidate_id:03d}_*.csv"))[0]
    return pd.read_csv(path).rename(columns={"rto": col})[["new_id", col]]


def build_prediction_tables() -> tuple[pd.DataFrame, pd.DataFrame]:
    _raw, rto, static = load_data()
    val_frame = build_target_frame(rto, static, 2024, 3, include_target=True)
    test_frame = build_target_frame(rto, static, 2025, 3, include_target=False)

    val = val_frame.copy()
    test = test_frame.copy()
    val["momentum_train_bin"] = momentum_bin(val)
    test["momentum_train_bin"] = momentum_bin(test)

    gru_oof = pd.read_csv(RESULTS_DIR / "temporal204_gru_residual" / "oof.csv")
    gru_oof["pred204"] = gru_oof["new_base"]
    gru_oof["pred210"] = gru_oof["new_base"] * np.exp(gru_oof["pred_log_gru"] - float(gru_oof["pred_log_gru"].median()))
    val = val.merge(gru_oof[["new_id", "pred204", "pred210", "fold"]], on="new_id", how="inner")

    old_oof = read_local_artifact_csv("results/gru_residual/gru_oof.csv")[["new_id", "pred_log_gru"]]
    old_spec = HeuristicSpec(0, "old_raw_mom025", 0.0, momentum_exp=0.25)
    cal050_spec = HeuristicSpec(0, "cal050_mom000", 0.50, momentum_exp=0.0)
    cal050_val = make_base(val_frame, rto, static, cal050_spec, is_test=False)
    old_base_val = make_base(val_frame, rto, static, old_spec, is_test=False)
    old_oof = val_frame[["new_id"]].assign(cal050_base=cal050_val, old_base=old_base_val).merge(old_oof, on="new_id", how="inner")
    old_oof = old_oof.merge(val[["new_id", "momentum_train_bin"]], on="new_id", how="left")
    old_oof["old_ratio_log"] = 0.70 * old_oof["pred_log_gru"].to_numpy(float)
    old_oof["old_ratio_log_mombin"] = center_by_group(old_oof["old_ratio_log"], old_oof["momentum_train_bin"])
    old_oof["pred118"] = old_oof["cal050_base"] * np.exp(old_oof["old_ratio_log_mombin"])
    val = val.merge(old_oof[["new_id", "pred118"]], on="new_id", how="inner")

    nb_oof = pd.read_csv(RESULTS_DIR / "new_baseline_gru_residual" / "oof.csv")
    nb_oof["pred150"] = nb_oof["new_base"] * np.exp(nb_oof["pred_log_gru"] - float(nb_oof["pred_log_gru"].median()))
    val = val.merge(nb_oof[["new_id", "pred150"]], on="new_id", how="inner")
    val["pred159"] = 0.75 * val["pred150"] + 0.25 * val["pred118"]

    val["pred215"] = 0.75 * val["pred210"] + 0.25 * val["pred118"]
    second_oof = pd.read_csv(RESULTS_DIR / "temporal204_second_residual" / "oof.csv")[["new_id", "pred_second_log_cb"]]
    second_oof["second_global"] = second_oof["pred_second_log_cb"] - float(second_oof["pred_second_log_cb"].median())
    val = val.merge(second_oof[["new_id", "second_global"]], on="new_id", how="inner")
    val["pred220"] = val["pred215"] * np.exp(np.clip(0.10 * val["second_global"].to_numpy(float), -0.005, 0.005))

    for cid, col in [(204, "pred204"), (210, "pred210"), (118, "pred118"), (159, "pred159"), (215, "pred215"), (220, "pred220")]:
        test = test.merge(load_submission(cid, col), on="new_id", how="inner")
    test["fold"] = test["new_id"] % 5
    return val, test


def add_stack_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for col in ["pred204", "pred210", "pred118", "pred159", "pred215"]:
        out[f"log_{col}_to_220"] = np.log(np.maximum(out[col].to_numpy(float), 1.0) / np.maximum(out["pred220"].to_numpy(float), 1.0))
    out["log_215_to_210"] = np.log(np.maximum(out["pred215"].to_numpy(float), 1.0) / np.maximum(out["pred210"].to_numpy(float), 1.0))
    out["log_220_to_215"] = np.log(np.maximum(out["pred220"].to_numpy(float), 1.0) / np.maximum(out["pred215"].to_numpy(float), 1.0))
    out["abs_210_118_disagree"] = np.abs(np.log(np.maximum(out["pred210"].to_numpy(float), 1.0) / np.maximum(out["pred118"].to_numpy(float), 1.0)))
    out["log_lag1"] = np.log(np.clip(out["lag1"], 1.0, None))
    out["weak_score"] = ((0.98 - out["momentum_raw"].fillna(1.0)) / 0.08).clip(0.0, 1.0)
    out["rebound_score"] = out["weak_score"] * out["dip_score"].fillna(0.0)
    out["volume_bin"] = pd.qcut(out["lag1"].rank(method="first"), 5, labels=[f"v{i}" for i in range(5)]).astype(str)
    return out.replace([np.inf, -np.inf], np.nan)


def feature_columns(frame: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric = [
        "log_pred204_to_220",
        "log_pred210_to_220",
        "log_pred118_to_220",
        "log_pred159_to_220",
        "log_pred215_to_220",
        "log_215_to_210",
        "log_220_to_215",
        "abs_210_118_disagree",
        "lag1",
        "lag2",
        "lag3",
        "lag12",
        "lag13",
        "rolling3_mean",
        "rolling2_pre_mean",
        "momentum_raw",
        "feb_to_jan",
        "jan_to_dec",
        "lag1_to_lag12",
        "lag1_to_lag13",
        "store_mar_feb_ratio_prev",
        "volatility_rel3",
        "dip_depth",
        "dip_score",
        "temporary_dip_score",
        "structural_decline_score",
        "log_lag1",
        "weak_score",
        "rebound_score",
    ]
    numeric += [col for col in STATIC_NUM_COLS if col in frame.columns]
    cats = ["momentum_train_bin", "volume_bin"] + [col for col in STATIC_CAT_COLS if col in frame.columns]
    return [col for col in numeric if col in frame.columns], [col for col in cats if col in frame.columns]


def make_model(kind: str, numeric: list[str], cats: list[str]):
    encoder = OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False)
    pre = ColumnTransformer(
        transformers=[
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
            ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), encoder), cats),
        ],
        remainder="drop",
    )
    if kind.startswith("ridge"):
        alpha = float(kind.replace("ridge", ""))
        model = Ridge(alpha=alpha, random_state=2026)
    elif kind == "huber010":
        model = HuberRegressor(epsilon=1.20, alpha=0.10, max_iter=300)
    elif kind == "huber030":
        model = HuberRegressor(epsilon=1.35, alpha=0.30, max_iter=300)
    else:
        raise ValueError(kind)
    return make_pipeline(pre, model)


def crossfit_meta(train: pd.DataFrame, test: pd.DataFrame, numeric: list[str], cats: list[str]) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    y = np.log(train[TARGET].to_numpy(float) / np.maximum(train["pred220"].to_numpy(float), 1.0))
    kinds = ["ridge10", "ridge30", "ridge100", "huber010", "huber030"]
    train_corr = pd.DataFrame({"new_id": train["new_id"], "fold": train["fold"]})
    test_corr = {}
    for kind in kinds:
        oof = np.zeros(len(train), dtype=float)
        for fold in sorted(train["fold"].unique()):
            tr_mask = train["fold"] != fold
            va_mask = train["fold"] == fold
            model = make_model(kind, numeric, cats)
            model.fit(train.loc[tr_mask, numeric + cats], y[tr_mask])
            oof[va_mask] = model.predict(train.loc[va_mask, numeric + cats])
        final = make_model(kind, numeric, cats)
        final.fit(train[numeric + cats], y)
        train_corr[kind] = oof
        test_corr[kind] = final.predict(test[numeric + cats])
    return train_corr, test_corr


def segment_correction(train: pd.DataFrame, test: pd.DataFrame, segment_cols: list[str], shrink_k: float) -> tuple[np.ndarray, np.ndarray]:
    resid = np.log(train[TARGET].to_numpy(float) / np.maximum(train["pred220"].to_numpy(float), 1.0))
    train_out = np.zeros(len(train), dtype=float)
    test_parts = []
    global_med = float(np.median(resid))
    for col in segment_cols:
        oof_col = np.zeros(len(train), dtype=float)
        for fold in sorted(train["fold"].unique()):
            tr = train[train["fold"] != fold].copy()
            va = train[train["fold"] == fold].copy()
            tr_resid = pd.Series(resid[train["fold"] != fold], index=tr.index)
            stats = tr.assign(_resid=tr_resid).groupby(col, observed=True)["_resid"].agg(["median", "count"])
            shrink = stats["count"] / (stats["count"] + shrink_k)
            mapping = (stats["median"] * shrink).to_dict()
            oof_col[train["fold"] == fold] = va[col].map(mapping).fillna(global_med).to_numpy(float)
        stats_all = train.assign(_resid=resid).groupby(col, observed=True)["_resid"].agg(["median", "count"])
        shrink_all = stats_all["count"] / (stats_all["count"] + shrink_k)
        mapping_all = (stats_all["median"] * shrink_all).to_dict()
        test_part = test[col].map(mapping_all).fillna(global_med).to_numpy(float)
        train_out += oof_col / len(segment_cols)
        test_parts.append(test_part)
    return train_out, np.mean(test_parts, axis=0)


def evaluate_and_write(train: pd.DataFrame, test: pd.DataFrame, train_corrs: dict[str, np.ndarray], test_corrs: dict[str, np.ndarray]) -> pd.DataFrame:
    rows = []
    test_preds: dict[str, np.ndarray] = {}
    base_oof = train["pred220"].to_numpy(float)
    y = train[TARGET].to_numpy(float)
    base_test = test["pred220"].to_numpy(float)
    rejected250 = load_submission(250, "pred250") if list(SUBMISSIONS_DIR.glob("stage2_candidate_250_*.csv")) else None
    rejected_corr = None
    if rejected250 is not None:
        tmp = test[["new_id", "pred220"]].merge(rejected250, on="new_id", how="inner")
        rejected_corr = np.log(tmp["pred250"].to_numpy(float) / tmp["pred220"].to_numpy(float))

    specs = []
    for name in train_corrs:
        for gamma in [0.25, 0.50, 0.75]:
            for clip in [0.002, 0.003, 0.005]:
                specs.append((f"{name}_g{gamma:.2f}_c{clip:.3f}", train_corrs[name], test_corrs[name], gamma, clip))
    for name_a, name_b in [("ridge30", "segment"), ("huber030", "segment")]:
        if name_a in train_corrs and name_b in train_corrs:
            specs.append((f"avg_{name_a}_{name_b}_g0.50_c0.003", 0.5 * train_corrs[name_a] + 0.5 * train_corrs[name_b], 0.5 * test_corrs[name_a] + 0.5 * test_corrs[name_b], 0.50, 0.003))
            specs.append((f"avg_{name_a}_{name_b}_g0.75_c0.005", 0.5 * train_corrs[name_a] + 0.5 * train_corrs[name_b], 0.5 * test_corrs[name_a] + 0.5 * test_corrs[name_b], 0.75, 0.005))

    for name, tr_corr_raw, te_corr_raw, gamma, clip in specs:
        tr_corr = np.clip(gamma * (tr_corr_raw - np.median(tr_corr_raw)), -clip, clip)
        te_corr = np.clip(gamma * (te_corr_raw - np.median(te_corr_raw)), -clip, clip)
        pred_oof = base_oof * np.exp(tr_corr)
        pred_test = base_test * np.exp(te_corr)
        ratio = pred_test / base_test - 1.0
        row = {
            "name": name,
            "oof_mape": mape(y, pred_oof),
            "delta_vs_220_oof": mape(y, pred_oof) - mape(y, base_oof),
            "mean_abs_vs_220_pct": float(np.mean(np.abs(ratio)) * 100),
            "p95_abs_vs_220_pct": float(np.quantile(np.abs(ratio), 0.95) * 100),
            "max_abs_vs_220_pct": float(np.max(np.abs(ratio)) * 100),
            "mean_delta_vs_220_pct": float(np.mean(ratio) * 100),
            "corr_with_250_corr": float(np.corrcoef(te_corr, rejected_corr)[0, 1]) if rejected_corr is not None and np.std(te_corr) > 0 else np.nan,
        }
        rows.append(row)
        test_preds[name] = pred_test
    summary = pd.DataFrame(rows).sort_values(["delta_vs_220_oof", "mean_abs_vs_220_pct"]).reset_index(drop=True)
    summary.to_csv(OUT_DIR / "stacking_v11_search.csv", index=False)

    chosen = summary[
        (summary["delta_vs_220_oof"] < -0.0005)
        & (summary["mean_abs_vs_220_pct"] >= 0.05)
        & (summary["mean_abs_vs_220_pct"] <= 0.40)
        & (summary["p95_abs_vs_220_pct"] <= 1.20)
    ].head(6).copy()
    if chosen.empty:
        chosen = summary.head(6).copy()
    rows_out = []
    for offset, (_, row) in enumerate(chosen.iterrows(), start=261):
        pred = test_preds[row["name"]]
        out = test[["new_id"]].copy()
        out["rto"] = np.round(np.maximum(pred, 0), 2)
        safe_name = row["name"].replace(".", "p")
        file_name = f"stage2_candidate_{offset:03d}_full_v11_stack_{safe_name}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)
        rows_out.append({"candidate_id": offset, "file": file_name, **row.to_dict()})
    candidates = pd.DataFrame(rows_out)
    candidates.to_csv(OUT_DIR / "submission_candidates.csv", index=False)
    return candidates


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    train, test = build_prediction_tables()
    train = add_stack_features(train)
    test = add_stack_features(test)
    numeric, cats = feature_columns(train)
    train_corr_df, test_corr = crossfit_meta(train, test, numeric, cats)
    train_corrs = {col: train_corr_df[col].to_numpy(float) for col in train_corr_df.columns if col not in {"new_id", "fold"}}
    seg_train, seg_test = segment_correction(train, test, ["Регион", "Торговая площадь, категориальный", "Дата открытия, категориальный", "momentum_train_bin", "volume_bin"], shrink_k=300.0)
    train_corrs["segment"] = seg_train
    test_corr["segment"] = seg_test

    diagnostics = []
    base_mape = mape(train[TARGET], train["pred220"])
    for col in ["pred204", "pred210", "pred118", "pred159", "pred215", "pred220"]:
        diagnostics.append({"prediction": col, "oof_mape": mape(train[TARGET], train[col]), "delta_vs_220": mape(train[TARGET], train[col]) - base_mape})
    pd.DataFrame(diagnostics).to_csv(OUT_DIR / "oof_prediction_diagnostics.csv", index=False)
    candidates = evaluate_and_write(train, test, train_corrs, test_corr)
    print("OOF prediction diagnostics:")
    print(pd.DataFrame(diagnostics).to_string(index=False))
    print("\nSubmission candidates:")
    print(candidates.to_string(index=False))


main()


ST_add_stack_features = add_stack_features
ST_build_prediction_tables = build_prediction_tables
ST_crossfit_meta = crossfit_meta
ST_feature_columns = feature_columns
ST_segment_correction = segment_correction


## 10. V14 Selective Correction

Селективный signal поверх v11.


In [ ]:

sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



add_stack_features = ST_add_stack_features
build_prediction_tables = ST_build_prediction_tables
crossfit_meta = ST_crossfit_meta
feature_columns = ST_feature_columns
segment_correction = ST_segment_correction


RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import numpy as np
import pandas as pd



OUT_DIR = RESULTS_DIR / "v14_selective_v11_correction"


def load_sub(candidate_id: int, col: str) -> pd.DataFrame:
    path = sorted(SUBMISSIONS_DIR.glob(f"stage2_candidate_{candidate_id:03d}_*.csv"))[0]
    return pd.read_csv(path).rename(columns={"rto": col})[["new_id", col]]


def bootstrap_delta(y: np.ndarray, ref: np.ndarray, pred: np.ndarray, n_boot: int = 2000) -> dict:
    rng = np.random.default_rng(2026)
    delta = np.abs((y - pred) / y) * 100 - np.abs((y - ref) / y) * 100
    idx = rng.integers(0, len(y), size=(n_boot, len(y)))
    boot = delta[idx].mean(axis=1)
    return {"boot_p05": float(np.quantile(boot, 0.05)), "boot_p50": float(np.quantile(boot, 0.50)), "boot_p95": float(np.quantile(boot, 0.95)), "prob_improve": float((boot < 0).mean())}


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    train, test = build_prediction_tables()
    train = add_stack_features(train)
    test = add_stack_features(test)
    numeric, cats = feature_columns(train)
    train_corr_df, test_corr = crossfit_meta(train, test, numeric, cats)
    train_corrs = {col: train_corr_df[col].to_numpy(float) for col in train_corr_df.columns if col not in {"new_id", "fold"}}
    seg_train, seg_test = segment_correction(train, test, ["Регион", "Торговая площадь, категориальный", "Дата открытия, категориальный", "momentum_train_bin", "volume_bin"], shrink_k=300.0)

    hub_tr = train_corrs["huber030"] - np.median(train_corrs["huber030"])
    hub_te = test_corr["huber030"] - np.median(test_corr["huber030"])
    seg_tr = seg_train - np.median(seg_train)
    seg_te = seg_test - np.median(seg_test)
    combo_tr = 0.5 * hub_tr + 0.5 * seg_tr
    combo_te = 0.5 * hub_te + 0.5 * seg_te

    # Public-safe anchor 265: avg_huber030_segment, gamma=0.75, clip=0.005.
    c265_tr = np.clip(0.75 * combo_tr, -0.005, 0.005)
    c265_te = np.clip(0.75 * combo_te, -0.005, 0.005)

    conf_tr = np.abs(combo_tr)
    conf_te = np.abs(combo_te)
    agree_tr = np.sign(hub_tr) == np.sign(seg_tr)
    agree_te = np.sign(hub_te) == np.sign(seg_te)

    specs = []
    for q in [0.50, 0.60, 0.70, 0.80]:
        thr = float(np.quantile(conf_tr, q))
        for gamma in [1.00, 1.25, 1.50, 1.75]:
            for clip in [0.008, 0.010, 0.012, 0.015]:
                specs.append((f"top{int((1-q)*100):02d}_g{gamma:.2f}_c{clip:.3f}", thr, gamma, clip, False))
                specs.append((f"agree_top{int((1-q)*100):02d}_g{gamma:.2f}_c{clip:.3f}", thr, gamma, clip, True))

    y = train[TARGET].to_numpy(float)
    base_train = train["pred220"].to_numpy(float)
    base_test = test["pred220"].to_numpy(float)
    pred265_train = base_train * np.exp(c265_tr)
    rows = []
    test_preds = {}
    for name, thr, gamma, clip, require_agree in specs:
        mask_tr = conf_tr >= thr
        mask_te = conf_te >= thr
        if require_agree:
            mask_tr &= agree_tr
            mask_te &= agree_te
        c_tr = c265_tr.copy()
        c_te = c265_te.copy()
        c_tr[mask_tr] = np.clip(gamma * combo_tr[mask_tr], -clip, clip)
        c_te[mask_te] = np.clip(gamma * combo_te[mask_te], -clip, clip)
        pred_tr = base_train * np.exp(c_tr)
        pred_te = base_test * np.exp(c_te)
        ratio265 = pred_te / (base_test * np.exp(c265_te)) - 1.0
        boot = bootstrap_delta(y, pred265_train, pred_tr, n_boot=1000)
        rows.append(
            {
                "name": name,
                "oof_mape": mape(y, pred_tr),
                "delta_vs_265_oof": mape(y, pred_tr) - mape(y, pred265_train),
                "delta_vs_220_oof": mape(y, pred_tr) - mape(y, base_train),
                "train_share_changed": float(mask_tr.mean()),
                "test_share_changed": float(mask_te.mean()),
                "mean_abs_vs_265_pct": float(np.mean(np.abs(ratio265)) * 100),
                "p95_abs_vs_265_pct": float(np.quantile(np.abs(ratio265), 0.95) * 100),
                "max_abs_vs_265_pct": float(np.max(np.abs(ratio265)) * 100),
                **boot,
            }
        )
        test_preds[name] = pred_te

    summary = pd.DataFrame(rows).sort_values(["delta_vs_265_oof", "mean_abs_vs_265_pct"]).reset_index(drop=True)
    summary.to_csv(OUT_DIR / "v14_search.csv", index=False)
    chosen = summary[
        (summary["delta_vs_265_oof"] < -0.0005)
        & (summary["mean_abs_vs_265_pct"].between(0.05, 0.60))
        & (summary["p95_abs_vs_265_pct"] <= 1.50)
        & (summary["boot_p95"] < 0.001)
    ].head(8)
    if chosen.empty:
        chosen = summary.head(8)
    out_rows = []
    for candidate_id, (_, row) in enumerate(chosen.iterrows(), start=284):
        pred = test_preds[row["name"]]
        out = test[["new_id"]].copy()
        out["rto"] = np.round(np.maximum(pred, 0), 2)
        safe = row["name"].replace(".", "p")
        file_name = f"stage2_candidate_{candidate_id:03d}_full_v14_selective_{safe}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)
        out_rows.append({"candidate_id": candidate_id, "file": file_name, **row.to_dict()})
    candidates = pd.DataFrame(out_rows)
    candidates.to_csv(OUT_DIR / "submission_candidates.csv", index=False)
    print("Top v14 search:")
    print(summary.head(30).to_string(index=False))
    print("\nCandidates:")
    print(candidates.to_string(index=False))


main()


## 11. Export Meta Inputs

Формирование meta-input таблиц из только что созданных артефактов.


In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def add_import_paths(workspace_root: Path | None) -> None:
    return None


def safe_log_ratio(num: np.ndarray, den: np.ndarray) -> np.ndarray:
    return np.log(np.maximum(num.astype(float), 1.0) / np.maximum(den.astype(float), 1.0))


def load_sub(submissions_dir: Path, candidate_id: int, col: str) -> pd.DataFrame:
    paths = sorted(submissions_dir.glob(f"stage2_candidate_{candidate_id:03d}_*.csv"))
    if not paths:
        raise FileNotFoundError(f"No submission for candidate {candidate_id:03d} in {submissions_dir}")
    return pd.read_csv(paths[0]).rename(columns={"rto": col})[["new_id", col]]


def build_tables(output_dir: Path, include_full_features: bool) -> tuple[Path, Path, Path]:

    output_dir.mkdir(parents=True, exist_ok=True)
    train, test = build_prediction_tables()
    train = add_stack_features(train)
    test = add_stack_features(test)
    numeric, cats = feature_columns(train)

    train_corr_df, test_corr = crossfit_meta(train, test, numeric, cats)
    train_corrs = {col: train_corr_df[col].to_numpy(float) for col in train_corr_df.columns if col not in {"new_id", "fold"}}
    seg_train, seg_test = segment_correction(
        train,
        test,
        ["Регион", "Торговая площадь, категориальный", "Дата открытия, категориальный", "momentum_train_bin", "volume_bin"],
        shrink_k=300.0,
    )

    hub_tr = train_corrs["huber030"] - np.median(train_corrs["huber030"])
    hub_te = test_corr["huber030"] - np.median(test_corr["huber030"])
    seg_tr = seg_train - np.median(seg_train)
    seg_te = seg_test - np.median(seg_test)
    combo_tr = 0.5 * hub_tr + 0.5 * seg_tr
    combo_te = 0.5 * hub_te + 0.5 * seg_te
    combo265_raw_tr = 0.5 * train_corrs["huber030"] + 0.5 * seg_train
    combo265_raw_te = 0.5 * test_corr["huber030"] + 0.5 * seg_test
    combo265_tr = combo265_raw_tr - np.median(combo265_raw_tr)
    combo265_te = combo265_raw_te - np.median(combo265_raw_te)

    c261_tr = np.clip(0.50 * hub_tr, -0.005, 0.005)
    c261_te = np.clip(0.50 * hub_te, -0.005, 0.005)
    c265_tr = np.clip(0.75 * combo265_tr, -0.005, 0.005)
    c265_te = np.clip(0.75 * combo265_te, -0.005, 0.005)
    c265_v14_internal_tr = np.clip(0.75 * combo_tr, -0.005, 0.005)
    c265_v14_internal_te = np.clip(0.75 * combo_te, -0.005, 0.005)

    pred220_tr = train["pred220"].to_numpy(float)
    pred220_te = test["pred220"].to_numpy(float)
    pred261_tr = pred220_tr * np.exp(c261_tr)
    pred261_te_calc = pred220_te * np.exp(c261_te)
    pred265_tr = pred220_tr * np.exp(c265_tr)
    pred265_te_calc = pred220_te * np.exp(c265_te)

    conf_tr = np.abs(combo_tr)
    conf_te = np.abs(combo_te)
    agree_tr = np.sign(hub_tr) == np.sign(seg_tr)
    agree_te = np.sign(hub_te) == np.sign(seg_te)
    top50_threshold = float(np.quantile(conf_tr, 0.50))
    top50_tr = conf_tr >= top50_threshold
    top50_te = conf_te >= top50_threshold
    agree_top50_tr = top50_tr & agree_tr
    agree_top50_te = top50_te & agree_te

    c284_tr = c265_v14_internal_tr.copy()
    c284_te = c265_v14_internal_te.copy()
    c284_tr[top50_tr] = np.clip(1.25 * combo_tr[top50_tr], -0.015, 0.015)
    c284_te[top50_te] = np.clip(1.25 * combo_te[top50_te], -0.015, 0.015)
    c285_tr = c265_v14_internal_tr.copy()
    c285_te = c265_v14_internal_te.copy()
    c285_tr[agree_top50_tr] = np.clip(1.25 * combo_tr[agree_top50_tr], -0.015, 0.015)
    c285_te[agree_top50_te] = np.clip(1.25 * combo_te[agree_top50_te], -0.015, 0.015)

    base_cols = [
        "new_id",
        "fold",
        TARGET,
        "pred204",
        "pred210",
        "pred118",
        "pred159",
        "pred215",
        "pred220",
    ]
    oof = train[base_cols].copy()
    oof = oof.rename(columns={TARGET: "target_rto"})
    oof["pred261_oof"] = pred261_tr
    oof["pred265_oof"] = pred265_tr
    oof["actual_log_residual_vs_265"] = safe_log_ratio(oof["target_rto"].to_numpy(float), oof["pred265_oof"].to_numpy(float))
    oof["actual_pct_residual_vs_265"] = oof["target_rto"].to_numpy(float) / np.maximum(oof["pred265_oof"].to_numpy(float), 1.0) - 1.0
    oof["ape_265"] = np.abs((oof["target_rto"].to_numpy(float) - oof["pred265_oof"].to_numpy(float)) / oof["target_rto"].to_numpy(float)) * 100.0

    test_out = test[["new_id", "pred204", "pred210", "pred118", "pred159", "pred215", "pred220"]].copy()
    test_out["pred261_calc"] = pred261_te_calc
    test_out["pred265_calc"] = pred265_te_calc
    fallback_submission_values = {
        "pred261_submission": pred261_te_calc,
        "pred265_submission": pred265_te_calc,
        "pred284_submission": pred220_te * np.exp(c284_te),
        "pred285_submission": pred220_te * np.exp(c285_te),
    }
    for candidate_id, col in [(261, "pred261_submission"), (265, "pred265_submission"), (284, "pred284_submission"), (285, "pred285_submission")]:
        try:
            test_out = test_out.merge(load_sub(SUBMISSIONS_DIR, candidate_id, col), on="new_id", how="left")
        except FileNotFoundError:
            test_out[col] = fallback_submission_values[col]

    signal_cols = {
        "v11_huber030_centered": (hub_tr, hub_te),
        "v11_segment_centered": (seg_tr, seg_te),
        "v11_combo_raw": (combo_tr, combo_te),
        "v11_combo265_centered": (combo265_tr, combo265_te),
        "v11_corr261_log": (c261_tr, c261_te),
        "v11_corr265_log": (c265_tr, c265_te),
        "v14_internal_corr265_log": (c265_v14_internal_tr, c265_v14_internal_te),
        "v14_conf_abs_combo": (conf_tr, conf_te),
        "v14_agree_huber_segment": (agree_tr.astype(int), agree_te.astype(int)),
        "v14_top50_mask": (top50_tr.astype(int), top50_te.astype(int)),
        "v14_agree_top50_mask": (agree_top50_tr.astype(int), agree_top50_te.astype(int)),
        "v14_corr284_log": (c284_tr, c284_te),
        "v14_corr285_log": (c285_tr, c285_te),
        "v14_delta284_vs_265_log": (c284_tr - c265_tr, c284_te - c265_te),
        "v14_delta285_vs_265_log": (c285_tr - c265_tr, c285_te - c265_te),
        "v14_delta284_vs_internal265_log": (c284_tr - c265_v14_internal_tr, c284_te - c265_v14_internal_te),
        "v14_delta285_vs_internal265_log": (c285_tr - c265_v14_internal_tr, c285_te - c265_v14_internal_te),
    }
    for col, (tr_values, te_values) in signal_cols.items():
        oof[col] = tr_values
        test_out[col] = te_values

    meta_features = [c for c in numeric + cats if c in train.columns]
    if not include_full_features:
        keep = {
            "log_pred204_to_220",
            "log_pred210_to_220",
            "log_pred118_to_220",
            "log_pred159_to_220",
            "log_pred215_to_220",
            "log_215_to_210",
            "log_220_to_215",
            "abs_210_118_disagree",
            "momentum_raw",
            "feb_to_jan",
            "jan_to_dec",
            "lag1_to_lag12",
            "lag1_to_lag13",
            "store_mar_feb_ratio_prev",
            "volatility_rel3",
            "dip_depth",
            "dip_score",
            "temporary_dip_score",
            "structural_decline_score",
            "weak_score",
            "rebound_score",
            "momentum_train_bin",
            "volume_bin",
            "Регион",
            "Торговая площадь, категориальный",
            "Дата открытия, категориальный",
        }
        meta_features = [c for c in meta_features if c in keep]
    feature_prefix = "feature__"
    for col in meta_features:
        oof[f"{feature_prefix}{col}"] = train[col].to_numpy()
        test_out[f"{feature_prefix}{col}"] = test[col].to_numpy()

    oof = oof.sort_values("new_id").reset_index(drop=True)
    test_out = test_out.sort_values("new_id").reset_index(drop=True)
    oof_path = output_dir / "oof_meta_inputs_v11_v14.csv"
    test_path = output_dir / "test_meta_inputs_v11_v14.csv"
    oof.to_csv(oof_path, index=False)
    test_out.to_csv(test_path, index=False)

    metrics = pd.DataFrame(
        [
            {"variant": "pred204", "oof_mape": mape(train[TARGET], train["pred204"])},
            {"variant": "pred210", "oof_mape": mape(train[TARGET], train["pred210"])},
            {"variant": "pred215", "oof_mape": mape(train[TARGET], train["pred215"])},
            {"variant": "pred220", "oof_mape": mape(train[TARGET], train["pred220"])},
            {"variant": "pred261_oof", "oof_mape": mape(train[TARGET], pred261_tr)},
            {"variant": "pred265_oof", "oof_mape": mape(train[TARGET], pred265_tr)},
            {"variant": "v14_284_oof", "oof_mape": mape(train[TARGET], pred220_tr * np.exp(c284_tr))},
            {"variant": "v14_285_oof", "oof_mape": mape(train[TARGET], pred220_tr * np.exp(c285_tr))},
        ]
    )
    metrics["delta_vs_pred265"] = metrics["oof_mape"] - float(metrics.loc[metrics["variant"] == "pred265_oof", "oof_mape"].iloc[0])
    metrics_path = output_dir / "meta_input_oof_metrics.csv"
    metrics.to_csv(metrics_path, index=False)
    return oof_path, test_path, metrics_path


def main() -> None:
    parser = argparse.ArgumentParser(description="Export temporal204 v11/v14 row-level meta inputs for teammate search.")
    parser.add_argument("--workspace-root", type=Path, default=None, help="Full workspace root with scripts/results/submissions. Use when running from Growth-Gradient but relying on root artifacts.")
    parser.add_argument("--output-dir", type=Path, default=ROOT / "results" / "temporal204_meta_inputs")
    parser.add_argument("--full-features", action="store_true", help="Export every feature from feature_columns instead of the compact selected set.")
    args = parser.parse_args()
    add_import_paths(args.workspace_root)
    oof_path, test_path, metrics_path = build_tables(args.output_dir, include_full_features=args.full_features)
    print("Wrote:")
    print(oof_path)
    print(test_path)
    print(metrics_path)


main()


## 12. V15 Meta Residual: Build Candidate 317 From Internal pred265 Anchor



In [ ]:
sys.argv = ["notebook"]
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
except Exception:
    pass
print("Running in-notebook block")



RESULTS_DIR = H_RESULTS_DIR
SUBMISSIONS_DIR = H_SUBMISSIONS_DIR
TARGET = H_TARGET
HeuristicSpec = H_HeuristicSpec
STATIC_CAT_COLS = H_STATIC_CAT_COLS
STATIC_NUM_COLS = H_STATIC_NUM_COLS
build_target_frame = H_build_target_frame
factors_for_spec = H_factors_for_spec
load_data = H_load_data
mape = H_mape
predict_from_spec = H_predict_from_spec
safe_div = H_safe_div
label = H_label
period_label = H_period_label

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


ROOT = ROOT
RESULTS_DIR = ROOT / "results" / "new_baseline"
SUBMISSIONS_DIR = ROOT / "submissions"
TARGET = "target_rto"
RANDOM_SEED = 2026


def set_random_seed(seed: int = RANDOM_SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.set_num_threads(1)
        torch.use_deterministic_algorithms(True)
        if hasattr(torch, "backends") and hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except Exception:
        pass


def read_input_csv(path: str) -> pd.DataFrame:
    local_path = ROOT / path
    if not local_path.exists():
        raise FileNotFoundError(f"Expected local meta input from stage 11 of this notebook: {local_path}")
    return pd.read_csv(local_path)


def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred) & (y_true > 0)
    return float(np.mean(np.abs((y_pred[mask] - y_true[mask]) / y_true[mask])) * 100.0)


def add_meta_ratios(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    anchor = "pred265_oof" if "pred265_oof" in out.columns else "pred265_submission"
    for col in ["pred204", "pred210", "pred118", "pred159", "pred215", "pred220"]:
        if col in out.columns:
            out[f"meta_log_{col}_to_265"] = np.log(
                np.maximum(out[col].to_numpy(float), 1.0)
                / np.maximum(out[anchor].to_numpy(float), 1.0)
            )
    if "pred284_submission" in out.columns and "pred265_submission" in out.columns:
        out["meta_log_284_to_265"] = np.log(
            np.maximum(out["pred284_submission"].to_numpy(float), 1.0)
            / np.maximum(out["pred265_submission"].to_numpy(float), 1.0)
        )
    return out.replace([np.inf, -np.inf], np.nan)


def feature_columns(train: pd.DataFrame) -> tuple[list[str], list[str]]:
    blocked = {
        "new_id",
        "fold",
        TARGET,
        "pred204",
        "pred210",
        "pred118",
        "pred159",
        "pred215",
        "pred220",
        "pred261_oof",
        "pred265_oof",
        "actual_log_residual_vs_265",
        "actual_pct_residual_vs_265",
        "ape_265",
    }
    candidates = [
        col
        for col in train.columns
        if col not in blocked
        and (
            col.startswith("feature__")
            or col.startswith("v11_")
            or col.startswith("v14_")
            or col.startswith("meta_log_")
        )
    ]
    cats = [col for col in candidates if not is_numeric_dtype(train[col])]
    numeric = [col for col in candidates if col not in cats]
    return numeric, cats


def make_model(kind: str, numeric: list[str], cats: list[str]):
    pre = ColumnTransformer(
        transformers=[
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
            (
                "cat",
                make_pipeline(
                    SimpleImputer(strategy="most_frequent"),
                    OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False),
                ),
                cats,
            ),
        ],
        remainder="drop",
    )
    if kind.startswith("ridge"):
        model = Ridge(alpha=float(kind.replace("ridge", "")))
    elif kind == "huber010":
        model = HuberRegressor(epsilon=1.20, alpha=0.10, max_iter=500)
    elif kind == "huber030":
        model = HuberRegressor(epsilon=1.35, alpha=0.30, max_iter=500)
    elif kind == "huber100":
        model = HuberRegressor(epsilon=1.50, alpha=1.00, max_iter=500)
    else:
        raise ValueError(kind)
    return make_pipeline(pre, model)


def crossfit_predictions(
    train: pd.DataFrame,
    test: pd.DataFrame,
    numeric: list[str],
    cats: list[str],
) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    y = train["actual_log_residual_vs_265"].to_numpy(float)
    kinds = ["ridge10", "ridge30", "ridge100", "ridge300", "huber010", "huber030", "huber100"]
    oof = pd.DataFrame({"new_id": train["new_id"], "fold": train["fold"]})
    test_pred: dict[str, np.ndarray] = {}
    x_cols = numeric + cats
    for kind in kinds:
        pred = np.zeros(len(train), dtype=float)
        for fold in sorted(train["fold"].unique()):
            tr = train["fold"] != fold
            va = train["fold"] == fold
            model = make_model(kind, numeric, cats)
            model.fit(train.loc[tr, x_cols], y[tr])
            pred[va] = model.predict(train.loc[va, x_cols])
        final = make_model(kind, numeric, cats)
        final.fit(train[x_cols], y)
        oof[kind] = pred
        test_pred[kind] = final.predict(test[x_cols])
    return oof, test_pred


def bootstrap_delta(
    y: np.ndarray,
    ref: np.ndarray,
    pred: np.ndarray,
    n_boot: int = 2000,
    seed: int = 2026,
) -> dict[str, float]:
    rng = np.random.default_rng(seed)
    delta = np.abs((y - pred) / y) * 100.0 - np.abs((y - ref) / y) * 100.0
    idx = rng.integers(0, len(delta), size=(n_boot, len(delta)))
    boot = delta[idx].mean(axis=1)
    return {
        "boot_p05": float(np.quantile(boot, 0.05)),
        "boot_p50": float(np.quantile(boot, 0.50)),
        "boot_p95": float(np.quantile(boot, 0.95)),
        "prob_improve": float((boot < 0).mean()),
    }


def evaluate_candidates(
    train: pd.DataFrame,
    test: pd.DataFrame,
    raw_oof: pd.DataFrame,
    raw_test: dict[str, np.ndarray],
) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    y = train[TARGET].to_numpy(float)
    base_oof = train["pred265_oof"].to_numpy(float)
    base_test = test["pred265_submission"].to_numpy(float)
    base_mape = mape(y, base_oof)
    rows = []
    test_predictions: dict[str, np.ndarray] = {}

    specs: list[tuple[str, np.ndarray, np.ndarray, float, float]] = []
    for name in [col for col in raw_oof.columns if col not in {"new_id", "fold"}]:
        for gamma in [0.25, 0.40, 0.60, 0.80, 1.00]:
            for clip in [0.002, 0.003, 0.005, 0.008, 0.012]:
                specs.append((name, raw_oof[name].to_numpy(float), raw_test[name], gamma, clip))

    # Include v14 deltas as a reference family in the same summary.
    for delta_col in ["v14_delta284_vs_265_log", "v14_delta285_vs_265_log"]:
        if delta_col in train.columns and delta_col in test.columns:
            for gamma in [0.50, 0.75, 1.00, 1.25]:
                specs.append((delta_col, train[delta_col].to_numpy(float), test[delta_col].to_numpy(float), gamma, 0.020))

    for name, tr_raw, te_raw, gamma, clip in specs:
        tr_center = tr_raw - float(np.nanmedian(tr_raw))
        te_center = te_raw - float(np.nanmedian(te_raw))
        tr_corr = np.clip(gamma * tr_center, -clip, clip)
        te_corr = np.clip(gamma * te_center, -clip, clip)
        pred_oof = base_oof * np.exp(tr_corr)
        pred_test = base_test * np.exp(te_corr)
        ratio = pred_test / base_test - 1.0
        candidate_name = f"{name}_g{gamma:.2f}_c{clip:.3f}"
        rows.append(
            {
                "name": candidate_name,
                "source": name,
                "gamma": gamma,
                "clip": clip,
                "oof_mape": mape(y, pred_oof),
                "delta_vs_265_oof": mape(y, pred_oof) - base_mape,
                "mean_abs_vs_265_pct": float(np.mean(np.abs(ratio)) * 100.0),
                "p95_abs_vs_265_pct": float(np.quantile(np.abs(ratio), 0.95) * 100.0),
                "max_abs_vs_265_pct": float(np.max(np.abs(ratio)) * 100.0),
                "mean_delta_vs_265_pct": float(np.mean(ratio) * 100.0),
                **bootstrap_delta(y, base_oof, pred_oof, n_boot=800),
            }
        )
        test_predictions[candidate_name] = pred_test

    summary = pd.DataFrame(rows).sort_values(["delta_vs_265_oof", "mean_abs_vs_265_pct"]).reset_index(drop=True)
    return summary, test_predictions


def write_candidates(test: pd.DataFrame, summary: pd.DataFrame, test_predictions: dict[str, np.ndarray]) -> pd.DataFrame:
    auto_selected = summary[
        (summary["delta_vs_265_oof"] < -0.0010)
        & (summary["mean_abs_vs_265_pct"] <= 0.35)
        & (summary["p95_abs_vs_265_pct"] <= 1.00)
        & (summary["prob_improve"] >= 0.95)
    ].head(8)
    if auto_selected.empty:
        auto_selected = summary.head(6)

    tier_names = [
        "huber030_g0.40_c0.003",  # conservative: small public-test shift, still solid OOF gain
        "huber030_g0.60_c0.005",  # safe: larger OOF gain, bounded by about 0.5% max shift
        "huber030_g0.80_c0.008",  # balanced: best middle ground before aggressive moves
        "huber030_g1.00_c0.012",  # aggressive: best OOF row, highest public-test risk
    ]
    tier_selected = summary[summary["name"].isin(tier_names)].copy()
    tier_selected["tier_order"] = tier_selected["name"].map({name: idx for idx, name in enumerate(tier_names)})
    tier_selected = tier_selected.sort_values("tier_order").drop(columns=["tier_order"])

    selected = pd.concat([auto_selected, tier_selected], ignore_index=True)
    selected = selected.drop_duplicates(subset=["name"], keep="first")

    rows = []
    for candidate_id, (_, row) in enumerate(selected.iterrows(), start=310):
        out = test[["new_id"]].copy()
        pred = test_predictions[row["name"]]
        out["rto"] = np.round(np.maximum(pred, 0), 2)
        safe_name = str(row["name"]).replace(".", "p")
        file_name = f"stage2_candidate_{candidate_id}_v15_meta265_{safe_name}.csv"
        out.to_csv(SUBMISSIONS_DIR / file_name, index=False)
        rows.append({"candidate_id": candidate_id, "file": file_name, **row.to_dict()})
    return pd.DataFrame(rows)


def main() -> None:
    set_random_seed()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

    train = read_input_csv("results/temporal204_meta_inputs/oof_meta_inputs_v11_v14.csv")
    test = read_input_csv("results/temporal204_meta_inputs/test_meta_inputs_v11_v14.csv")
    train = add_meta_ratios(train)
    test = add_meta_ratios(test)
    numeric, cats = feature_columns(train)

    raw_oof, raw_test = crossfit_predictions(train, test, numeric, cats)
    raw_oof.to_csv(RESULTS_DIR / "v15_meta265_raw_oof.csv", index=False)

    summary, test_predictions = evaluate_candidates(train, test, raw_oof, raw_test)
    summary.to_csv(RESULTS_DIR / "v15_meta265_search_summary.csv", index=False)
    candidates = write_candidates(test, summary, test_predictions)
    candidates.to_csv(RESULTS_DIR / "v15_meta265_submission_candidates.csv", index=False)

    print("Features:", len(numeric), "numeric,", len(cats), "categorical")
    print("\nTop search rows:")
    print(summary.head(20).to_string(index=False))
    print("\nWritten candidates:")
    print(candidates.to_string(index=False))


main()


## Build Final test.csv


In [ ]:

candidates_path = ROOT / "results" / "new_baseline" / "v15_meta265_submission_candidates.csv"
assert candidates_path.exists(), f"Missing candidates table: {candidates_path}"
candidates = pd.read_csv(candidates_path)
# Fixed pre-selected candidate 317 variant. This is not leaderboard/closed-test selection.
final_row = candidates.loc[candidates["name"].eq(SELECTED_CANDIDATE_317_CONFIG)]
assert not final_row.empty, f"Candidate 317 configuration not found: {SELECTED_CANDIDATE_317_CONFIG}"
source_candidate = ROOT / "submissions" / str(final_row.iloc[0]["file"])
assert source_candidate.exists(), f"Missing final candidate file: {source_candidate}"
submission = pd.read_csv(source_candidate)
root_submission_path = ROOT / "test.csv"
mirror_submission_path = ROOT / "submissions" / "test.csv"
submission.to_csv(root_submission_path, index=False)
submission.to_csv(mirror_submission_path, index=False)
print("Fixed candidate 317 config:", SELECTED_CANDIDATE_317_CONFIG)
print("Internal source file:", source_candidate)
print("Final output for submission:", root_submission_path)
print("Mirrored output:", mirror_submission_path)
display(final_row)


## Validate Output Format


In [ ]:

submission_path = ROOT / "test.csv"
mirror_submission_path = ROOT / "submissions" / "test.csv"
assert submission_path.exists(), "test.csv was not generated"
assert mirror_submission_path.exists(), "submissions/test.csv mirror was not generated"
submission = pd.read_csv(submission_path)
mirror_submission = pd.read_csv(mirror_submission_path)
assert submission.equals(mirror_submission), "Root test.csv and submissions/test.csv differ"
assert submission.shape == (18657, 2), submission.shape
assert submission.columns.tolist() == ["new_id", "rto"], submission.columns.tolist()
assert submission.isna().sum().sum() == 0
assert (submission["rto"] < 0).sum() == 0
print("Output:", submission_path)
print("Mirror:", mirror_submission_path)
print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())
print("NaN count:", int(submission.isna().sum().sum()))
print("Negative rto count:", int((submission["rto"] < 0).sum()))
display(submission.head())
